# 로컬 저장 위치

예측과 실제값 비교 기록은 현재 런타임의 `samsung_direction_outputs/` 폴더에 저장합니다. Google Drive 연결과 인증은 사용하지 않습니다.

Colab 기본 경로는 `/content/samsung_direction_outputs/`입니다. 런타임이 종료·삭제되면 파일이 사라질 수 있으므로 실행 후 왼쪽 파일 패널에서 `latest_outputs.zip`을 다운로드하세요.

다음 런타임에서 기록을 이어 쓰려면 첫 셀 실행 후, 보관한 ZIP의 `forecast_log.csv`를 같은 저장 폴더에 업로드하고 나머지 셀을 실행하세요. KOSIS CSV를 사용하는 경우 `macro_inputs/`에도 다시 업로드해야 합니다.


In [ ]:
# 결과는 현재 런타임의 로컬 폴더에 저장합니다.
# json은 아래 github_get이 쓴다. 이 셀에서 import하지 않으면 첫 번째 종목만
# NameError로 원장 동기화가 꺼지고(두 번째 종목은 뒤 셀이 올려놓은 전역을 써서 통과),
# 그 종목의 보고서만 조용히 발행되지 않는다.
import json
import os
from pathlib import Path

STORAGE_PATH_OVERRIDE = os.environ.get("PREDICT_STOCK_STORAGE")
# ---- 예측 대상 종목 --------------------------------------------------------
# "samsung"(005930) 또는 "sk_hynix"(000660). 종목별로 원장·보고서가 따로 쌓인다.
# 주의: 해외 피처가 반도체 대형주에 맞춰져 있다(SOX·Micron·Nvidia·TSMC). 다른 업종
# 종목을 넣으면 코드는 돌지만 그 종목과 무관한 정보로 예측하게 된다.
# 여기 적은 순서대로 모두 실행한다. 한 종목만 원하면 하나만 남긴다.
# 자동 실행(GitHub Actions)은 프로세스마다 한 종목씩 돌린다. 아래 '이어서 실행'
# 셀은 IPython 실행 히스토리에 기대므로 Jupyter 밖에서는 동작하지 않기 때문이다.
# 환경변수가 없으면 Colab에서 쓰던 기본값 그대로다.
# 환경변수가 비어 있는 경우(Actions에서 입력을 비우면 빈 문자열이 온다)도
# 미설정과 똑같이 기본값으로 떨어뜨린다. get(..., 기본값)만 쓰면 빈 문자열이
# 그대로 통과해 목록이 비고, 바로 아래 RUN_TARGETS[0]에서 IndexError가 난다.
RUN_TARGETS = [t.strip() for t in
               ((os.environ.get("PREDICT_STOCK_TARGETS") or "").strip()
                or "samsung,sk_hynix").split(",")
               if t.strip()]
TARGET = RUN_TARGETS[0]
TARGET_SPEC = {
    "samsung":  {"ticker": "005930.KS", "name": "삼성전자",   "peer": "000660.KS", "gdr": "SMSN.IL"},
    "sk_hynix": {"ticker": "000660.KS", "name": "SK하이닉스", "peer": "005930.KS", "gdr": None},
}[TARGET]
TARGET_NAME = TARGET_SPEC["name"]
print(f"예측 대상: {TARGET_NAME} ({TARGET_SPEC['ticker']})")

STORAGE_ROOT = Path(STORAGE_PATH_OVERRIDE) if STORAGE_PATH_OVERRIDE else Path.cwd() / f"{TARGET}_outputs"
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_CACHE_DIR = str(STORAGE_ROOT / "data_cache")
print("로컬 저장 위치:", STORAGE_ROOT)
print("Colab 런타임이 삭제되기 전에 latest_outputs.zip을 다운로드하세요.")


# ---- 예측 원장을 GitHub에서 받아오기 ------------------------------------------
# Colab 런타임은 삭제되면 사라진다. 원장을 저장소에 두면 실행 기록이 계속 쌓인다.
# 토큰이 없으면 조용히 건너뛰고 런타임 로컬에만 저장한다(전체 실행은 멈추지 않는다).
GITHUB_REPO = "namyikim/predict_stock"   # 원장을 보관할 저장소
GITHUB_BRANCH = "main"
GITHUB_LEDGER_DIR = f"forecast_history/{TARGET}"   # 종목별 폴더
GITHUB_PAGES_DIR = f"docs/{TARGET}"                # 종목별 보고서 폴더
LEDGER_FILES = ["forecast_log.csv", "daily_forecast_comparison.csv",
                "forecast_accuracy_summary.csv"]

# ---- 실행 환경 ---------------------------------------------------------------
# 아래 발행 스위치가 이 값을 쓰므로 여기서 정한다(셀 4의 VERSIONS_HASH도 함께 쓴다).
if os.environ.get("PREDICT_STOCK_RUNTIME"):
    RUNTIME = os.environ["PREDICT_STOCK_RUNTIME"].strip()
elif os.environ.get("GITHUB_ACTIONS") == "true":
    RUNTIME = "github-actions"
else:
    try:
        import google.colab  # noqa: F401
        RUNTIME = "colab"
    except Exception:
        RUNTIME = "local"

# ---- 발행 스위치 -------------------------------------------------------------
# 발행은 한 곳에서만 한다. 양쪽에서 올리면 라이브러리 버전이 다른 예측이 한 원장에
# 쌓여 정확도 이력의 계보가 섞인다. 원장의 runtime/versions_hash로 구분은 되지만
# 애초에 섞지 않는 편이 낫다. 기본값은 자동 실행에서만 발행이고, Colab은 탐색용이다.
# Colab에서 그때만 올리고 싶으면 PREDICT_STOCK_PUBLISH=true 로 실행한다.
_publish = os.environ.get("PREDICT_STOCK_PUBLISH", "").strip().lower()
PUBLISH_SKIP_REASON = ""
if _publish in ("1", "true", "yes", "on"):
    SYNC_LEDGER_TO_GITHUB = True
elif _publish in ("0", "false", "no", "off"):
    SYNC_LEDGER_TO_GITHUB = False
    PUBLISH_SKIP_REASON = "PREDICT_STOCK_PUBLISH=false"
elif RUNTIME == "github-actions":
    SYNC_LEDGER_TO_GITHUB = True
else:
    SYNC_LEDGER_TO_GITHUB = False
    PUBLISH_SKIP_REASON = (f"{RUNTIME} 실행은 기본적으로 발행하지 않습니다"
                           " — 올리려면 PREDICT_STOCK_PUBLISH=true")
if PUBLISH_SKIP_REASON:
    print("ℹ️ 이번 실행은 저장소에 발행하지 않습니다 —", PUBLISH_SKIP_REASON)
    print("   결과는 런타임 로컬에만 저장됩니다.")

# 종목별 발행 결과를 모아 마지막에 한 번에 보여준다. 출력이 길어 중간 경고를
# 놓치기 쉬운데, 한 종목이 조용히 빠져도 알아채지 못하면 그 보고서만 옛날 것으로
# 남는다. 첫 종목에서만 비운다 — 셀 45의 재실행은 TARGET을 다른 값으로 바꾸므로
# 이 조건에서 자연히 걸러진다.
if TARGET == RUN_TARGETS[0]:
    PUBLISH_STATUS = {}


# ---- 보고서 조회수 카운터 ------------------------------------------------------
# GitHub Pages에는 서버가 없어 접속을 셀 수 없다. counter/worker.js 를 Cloudflare에
# 배포하고 그 주소를 여기 적으면 보고서 하단에 조회수가 표시된다.
# 빈 문자열("")로 두면 카운터 코드가 아예 들어가지 않는다(보고서는 정상).
COUNTER_ENDPOINT = "https://predict-stock-counter.kimname1.workers.dev"


def github_token():
    """Colab 보안 비밀 GITHUB_TOKEN 또는 같은 이름의 환경변수."""
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


def github_get(path, token):
    """(내용 문자열, sha) 또는 파일이 없으면 (None, None).
    오류 메시지에 토큰이 든 URL이 섞이지 않도록 종류만 남긴다."""
    import base64
    import urllib.request
    url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{path}?ref={GITHUB_BRANCH}"
    request = urllib.request.Request(url, headers={
        "Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = json.loads(response.read().decode())
        return base64.b64decode(payload["content"]).decode("utf-8"), payload["sha"]
    except Exception as exc:
        if getattr(exc, "code", None) == 404:
            return None, None
        raise RuntimeError(f"GitHub 조회 실패({type(exc).__name__} "
                           f"{getattr(exc, 'code', '')})") from None


if SYNC_LEDGER_TO_GITHUB:
    _token = github_token()
    if not _token:
        print("ℹ️ GITHUB_TOKEN이 없어 원장을 런타임 로컬에만 저장합니다.")
        print("   (Colab 보안 비밀에 GITHUB_TOKEN을 넣으면 저장소에 누적됩니다.)")
        SYNC_LEDGER_TO_GITHUB = False
        PUBLISH_SKIP_REASON = "GITHUB_TOKEN 없음"
    else:
        pulled = []
        try:
            for _name in LEDGER_FILES:
                _text, _ = github_get(f"{GITHUB_LEDGER_DIR}/{_name}", _token)
                if _text is not None:
                    (STORAGE_ROOT / _name).write_text(_text, encoding="utf-8")
                    pulled.append(f"{_name}({_text.count(chr(10))}행)")
            print("GitHub 원장 불러옴:", ", ".join(pulled) if pulled
                  else "(저장소에 아직 원장 없음 — 이번 실행이 첫 기록입니다)")
        except Exception as exc:
            print("⚠️ GitHub 원장 조회 실패:", exc)
            print("   런타임 로컬에만 저장하고 계속 진행합니다.")
            SYNC_LEDGER_TO_GITHUB = False
            PUBLISH_SKIP_REASON = f"원장 조회 실패 ({exc})"

## 0. Colab 런타임

`런타임 → 런타임 유형 변경 → T4 GPU`를 권장합니다. `QUICK_MODE=True`이면 Transformer와 Kronos 평가 구간을 줄여 빠르게 확인합니다. 전체 검증은 설정 셀에서 `QUICK_MODE=False`로 바꾸세요.


In [ ]:
%%capture
# 숫자에 직접 영향을 주는 두 패키지만 버전을 고정한다.
#   - lightgbm: 빌드가 다르면 같은 시드에서도 폴드별 balanced accuracy가 0.07까지 움직인다.
#   - yfinance: 데이터 스키마와 조정계수 처리가 버전마다 바뀐다.
# 나머지는 Colab 런타임 버전을 쓰되, 실제 사용된 버전을 config.json에 기록한다.
# (Kronos 관련 설치는 RUN_KRONOS=True일 때 7절에서만 수행한다.)
!pip -q install "yfinance==1.7.0" "lightgbm==4.7.0" seaborn joblib pyarrow exchange_calendars

In [ ]:
import os
import sys
import gc
import json
import math
import random
import time
import hashlib
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import sklearn
import lightgbm as lgb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
    confusion_matrix,
)
from lightgbm import LGBMClassifier, LGBMRegressor
import joblib

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except Exception as exc:  # torch 없는 CPU 전용 환경도 지원한다.
    torch = None
    TORCH_AVAILABLE = False
    print("torch를 불러오지 못했습니다. Transformer/Kronos 섹션은 건너뜁니다:", exc)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    DEVICE = "cpu"

# 실행 환경을 기록해 두면 나중에 숫자를 재현할 수 있다.
VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "lightgbm": lgb.__version__,
    "yfinance": yf.__version__,
    "torch": torch.__version__ if TORCH_AVAILABLE else None,
}
VERSIONS_HASH = hashlib.sha256(
    json.dumps(VERSIONS, sort_keys=True).encode()).hexdigest()[:12]

# RUNTIME은 첫 셀에서 정한다(발행 스위치가 그 값을 쓴다). config_hash에는
# '무엇을 선택했는가'만 들어가므로, 라이브러리 버전이 다른 환경의 예측이 같은
# 해시를 달고 한 계보에 섞인다. 그래서 원장에 이 둘을 따로 남긴다.
print(json.dumps(VERSIONS, indent=2))
print(f"실행 환경: {RUNTIME} · versions_hash {VERSIONS_HASH}")
print("Device:", DEVICE)
if TORCH_AVAILABLE and getattr(DEVICE, "type", "cpu") != "cuda":
    print("CPU에서도 실행되지만 Transformer/Kronos는 느립니다. Colab GPU를 권장합니다.")

In [ ]:
"""KOSIS 월별 경기/반도체 지표. 최신 수정치의 지연 정렬은 과거 빈티지를 복원하지 않는다."""
import hashlib
import io
import json
import os
import re
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import numpy as np
import pandas as pd


MACRO_SERIES = {
    'leading_cycle': {'orgId': '101', 'tblId': 'DT_1C8015', 'name': '선행지수 순환변동치',
                      'itmId': 'T1', 'unit': '2020=100'},
    'semiconductor_exports': {'orgId': '127', 'tblId': 'DT_092_115_2009_S023',
                              'name': '반도체', 'itmId': '13103131003T1', 'unit': 'USD'},
}
OPTIONAL_MACRO_SERIES = {
    'daily_exports': {'name': '한국 일평균 수출액', 'unit': 'USD per working day'},
    'oecd_g20_cli': {'name': 'OECD Major G20 CLI (amplitude adjusted)', 'unit': 'long-term average=100'},
}
MACRO_HISTORY_NOTE = ('발표 이력 CSV는 각 발표/수정 시각 이후 사용. 이력이 없는 자료는 '
                      'lagged_latest_vintage(월+2 첫날) 가정이며 과거 수정치 누출 가능; 실시간 재검증 필요')


def kosis_key():
    key = os.environ.get('KOSIS_API_KEY')
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get('KOSIS_API_KEY')
        except Exception:
            pass
    return key


def _kosis_request(key, params):
    # HTTP exceptions contain the full URL and API key: never propagate or log them.
    query = {'method': 'getList', 'apiKey': key, 'format': 'json', 'jsonVD': 'Y',
             'prdSe': 'M', **params}
    try:
        with urlopen('https://kosis.kr/openapi/Param/statisticsParameterData.do?' + urlencode(query),
                     timeout=60) as response:
            result = json.loads(response.read().decode('utf-8-sig'))
    except Exception:
        raise RuntimeError('KOSIS API 조회 실패. 키 권한/연결을 확인하거나 macro_inputs CSV를 사용하세요.') from None
    if not isinstance(result, list) or not result:
        raise ValueError('KOSIS API에 수치가 없습니다. API 키, 통계표 접근 권한과 조회 기간을 확인하세요.')
    return result


def _series_rows(rows, series):
    if not isinstance(rows, list):
        raise ValueError('KOSIS가 통계 배열을 반환하지 않았습니다.')
    target = MACRO_SERIES[series]['name'].replace(' ', '')
    selected = [r for r in rows if str(r.get('C1_NM', '')).replace(' ', '') == target]
    if not selected:
        raise ValueError(f'KOSIS 통계표에서 정확한 {target} 합계 항목을 찾지 못했습니다.')
    return selected


def normalize_monthly(frame):
    columns = ['month', 'value'] + [c for c in ('released_at', 'vintage', 'source') if c in frame]
    out = frame[columns].copy()
    months = out['month'].astype(str).str.replace(r'^(\d{4})[./](\d{1,2})$', r'\1-\2', regex=True)
    months = months.str.replace(r'^(\d{4})(\d{2})$', r'\1-\2', regex=True)
    out['month'] = pd.to_datetime(months, format='mixed', errors='raise').dt.to_period('M').dt.to_timestamp()
    out['value'] = pd.to_numeric(out['value'].astype(str).str.replace(',', ''), errors='coerce')
    keys = ['month']
    if 'released_at' in out:
        releases = []
        for value in out['released_at']:
            stamp = pd.Timestamp(value)
            if pd.isna(stamp) or stamp.tzinfo is None:
                raise ValueError('released_at에는 시간대가 포함된 정확한 발표/수정 시각이 필요합니다.')
            releases.append(stamp.tz_convert('UTC'))
        out['released_at'] = pd.to_datetime(releases, utc=True)
        earliest = (out['month'] + pd.offsets.MonthBegin(1)).dt.tz_localize('Asia/Seoul')
        if (out['released_at'] < earliest).any():
            raise ValueError('완결 월별 통계의 발표 시각이 해당 월 종료보다 빠릅니다.')
        keys.append('released_at')
    if out.duplicated(keys).any():
        raise ValueError('월별 통계에 같은 월이 중복됩니다. 하나의 지표/단위만 선택하세요.')
    out.loc[~np.isfinite(out.value) | (out.value <= 0), 'value'] = np.nan
    if out.value.notna().sum() == 0:
        raise ValueError('월별 통계에 유효한 양수 값이 없습니다.')
    return out.sort_values(keys).reset_index(drop=True)


def parse_kosis_rows(rows, series):
    selected = _series_rows(rows, series)
    parsed = []
    for row in selected:
        if row.get('PRD_SE', 'M') != 'M':
            raise ValueError('월별 통계가 필요합니다.')
        value = pd.to_numeric(str(row['DT']).replace(',', ''), errors='coerce')
        if series == 'semiconductor_exports':
            unit = str(row.get('UNIT_NM', '')).replace(' ', '').lower()
            multipliers = {'달러': 1, 'dollars': 1, 'us$': 1, 'usd': 1,
                           '천달러': 1000, '백만달러': 1000000, '억달러': 100000000}
            if unit not in multipliers:
                raise ValueError(f'반도체 수출액의 달러 단위를 확인할 수 없습니다: {unit}')
            value *= multipliers[unit]
        parsed.append({'month': row['PRD_DE'], 'value': value})
    return normalize_monthly(pd.DataFrame(parsed))


def fetch_kosis_monthly(series, start, end, key):
    spec = MACRO_SERIES[series]
    params = {k: spec[k] for k in ('orgId', 'tblId', 'itmId')}
    # Resolve exact aggregate by official classification name; never guess a component code.
    latest = _kosis_request(key, {**params, 'objL1': 'ALL', 'newEstPrdCnt': '1'})
    codes = {row['C1'] for row in _series_rows(latest, series)}
    if len(codes) != 1:
        raise ValueError('KOSIS 항목이 여러 코드에 대응합니다. 분류 개편 여부를 확인하세요.')
    rows = _kosis_request(key, {**params, 'objL1': codes.pop(),
                               'startPrdDe': pd.Timestamp(start).strftime('%Y%m'),
                               'endPrdDe': pd.Timestamp(end).strftime('%Y%m')})
    return parse_kosis_rows(rows, series)


def read_macro_csv(path, series):
    """Normalized month,value CSV or KOSIS time-on-columns CSV (exports: dollars)."""
    content = Path(path).read_bytes()
    try:
        text = content.decode('utf-8-sig')
    except UnicodeDecodeError:
        text = content.decode('cp949')
    frame = pd.read_csv(io.StringIO(text), dtype=str)
    if {'month', 'value'}.issubset(frame.columns):
        return normalize_monthly(frame)
    if series in OPTIONAL_MACRO_SERIES:
        raise ValueError(f'{path.name}: month,value와 선택적 released_at 형식이 필요합니다.')
    month_cols = [c for c in frame if re.fullmatch(r'\d{4}[./-]\d{1,2}|\d{6}', c.strip())]
    target = MACRO_SERIES[series]['name'].replace(' ', '')
    mask = frame.apply(lambda col: col.fillna('').str.replace(' ', '').eq(target)).any(axis=1)
    selected = frame.loc[mask]
    if len(selected) != 1 or not month_cols:
        raise ValueError(f'{path.name}: KOSIS에서 {target}만 선택하여 시점을 열로 CSV를 받거나 month,value 형식으로 저장하세요.')
    # KOSIS table's native export unit is dollars. Other CSV units must be explicit.
    factor = 1
    unit_cols = [c for c in frame if '단위' in c]
    if series == 'semiconductor_exports' and unit_cols:
        unit = str(selected.iloc[0][unit_cols[0]]).replace(' ', '')
        if unit not in {'달러', '천달러', '백만달러', '억달러'}:
            raise ValueError('CSV 수출액 단위를 확인하세요.')
        factor = {'달러': 1, '천달러': 1000, '백만달러': 1000000, '억달러': 100000000}[unit]
    result = normalize_monthly(pd.DataFrame({'month': month_cols, 'value': selected.iloc[0][month_cols].values}))
    result['value'] *= factor
    return result


def load_macro_data(storage, start, end, use_cache=False):
    storage = Path(storage)
    cache = storage / 'macro_cache'
    cache.mkdir(parents=True, exist_ok=True)
    key = kosis_key()
    data, sources = {}, {}
    for series, spec in MACRO_SERIES.items():
        local = storage / 'macro_inputs' / f'{series}.csv'
        cached = cache / f'{series}.csv'
        if use_cache and cached.exists():
            data[series] = read_macro_csv(cached, series)
            sources[series] = 'explicit_cache_replay'
        elif local.exists():
            data[series] = read_macro_csv(local, series)
            sources[series] = 'user_csv'
        elif key:
            data[series] = fetch_kosis_monthly(series, start, end, key)
            sources[series] = 'KOSIS_API'
        else:
            raise RuntimeError('월별 지표가 없습니다. Colab 보안 비밀에 KOSIS_API_KEY를 등록하거나 '
                               f'{local}에 공식 CSV를 저장하세요. 기존 모델만 실행하려면 USE_MACRO_FEATURES=False.')
        data[series].to_csv(cached, index=False)
    optional_status = {}
    for series in OPTIONAL_MACRO_SERIES:
        local = storage / 'macro_inputs' / f'{series}.csv'
        cached = cache / f'{series}.csv'
        path = cached if use_cache and cached.exists() else local
        if not path.exists():
            optional_status[series] = 'not_provided'
            continue
        try:
            frame = read_macro_csv(path, series)
            frame.to_csv(cached, index=False)
        except (ValueError, OSError, KeyError) as exc:
            optional_status[series] = f'invalid:{type(exc).__name__}'
            continue
        data[series] = frame
        sources[series] = 'explicit_cache_replay' if path == cached else 'user_csv'
        optional_status[series] = 'loaded'
    combined = pd.concat([f.assign(series=s) for s, f in data.items()], ignore_index=True)
    payload = combined.to_csv(index=False)
    digest = hashlib.sha256(payload.encode()).hexdigest()[:20]
    snapshots = storage / 'macro_snapshots'
    snapshots.mkdir(parents=True, exist_ok=True)
    info = {'snapshot_hash': digest, 'retrieved_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
            'history_note': MACRO_HISTORY_NOTE, 'sources': sources,
            'series': {s: {**MACRO_SERIES, **OPTIONAL_MACRO_SERIES}[s] for s in data},
            'optional_status': optional_status,
            'availability_modes': {s: ('supplied_release_history' if 'released_at' in f else
                                       'lagged_latest_vintage') for s, f in data.items()},
            'latest_month': {s: f.loc[f.value.notna(), 'month'].max().date().isoformat() for s, f in data.items()}}
    path = snapshots / f'{digest}.csv'
    if not path.exists():
        path.write_text(payload, encoding='utf-8')
        path.with_suffix('.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
    return data, info


def _monthly_features(series, monthly):
    f = pd.DataFrame(index=monthly.index)
    if series in ('leading_cycle', 'oecd_g20_cli'):
        prefix = 'macro_leading' if series == 'leading_cycle' else 'macro_oecd_g20_cli'
        f[prefix + ('_cycle' if series == 'leading_cycle' else '_level')] = monthly - 100
        f[prefix + '_change_1m'] = monthly.diff()
        f[prefix + '_change_3m'] = monthly.diff(3)
    elif series in ('semiconductor_exports', 'daily_exports'):
        prefix = 'macro_semiconductor' if series == 'semiconductor_exports' else 'macro_daily_exports'
        f[prefix + '_log_usd'] = np.log(monthly)
        yoy = monthly.pct_change(12, fill_method=None)
        f[prefix + '_yoy'] = yoy
        f[prefix + '_mom'] = monthly.pct_change(1, fill_method=None)
        f[prefix + '_yoy_3m'] = yoy.rolling(3).mean()
        f[prefix + '_yoy_change_1m'] = yoy.diff()
        f[prefix + '_yoy_change_3m'] = yoy.diff(3)
    else:
        raise ValueError(f'Unknown macro series: {series}')
    return f


def macro_features(data, dates, max_age_days=100, prediction_hour=7):
    """Use releases known at the specified Seoul prediction hour; never backfill revisions.

    Date-only monthly inputs retain the conservative MONTH+2 latest-vintage assumption.
    Explicit released_at rows must contain the actual values published at those instants,
    not today's revised values labelled with original publication dates.
    """
    if not 0 <= prediction_hour < 24:
        raise ValueError('prediction_hour must be in [0, 24)')
    dates = pd.DatetimeIndex(dates)
    local = dates.tz_localize('Asia/Seoul') if dates.tz is None else dates.tz_convert('Asia/Seoul')
    cutoff = (local.normalize() + pd.Timedelta(hours=prediction_hour)).tz_convert('UTC').as_unit('ns')
    left = pd.DataFrame({'available_date': cutoff, '_order': np.arange(len(dates))}).sort_values('available_date')
    result = pd.DataFrame(index=dates)
    for series, frame in data.items():
        monthly = normalize_monthly(frame)
        if 'released_at' not in monthly:
            f = _monthly_features(series, monthly.set_index('month').asfreq('MS')['value'])
            f.index = (f.index + pd.offsets.MonthBegin(2)).tz_localize('Asia/Seoul').tz_convert('UTC').as_unit('ns')
            f['_expires'] = f.index + pd.Timedelta(days=max_age_days, hours=prediction_hour)
        else:
            events = []
            known = pd.Series(dtype=float)
            first_release = {}
            for released, batch in monthly.sort_values('released_at').groupby('released_at', sort=True):
                if released > cutoff.max():
                    break
                for row in batch.itertuples():
                    known.loc[row.month] = row.value
                    first_release.setdefault(row.month, released)
                known = known.sort_index()
                values = _monthly_features(series, known.asfreq('MS')).iloc[-1].to_dict()
                values['available_date'] = released
                # Revising an old month must not make an obsolete last observation fresh.
                values['_expires'] = first_release[known.index[-1]] + pd.Timedelta(days=max_age_days)
                events.append(values)
            if not events:
                cols = _monthly_features(series, pd.Series(dtype=float)).columns
                result[cols] = np.nan
                continue
            f = pd.DataFrame(events).set_index('available_date')
            f.index = pd.DatetimeIndex(f.index).as_unit('ns')
        joined = pd.merge_asof(left, f.rename_axis('available_date').reset_index(),
                               on='available_date', direction='backward').sort_values('_order')
        valid = joined['available_date'] <= joined['_expires']
        for col in f.columns.difference(['_expires'], sort=False):
            result[col] = joined[col].where(valid).to_numpy()
    return result


In [ ]:
"""시간순 예측 모델 선택과 누적 예측 원장. 노트북에도 동일 소스를 포함한다."""
import hashlib
import json
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def rolling_train_indices(date_index, before, years=5):
    dates = pd.DatetimeIndex(date_index)
    before = pd.Timestamp(before)
    return np.flatnonzero((dates >= before - pd.DateOffset(years=years)) & (dates < before))


def direction_estimator(family, params, seed=42):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    if family == "Logistic":
        return make_pipeline(StandardScaler(), LogisticRegression(
            C=params["C"], class_weight=params["class_weight"], max_iter=3000, random_state=seed))
    if family != "LightGBM":
        raise ValueError(f"Unknown model: {family}")
    from lightgbm import LGBMClassifier
    return LGBMClassifier(n_estimators=params["n_estimators"], num_leaves=7,
                          learning_rate=.03, min_child_samples=50, colsample_bytree=.85,
                          reg_alpha=.5, reg_lambda=2., class_weight=params["class_weight"],
                          random_state=seed, n_jobs=2, verbosity=-1)


def aligned_probabilities(estimator, X):
    p = np.full((len(X), 3), 1e-7)
    p[:, np.asarray(estimator.classes_, dtype=int)] = estimator.predict_proba(X)
    p = np.clip(p, 1e-7, 1.)
    return p / p.sum(axis=1, keepdims=True)


def temperature_probabilities(probs, temperature):
    logits = np.log(np.clip(probs, 1e-7, 1.)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    return p / p.sum(axis=1, keepdims=True)


def probability_loss(y, p):
    return float(-np.log(np.clip(p[np.arange(len(y)), np.asarray(y, dtype=int)], 1e-7, 1.)).mean())


def fit_direction_model(X, y, train_indices, family, seed=42):
    """바깥 평가 구간을 보지 않고, 과거 내부 3개 구간의 정확도로 설정을 고른다.

    정확도가 같으면 log loss가 낮은 설정을 선택한다. 온도 보정은 argmax를 유지한다.
    반환값은 표준 estimator와 dict뿐이어서 노트북/로컬 모두 joblib로 다시 읽을 수 있다.
    """
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.dummy import DummyClassifier
    indices = np.asarray(train_indices, dtype=int)
    if len(indices) < 100 or np.any(np.diff(indices) <= 0):
        raise ValueError("At least 100 chronologically ordered training rows are required")
    xt, yt = np.asarray(X)[indices], np.asarray(y)[indices]
    if family == "Logistic":
        candidates = [{"C": c, "class_weight": w} for c in (.003, .01, .03) for w in (None, "balanced")]
    elif family == "LightGBM":
        candidates = [{"n_estimators": n, "class_weight": w} for n in (60, 120) for w in (None, "balanced")]
    else:
        raise ValueError(family)
    splits = list(TimeSeriesSplit(n_splits=3, test_size=min(126, len(yt) // 5)).split(xt))
    labels = np.concatenate([yt[va] for _, va in splits])
    trials = []
    for params in candidates:
        predictions = []
        for tr, va in splits:
            estimator = (direction_estimator(family, params, seed) if len(np.unique(yt[tr])) > 1
                         else DummyClassifier(strategy="prior"))
            estimator.fit(xt[tr], yt[tr])
            predictions.append(aligned_probabilities(estimator, xt[va]))
        probs = np.vstack(predictions)
        accuracy = float(np.mean(probs.argmax(axis=1) == labels))
        trials.append((accuracy, probability_loss(labels, probs), params, probs))
    best = min(trials, key=lambda t: (-t[0], t[1]))
    temperatures = (1., .75, 1.5, 2.)
    temperature = min(temperatures, key=lambda t: probability_loss(labels, temperature_probabilities(best[3], t)))
    estimator = (direction_estimator(family, best[2], seed) if len(np.unique(yt)) > 1
                 else DummyClassifier(strategy="prior"))
    estimator.fit(xt, yt)
    return {"estimator": estimator, "temperature": temperature, "selection": {
        "family": family, "params": best[2], "temperature": temperature,
        "inner_accuracy": best[0], "inner_log_loss": probability_loss(labels, temperature_probabilities(best[3], temperature)),
        "last_validation_position": int(indices[splits[-1][1][-1]]), "training_rows": len(indices),
    }}


def predict_direction_model(fitted, X):
    return temperature_probabilities(aligned_probabilities(fitted["estimator"], X), fitted["temperature"])


def calibrate_price_forecast(y, prediction, sigma, dates, horizon, ci_function, coverage=.8):
    """OOF를 보정 50% / 신호 선택 25% / 최종 평가 25%로 나누고 경계 라벨을 제거한다.

    최종 평가 정답은 slope, 신호 선택, 구간 폭 결정에 사용하지 않는다.
    """
    y, prediction, sigma = map(lambda a: np.asarray(a, dtype=float), (y, prediction, sigma))
    n = len(y)
    cut1, cut2, gap = n // 2, n * 3 // 4, horizon - 1
    cal = np.arange(max(0, cut1 - gap))
    gate = np.arange(cut1, max(cut1, cut2 - gap))
    evaluation = np.arange(cut2, n)
    if min(len(cal), len(gate), len(evaluation)) < 30:
        raise ValueError("Not enough OOF rows for separate price calibration, selection and evaluation")
    denom = np.sum(prediction[cal] ** 2)
    slope = float(np.clip(np.sum(prediction[cal] * y[cal]) / denom, 0., 1.)) if denom > 0 else 0.
    gate_diff = np.abs(y[gate] - slope * prediction[gate]) - np.abs(y[gate])
    gate_lo, gate_hi = ci_function(pd.DatetimeIndex(dates)[gate], lambda i: float(gate_diff[i].mean()))
    beats_baseline = bool(np.isfinite(gate_hi) and gate_hi < 0)
    if not beats_baseline:
        slope = 0.
    residual = np.abs(y[cal] - slope * prediction[cal]) / np.maximum(sigma[cal], 1e-6)
    q = float(np.quantile(residual, coverage))
    test_error = np.abs(y[evaluation] - slope * prediction[evaluation])
    diff = test_error - np.abs(y[evaluation])
    lo, hi = ci_function(pd.DatetimeIndex(dates)[evaluation], lambda i: float(diff[i].mean()))
    return {
        "zero_baseline_mae": float(np.abs(y[evaluation]).mean()),
        "raw_model_mae": float(np.abs(y[evaluation] - prediction[evaluation]).mean()),
        "shrunk_model_mae": float(test_error.mean()), "mae_diff_vs_zero": float(diff.mean()),
        "mae_diff_lo": float(lo), "mae_diff_hi": float(hi),
        "selection_mae_diff_lo": float(gate_lo), "selection_mae_diff_hi": float(gate_hi),
        "oof_slope": slope, "beats_baseline": beats_baseline, "band_q": q,
        "band_coverage_realized": float(np.mean(test_error <= q * sigma[evaluation])),
        "band_halfwidth_mean": float(np.mean(q * sigma[evaluation])),
        "n_oof": n, "n_evaluation": len(evaluation), "calibration_end": int(cal[-1]),
        "gate_start": int(gate[0]), "gate_end": int(gate[-1]), "evaluation_start": int(evaluation[0]),
    }


def price_macro_ablation(X, y, sigma, dates, feature_names, horizon, estimator,
                         ci_function, n_splits=5, coverage=.8):
    """Paired macro-vs-market price evaluation; never choose deployment on the final test.

    Uses identical rows, purged chronological folds and separate calibration/gating for
    both models. Errors are in return units (0.01 = one percentage point), not currency.
    """
    from sklearn.base import clone
    from sklearn.model_selection import TimeSeriesSplit
    X, y, sigma = np.asarray(X), np.asarray(y), np.asarray(sigma)
    dates = pd.DatetimeIndex(dates)
    market = [i for i, name in enumerate(feature_names) if not name.startswith('macro_')]
    if not market or len(market) == len(feature_names):
        raise ValueError('Both market and macro features are required for comparison')
    if (len(X) != len(y) or len(y) != len(sigma) or len(dates) != len(y)
            or not dates.is_monotonic_increasing or not dates.is_unique
            or not np.isfinite(X).all() or not np.isfinite(y).all()
            or not np.isfinite(sigma).all() or np.any(sigma <= 0)):
        raise ValueError('Comparison requires aligned finite rows and positive volatility')
    splits = list(TimeSeriesSplit(n_splits=n_splits, gap=horizon - 1).split(X))
    outputs, stats = {}, {}
    mask = np.zeros(len(y), dtype=bool)
    for _, valid in splits:
        mask[valid] = True
    for name, columns in [('macro', np.arange(X.shape[1])), ('market', market)]:
        oof = np.full(len(y), np.nan)
        for train, valid in splits:
            fitted = clone(estimator).fit(X[train][:, columns], y[train] / sigma[train])
            oof[valid] = fitted.predict(X[valid][:, columns]) * sigma[valid]
        outputs[name] = oof[mask]
        stats[name] = calibrate_price_forecast(y[mask], oof[mask], sigma[mask], dates[mask],
                                               horizon, ci_function, coverage=coverage)
    rows = pd.DataFrame({'actual_return': y[mask], 'macro_raw': outputs['macro'],
                         'market_raw': outputs['market']}, index=dates[mask])
    rows.index.name = 'prediction_date'
    start = stats['macro']['evaluation_start']
    rows['is_evaluation'] = np.arange(len(rows)) >= start
    summary = {'horizon_days': int(horizon), 'n_evaluation': len(rows) - start,
               'evaluation_start': rows.index[start].date().isoformat(),
               'evaluation_end': rows.index[-1].date().isoformat(), 'unit': 'return',
               'comparison': 'macro_minus_market', 'deployment_changed': False}
    for name in outputs:
        rows[name + '_center'] = outputs[name] * stats[name]['oof_slope']
        summary[name + '_slope'] = stats[name]['oof_slope']
        summary[name + '_signal'] = stats[name]['beats_baseline']
        summary[name + '_interval_coverage'] = stats[name]['band_coverage_realized']
    evaluation = rows.iloc[start:]
    for label, suffix in [('raw', 'raw'), ('center', 'center')]:
        macro_error = np.abs(evaluation.actual_return - evaluation['macro_' + suffix]).to_numpy()
        market_error = np.abs(evaluation.actual_return - evaluation['market_' + suffix]).to_numpy()
        delta = macro_error - market_error
        low, high = ci_function(evaluation.index, lambda i: float(delta[i].mean()))
        summary.update({label + '_mae_delta': float(delta.mean()), label + '_mae_delta_lo': float(low),
                        label + '_mae_delta_hi': float(high), label + '_macro_mae': float(macro_error.mean()),
                        label + '_market_mae': float(market_error.mean())})
    return summary, rows


def har_sigma_forecast(returns, horizon, refit_every=60, min_train=500):
    """h거래일 수익률의 스케일(sigma)을 HAR로 예측한다. (시계열, 다음 시점 예측값) 반환.

    HAR: 일/주/월 실현변동성으로 다음 구간 변동성을 회귀한다. 고정된 20일 표준편차보다
    최근 변화에 빠르게 반응해, 같은 적중률에서 예측 구간이 좁아진다.

    각 시점의 예측은 그 시점 이전 자료로만 적합한다. 학습 표본도 목표가 이미 실현된
    구간(i - horizon 이전)으로 제한해 겹치는 라벨이 계수에 들어가지 않게 한다.
    """
    r = pd.Series(returns).astype(float)
    v = r ** 2
    X = pd.DataFrame({
        "const": 1.0,
        "d": np.log(v.shift(1).clip(lower=1e-12)),
        "w": np.log(v.rolling(5).mean().shift(1).clip(lower=1e-12)),
        "m": np.log(v.rolling(22).mean().shift(1).clip(lower=1e-12)),
    }, index=r.index)
    forward = np.sqrt(v.rolling(horizon).sum().shift(-(horizon - 1)))
    y = np.log(forward.clip(lower=1e-12))

    Xv, yv = X.to_numpy(dtype=float), y.to_numpy(dtype=float)
    rows_ok = np.isfinite(Xv).all(axis=1)
    out = np.full(len(r), np.nan)
    beta, last_fit = None, -(10 ** 9)
    for i in range(len(r)):
        end = i - horizon                     # 목표가 이미 실현된 구간만 학습에 쓴다
        if end > min_train and i - last_fit >= refit_every:
            usable = rows_ok[:end] & np.isfinite(yv[:end])
            if usable.sum() >= min_train:
                beta = np.linalg.lstsq(Xv[:end][usable], yv[:end][usable], rcond=None)[0]
                last_fit = i
        if beta is not None and rows_ok[i]:
            out[i] = float(np.exp(Xv[i] @ beta))

    live = np.nan
    usable = rows_ok & np.isfinite(yv)         # y가 NaN인 마지막 h-1행은 자동 제외된다
    if usable.sum() >= min_train:
        full = np.linalg.lstsq(Xv[usable], yv[usable], rcond=None)[0]
        latest = np.array([1.0,
                           np.log(max(float(v.iloc[-1]), 1e-12)),
                           np.log(max(float(v.iloc[-5:].mean()), 1e-12)),
                           np.log(max(float(v.iloc[-22:].mean()), 1e-12))])
        live = float(np.exp(latest @ full))
    return pd.Series(out, index=r.index), live


def snapshot_hash(raw):
    digest = hashlib.sha256()
    for name, frame in sorted(raw.items()):
        digest.update(name.encode())
        digest.update(json.dumps(list(frame.columns)).encode())
        digest.update(pd.util.hash_pandas_object(frame.sort_index(), index=True).values.tobytes())
    return digest.hexdigest()[:20]


def atomic_csv(frame, path):
    """단일 실행자용 원자적 교체. 중간에 런타임이 끊겨도 기존 원장을 보존한다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=path.name, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8-sig", newline="") as stream:
            frame.to_csv(stream, index=False)
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def append_forecasts(path, new_rows):
    """예측은 불변: 같은 record_id 재실행은 무시하고, 새 run_id는 추가한다."""
    path = Path(path)
    previous = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(previous):
        if "record_id" not in previous:
            previous["record_id"] = pd.Series(index=previous.index, dtype="str")
        for i in previous.index[previous.record_id.isna()]:
            payload = previous.loc[i].to_json() + str(i)
            previous.loc[i, "record_id"] = "legacy-" + hashlib.sha256(payload.encode()).hexdigest()[:20]
    combined = pd.concat([previous, new_rows], ignore_index=True)
    combined = combined.drop_duplicates("record_id", keep="first")
    atomic_csv(combined, path)
    return combined


def evaluate_forecasts(log, bars, now=None):
    """확정 봉으로 실제값/오차를 갱신한다. 당시 band/mode 및 원래 예측은 보존한다."""
    result = log.copy()
    now = pd.Timestamp.now(tz="UTC") if now is None else pd.Timestamp(now)
    now = now.tz_localize("UTC") if now.tzinfo is None else now.tz_convert("UTC")
    market_now = now.tz_convert("Asia/Seoul")
    bars = bars.sort_index().copy()
    bars.index = pd.DatetimeIndex(bars.index).tz_localize(None).normalize()
    numeric = ["actual_open", "actual_close", "actual_return", "actual_class", "direction_correct",
               "price_error", "absolute_price_error", "price_ape", "return_error", "interval_hit",
               "center_price_error", "log_loss", "brier"]
    for col in numeric:
        result[col] = np.nan
    result["status"] = "pending"
    result["is_prospective"] = False
    result["actual_updated_at_utc"] = now.isoformat()
    for i, row in result.iterrows():
        target_value = row.get("target_date")
        target_value = row.get("prediction_date") if pd.isna(target_value) else target_value
        target = pd.to_datetime(target_value, errors="coerce")
        if pd.isna(target):
            result.loc[i, "status"] = "invalid_target"
            continue
        target = pd.Timestamp(target).tz_localize(None).normalize()
        start = pd.to_datetime(row.get("prediction_date", target), errors="coerce")
        created = pd.to_datetime(row.get("created_at_utc"), utc=True, errors="coerce")
        mode = row.get("target_mode", "close_to_close")
        # 09:00 시가 기반 예측은 시가 확인 직후(09:05까지)만 별도 집계한다.
        if pd.notna(start) and pd.notna(created):
            deadline = pd.Timestamp(start).tz_localize(None).normalize().tz_localize("Asia/Seoul") + pd.Timedelta(hours=9)
            if mode == "open_to_close":
                deadline += pd.Timedelta(minutes=5)
            result.loc[i, "is_prospective"] = bool(created < deadline)
        if target.date() > market_now.date() or (target.date() == market_now.date() and (market_now.hour, market_now.minute) < (15, 40)):
            continue
        if target not in bars.index:
            result.loc[i, "status"] = "missing_actual"
            continue
        bar = bars.loc[target]
        if not np.isfinite(bar["close"]) or bar["close"] <= 0:
            result.loc[i, "status"] = "missing_actual"
            continue
        result.loc[i, ["actual_open", "actual_close"]] = [bar["open"], bar["close"]]
        kind = row.get("kind", "direction")
        if pd.isna(kind):
            kind = "direction"
        if kind in ("price", "open"):
            # "price": 전일 종가 대비 target_date 종가. "open": 전일 종가 대비 target_date 시가(갭).
            # 시가 예측은 09:00 이전에만 의미가 있으므로 열 이름을 *_open 으로 분리해 종가 예측과 섞지 않는다.
            base = row.get("current_close", np.nan)
            if not np.isfinite(base) or base <= 0:
                result.loc[i, "status"] = "missing_reference"
                continue
            price_col = "close" if kind == "price" else "open"
            actual_price = bar[price_col]
            if not np.isfinite(actual_price) or actual_price <= 0:
                result.loc[i, "status"] = "missing_actual"
                continue
            actual_return = float(actual_price / base - 1)
            predicted = row.get(f"predicted_{price_col}", np.nan)
            if pd.notna(predicted):
                error = float(predicted - actual_price)
                result.loc[i, ["price_error", "absolute_price_error", "price_ape"]] = [error, abs(error), abs(error) / actual_price]
            center = row.get(f"center_{price_col}", np.nan)
            if pd.notna(center):
                result.loc[i, "center_price_error"] = center - actual_price
            predicted_return = row.get("predicted_return", np.nan)
            if pd.notna(predicted_return):
                result.loc[i, "return_error"] = predicted_return - actual_return
            low, high = row.get(f"low_{price_col}", np.nan), row.get(f"high_{price_col}", np.nan)
            if pd.notna(low) and pd.notna(high):
                result.loc[i, "interval_hit"] = float(low <= actual_price <= high)
        else:
            if mode == "open_to_close":
                if not np.isfinite(bar["open"]) or bar["open"] <= 0:
                    result.loc[i, "status"] = "missing_actual"
                    continue
                actual_return = float(bar["close"] / bar["open"] - 1)
            elif mode == "close_to_close":
                if not np.isfinite(bar["adj_close"]) or bar["adj_close"] <= 0:
                    result.loc[i, "status"] = "missing_actual"
                    continue
                base_date = pd.to_datetime(row.get("as_of_date"), errors="coerce")
                if pd.isna(base_date):
                    earlier = bars.index[bars.index < target]
                    base_date = earlier[-1] if len(earlier) else pd.NaT
                if base_date not in bars.index or base_date >= target:
                    result.loc[i, "status"] = "missing_reference"
                    continue
                if not np.isfinite(bars.loc[base_date, "adj_close"]) or bars.loc[base_date, "adj_close"] <= 0:
                    result.loc[i, "status"] = "missing_reference"
                    continue
                # 양쪽 배당조정 가격은 같은 최신 스냅샷에서 읽어 조정계수 변경을 상쇄한다.
                actual_return = float(bar["adj_close"] / bars.loc[base_date, "adj_close"] - 1)
            else:
                result.loc[i, "status"] = "invalid_target_mode"
                continue
            band = row.get("band", np.nan)
            if pd.isna(band) or band < 0:
                result.loc[i, "status"] = "missing_band"
                continue
            actual_class = 0 if actual_return < -band else 2 if actual_return > band else 1
            p = np.asarray([row.get(c, np.nan) for c in ["p_down", "p_flat", "p_up"]], dtype=float)
            result.loc[i, "actual_class"] = actual_class
            if np.isfinite(p).all() and (p >= 0).all() and p.sum() > 0:
                p = p / p.sum()
                result.loc[i, "direction_correct"] = float(p.argmax() == actual_class)
                result.loc[i, "log_loss"] = -np.log(max(p[actual_class], 1e-7))
                result.loc[i, "brier"] = float(np.sum((p - np.eye(3)[actual_class]) ** 2))
        result.loc[i, "actual_return"] = actual_return
        result.loc[i, "status"] = "scored"
    return result


def daily_comparison(evaluated):
    """동일 날짜/모델/설정의 최초 사전 예측만 선택하여 재실행으로 표본이 늘지 않게 한다."""
    if evaluated.empty:
        return evaluated.copy()
    eligible = evaluated.loc[evaluated["is_prospective"].eq(True)].copy()
    if "created_at_utc" not in eligible:
        eligible["created_at_utc"] = pd.NaT
    if "record_id" not in eligible:
        eligible["record_id"] = pd.Series(index=eligible.index, dtype="str")
    eligible["created_at_utc"] = pd.to_datetime(eligible["created_at_utc"], utc=True)
    keys = ["target_date", "model", "kind", "horizon_days", "target_mode", "config_hash"]
    for key in keys:
        if key not in eligible:
            eligible[key] = "legacy"
    return eligible.sort_values("created_at_utc").drop_duplicates(keys, keep="first")


def summarize_daily(daily):
    scored = daily.loc[daily.status == "scored"]
    return scored.groupby(["model", "kind", "horizon_days", "target_mode", "config_hash"], dropna=False).agg(
        n=("record_id", "size"), accuracy=("direction_correct", "mean"),
        mean_log_loss=("log_loss", "mean"), mean_brier=("brier", "mean"),
        point_forecasts=("absolute_price_error", "count"), price_mae=("absolute_price_error", "mean"),
        price_mape=("price_ape", "mean"), interval_coverage=("interval_hit", "mean"),
    ).reset_index()


def review_ledger(daily, bars, ensemble_model="Mean ensemble", windows=(20, 60), min_alert_n=20):
    """실제 사전 예측(daily_comparison)만으로 최근 성능을 계산하고 경고를 만든다.

    백테스트 숫자와 섞지 않는다. 반환:
      latest   — 마지막으로 채점된 예측일의 행(갭/세션 분해 포함)
      rolling  — 최근 w거래일 창별 지표. 방향: 적중률·log loss vs 클래스빈도 기준선.
                 시가/종가: 구간 적중률, 수익률 MAE vs '변화 없음' 기준선, 실현 기울기 vs OOF 축소계수.
      alerts   — 가장 긴 창(표본 min_alert_n 이상)에서 나온 경고 문구
    """
    empty = {"latest": pd.DataFrame(), "rolling": pd.DataFrame(), "alerts": [], "n_scored_days": 0,
             "latest_date": None}
    if daily is None or daily.empty or "status" not in daily:
        return empty
    scored = daily.loc[daily["status"] == "scored"].copy()
    if scored.empty:
        return empty
    scored["target_date"] = pd.to_datetime(scored["target_date"]).dt.tz_localize(None).dt.normalize()
    scored["horizon_days"] = pd.to_numeric(scored.get("horizon_days", 1), errors="coerce").fillna(1).astype(int)
    if "raw_predicted_return" not in scored:
        scored["raw_predicted_return"] = np.nan
    if "oof_slope" not in scored:
        scored["oof_slope"] = np.nan

    bars = bars.sort_index().copy()
    bars.index = pd.DatetimeIndex(bars.index).tz_localize(None).normalize()
    gap = (bars["open"] / bars["close"].shift(1) - 1).rename("actual_gap")
    session = (bars["close"] / bars["open"] - 1).rename("actual_session")
    scored = scored.join(gap, on="target_date").join(session, on="target_date")

    latest_date = scored["target_date"].max()
    keep = [c for c in ["target_date", "kind", "horizon_days", "model", "prediction", "p_down", "p_flat",
                        "p_up", "band", "actual_class", "direction_correct", "log_loss", "current_close",
                        "predicted_return", "raw_predicted_return", "predicted_close", "center_close",
                        "low_close", "high_close", "predicted_open", "center_open", "low_open", "high_open",
                        "actual_open", "actual_close", "actual_return", "actual_gap", "actual_session",
                        "return_error", "interval_hit", "oof_slope", "run_id"] if c in scored]
    latest = scored.loc[scored["target_date"] == latest_date, keep].reset_index(drop=True)

    dates = np.sort(scored["target_date"].unique())
    rows = []
    for w in windows:
        recent = scored[scored["target_date"].isin(dates[-w:])]
        d = recent[(recent["kind"] == "direction") & (recent["model"] == ensemble_model)]
        d = d.dropna(subset=["actual_class"])
        if len(d):
            freq = d["actual_class"].astype(int).value_counts(normalize=True).reindex([0, 1, 2]).fillna(0.)
            prior_ll = float(-np.sum(freq * np.log(np.clip(freq, 1e-7, 1.))))
            rows.append({"window": w, "kind": "direction", "horizon_days": 1, "n": len(d),
                         "hit_rate": float(d["direction_correct"].mean()),
                         "flat_share": float(freq.loc[1]),
                         "mean_log_loss": float(d["log_loss"].mean()), "prior_log_loss": prior_ll})
        for kind in ("open", "price"):
            for h, g in recent[recent["kind"] == kind].groupby("horizon_days"):
                g = g.dropna(subset=["actual_return"])
                if not len(g):
                    continue
                actual = g["actual_return"].to_numpy(dtype=float)
                # 신호가 없어 점 예측을 비운 날은 '변화 없음'(0%)으로 예측한 것과 같다.
                point = g["predicted_return"].fillna(0.).to_numpy(dtype=float)
                raw = g["raw_predicted_return"].to_numpy(dtype=float)
                slope = np.nan
                ok = np.isfinite(raw)
                if ok.sum() >= 5 and np.sum(raw[ok] ** 2) > 0:
                    slope = float(np.sum(raw[ok] * actual[ok]) / np.sum(raw[ok] ** 2))
                rows.append({"window": w, "kind": kind, "horizon_days": int(h), "n": len(g),
                             "interval_coverage": float(g["interval_hit"].mean()) if g["interval_hit"].notna().any() else np.nan,
                             "mae_return": float(np.mean(np.abs(actual - point))),
                             "zero_mae_return": float(np.mean(np.abs(actual))),
                             "signal_days": int(g["predicted_return"].notna().sum()),
                             "realized_slope": slope,
                             "oof_slope_mean": float(g["oof_slope"].mean()) if g["oof_slope"].notna().any() else np.nan})
    rolling = pd.DataFrame(rows)

    alerts = []
    if len(rolling):
        w_max = max(windows)
        big = rolling[rolling["window"] == w_max]
        d = big[big["kind"] == "direction"]
        if len(d) and d["n"].iloc[0] >= min_alert_n and d["mean_log_loss"].iloc[0] > d["prior_log_loss"].iloc[0]:
            alerts.append(f"방향 모델 열화: 최근 {w_max}일 log loss {d['mean_log_loss'].iloc[0]:.3f} > "
                          f"클래스빈도 기준선 {d['prior_log_loss'].iloc[0]:.3f}")
        for _, r in big[big["kind"].isin(["open", "price"])].iterrows():
            label = "시초가" if r["kind"] == "open" else f"{int(r['horizon_days'])}거래일 종가"
            if r["n"] < min_alert_n:
                continue
            if np.isfinite(r["interval_coverage"]) and r["interval_coverage"] < 0.70:
                alerts.append(f"{label} 구간 과소: 최근 {w_max}일 적중률 {r['interval_coverage']:.0%} (목표 80%)")
            if (np.isfinite(r["realized_slope"]) and np.isfinite(r["oof_slope_mean"]) and r["oof_slope_mean"] > 0
                    and not 0.5 <= r["realized_slope"] / r["oof_slope_mean"] <= 1.5):
                direction = "과소" if r["realized_slope"] > r["oof_slope_mean"] else "과대"
                alerts.append(f"{label} {direction}예측: 실현 기울기 {r['realized_slope']:.2f} vs "
                              f"OOF 축소계수 {r['oof_slope_mean']:.2f}")
            if r["mae_return"] > r["zero_mae_return"]:
                alerts.append(f"{label} 점 예측이 '변화 없음'보다 나쁨: MAE {r['mae_return']:.2%} vs {r['zero_mae_return']:.2%}")
    return {"latest": latest, "rolling": rolling, "alerts": alerts, "n_scored_days": int(len(dates)),
            "latest_date": pd.Timestamp(latest_date)}


## 1. 실험 설정

처음에는 아래 기본값으로 실행하는 것을 권장합니다. `KRONOS_EVAL_DAYS`는 파운데이션 모델의 과거 예측 횟수이며, 크게 잡을수록 훨씬 오래 걸립니다.


In [ ]:
START_DATE = "2015-01-01"
USE_MACRO_FEATURES = True       # KOSIS 키 또는 macro_inputs CSV 필요
# 월별 지표를 못 받았을 때 어떻게 할지.
#   False(기본): 크게 경고하고 시세 모델만으로 계속 진행한다. Colab '모두 실행'이
#                중간에 멈추지 않고 마지막 종합 보고서까지 나온다.
#   True       : 예전처럼 즉시 중단한다. 월별 지표가 반드시 들어가야 하는 운영용 실행에 쓴다.
# 어느 쪽이든 비활성 사유는 화면·config.json·종합 보고서에 남으므로 조용히 빠지지 않는다.
# 자동 실행은 사람이 출력을 지켜보지 않는다. 지표가 빠진 채 조용히 발행되느니
# 실패해서 메일을 받는 편이 낫다. 그래서 Actions에서만 환경변수로 켠다.
# 값이 없으면 지금까지처럼 False라 Colab 사용법은 그대로다.
MACRO_STRICT = (os.environ.get("PREDICT_STOCK_MACRO_STRICT", "").strip().lower()
                in ("1", "true", "yes", "on"))
END_DATE = None                 # None이면 현재까지
NEUTRAL_BAND = 0.005            # BAND_MODE="fixed"일 때 ±0.5%

# --- 타깃 정의 ---
# TARGET_MODE
#   "close_to_close": 전일 종가 대비 당일 종가 수익률 (07:00 예측)
#                     ⚠️ 이 수익률의 약 48%는 "전일 종가→당일 시가" 갭이며, 갭은 09:00에
#                     이미 가격에 반영되어 있어 07:00 예측으로는 취할 수 없다. 8절의
#                     갭/세션 분해 표에서 실제로 거래 가능한 예측력을 확인하라.
#   "open_to_close" : 당일 시가 대비 당일 종가 수익률 (09:00 시가 확정 후 예측).
#                     실제로 거래 가능한 타깃이지만, 검증 결과 이 구간의 예측력은
#                     클래스 사전확률과 통계적으로 구분되지 않는다.
TARGET_MODE = "close_to_close"
# BAND_MODE
#   "fixed"     : ±NEUTRAL_BAND 고정
#   "vol_scaled": ±VOL_BAND_MULT × 최근 20일 일간수익률 표준편차(전일까지).
# ⚠️ balanced accuracy는 밴드 폭에 따라 단조 증가하므로, 이 값을 balanced accuracy로
#    튜닝하지 말 것. 왕복 거래비용(약 0.2~0.3%) 기준으로 고정하는 것이 옳다.
BAND_MODE = "vol_scaled"
VOL_BAND_MULT = 0.3
# open_to_close 모드에서 예측일 09:00 시가를 직접 입력한다.
# None이면 라이브 예측을 만들지 않고 중단한다(과거 백테스트는 정상 수행).
LIVE_OPEN_PRICE = None

FIRST_TEST_DATE = "2021-01-01"
TEST_MONTHS = 6                 # 워크포워드 한 구간의 길이
ROLLING_TRAIN_YEARS = 5         # 각 구간에서 최근 5년만 학습

QUICK_MODE = False
MAX_FOLDS = 3 if QUICK_MODE else None

# --- 모델 구성 ---
# 라이브 앙상블은 검증에서 가장 좋았던 조합(정규화한 Logistic + 작은 LightGBM의 단순 평균)을
# 사용한다. 스태킹 메타모델과 log loss 기반 지수가중은 검증에서 단순 평균보다 나빴으므로
# 제거했다.
ENSEMBLE_MODELS = ["Logistic", "LightGBM"]

# 비교·실험용. 앙상블에는 ENSEMBLE_MODELS에 넣어야 들어간다. True로 두면 13절이 결과를
# experiments/transformer/ 에 기록한다(torch 없는 Actions에서는 자동 건너뜀).
# 2026-09-06 측정(삼성전자, A100, 12폴드·25epoch, 평가 1,362일): Transformer 단독 log loss
# 1.0849로 Logistic 1.0321·LightGBM 1.0428보다 나쁘고, 앙상블에 넣으면 +0.0091
# [+0.0044, +0.0141]로 유의하게 나빠졌다(experiments/transformer/summary.csv). 그래서 끈다.
RUN_TRANSFORMER = False
SEQ_LEN = 30
TRANSFORMER_EPOCHS = 8 if QUICK_MODE else 25
TRANSFORMER_PATIENCE = 3 if QUICK_MODE else 5
BATCH_SIZE = 64

# Kronos는 zero-shot 성능이 "항상 보합" 기준선보다 나빴고(log loss 1.87 vs 1.10),
# 무거운 의존성(git clone + HF 가중치)의 유일한 원인이므로 기본값을 False로 둔다.
RUN_KRONOS = False
KRONOS_LOOKBACK = 400
KRONOS_EVAL_DAYS = 20 if QUICK_MODE else 60
KRONOS_MC_SAMPLES = 3 if QUICK_MODE else 8

# --- 평가 설정 ---
COST_BP = 20.0                  # 왕복 거래비용(bp). 한국 단일종목은 매도 거래세 0.15% 포함.
BOOTSTRAP_B = 2000 if not QUICK_MODE else 400   # 월 블록 부트스트랩 반복수
# 자동 실행은 09:00 장 시작 전에 끝나야 원장에 '사전 예측'으로 남는다. 그래서 Actions는
# 환경변수로 반복수를 줄인다(신뢰구간 폭만 조금 거칠어지고 예측값은 바뀌지 않는다).
if (os.environ.get("PREDICT_STOCK_BOOTSTRAP_B") or "").strip():
    BOOTSTRAP_B = int(os.environ["PREDICT_STOCK_BOOTSTRAP_B"])
USE_DATA_CACHE = False          # 매일 실제값을 갱신. True는 고정 스냅샷 재현용이다.
import uuid

# 한국 휴일 때문에 자동 계산 날짜가 틀리면 "2026-09-08"처럼 직접 지정하세요.
PREDICTION_DATE_OVERRIDE = None

LABEL_NAMES = {0: "하락", 1: "보합", 2: "상승"}
PROB_COLS = ["p_down", "p_flat", "p_up"]
# 표에서 모델 순서를 고정한다(성능순 정렬은 잡음을 순위로 보이게 만든다).
MODEL_ORDER = ["Always flat", "Previous ensemble", "No macro ensemble", "Logistic", "LightGBM", "Transformer", "Mean ensemble", "Kronos-small"]

print({
    "QUICK_MODE": QUICK_MODE,
    "TARGET_MODE": TARGET_MODE,
    "BAND_MODE": BAND_MODE,
    "ENSEMBLE_MODELS": ENSEMBLE_MODELS,
    "RUN_TRANSFORMER": RUN_TRANSFORMER,
    "RUN_KRONOS": RUN_KRONOS,
    "COST_BP": COST_BP,
})

## 2. 데이터 다운로드

Colab에서 별도 API 키 없이 재현할 수 있도록 Yahoo Finance를 사용합니다. 일부 티커가 일시적으로 실패해도 나머지 데이터로 계속 진행합니다.

**예측일 오전 7시까지 사용할 신호**

- 삼성전자·KOSPI·KOSPI200·SK하이닉스의 과거 수익률, 추세, 변동성, 거래량
- 런던 삼성전자 GDR(SMSN.IL): 한국 장 마감 후 거래되는 삼성전자 가격의 대리변수
- 요일, 월말 여부 등 달력 변수
- SOX, Nasdaq, S&P 500, Micron, Nvidia, TSMC ADR, EWY
- 원/달러, 달러지수, VIX, 미국 10년물 금리, WTI

Yahoo Finance는 연구·교육용 편의 데이터입니다. 실제 운영에서는 KRX, 한국은행 ECOS, CME 등 원천 자료로 교체하는 것이 좋습니다.


In [ ]:
# KOSPI200(^KS200)은 KOSPI와 사실상 중복이면서(ret_1 상관 0.964, vol_20 상관 0.996)
# Yahoo 시계열에 긴 공백이 생기는 일이 잦아, 그 공백이 전체 학습 구간을 잘라내고
# 라이브 피처를 오염시켰다. 그래서 자산 목록에서 제외했다.
# 대상 종목은 "target", 같은 업종 비교 종목은 "peer"로 부른다.
# GDR(런던 상장 대리증권)은 삼성전자에만 있으므로 대상이 삼성일 때만 넣는다.
ASSETS = {
    "target": TARGET_SPEC["ticker"],
    "kospi": "^KS11",
    "peer": TARGET_SPEC["peer"],
    "sox": "^SOX",
    "nasdaq": "^IXIC",
    "sp500": "^GSPC",
    "micron": "MU",
    "nvidia": "NVDA",
    "tsmc_adr": "TSM",
    "korea_etf": "EWY",
    "usdkrw": "KRW=X",
    "dxy": "DX-Y.NYB",
    "vix": "^VIX",
    "us10y": "^TNX",
    "wti": "CL=F",
}

# 자산군별 "마지막 봉이 마감되었다고 볼 수 있는 시각". 이보다 이른 시각에 실행하면
# Yahoo가 아직 진행 중인 당일 봉을 돌려주므로, 그 봉은 버린다.
if TARGET_SPEC["gdr"]:
    ASSETS["target_gdr"] = TARGET_SPEC["gdr"]

KOREAN_ASSETS = ["target", "kospi", "peer"]
GLOBAL_ASSETS = [
    "sox", "nasdaq", "sp500", "micron", "nvidia", "tsmc_adr",
    "korea_etf", "usdkrw", "dxy", "vix", "us10y", "wti",
] + (["target_gdr"] if TARGET_SPEC["gdr"] else [])
SESSION_CLOSE = {           # (tz, 마감 시각) — 이 시각 이전이면 당일 봉을 미완성으로 본다.
    "korea":  ("Asia/Seoul", 15, 40),
    "us":     ("America/New_York", 16, 5),
    "london": ("Europe/London", 16, 35),
    "cont":   ("America/New_York", 17, 5),   # FX·선물 등 사실상 24시간 시장
}
ASSET_SESSION = {name: "korea" for name in KOREAN_ASSETS}
ASSET_SESSION.update({n: "us" for n in ["sox", "nasdaq", "sp500", "micron", "nvidia", "tsmc_adr", "korea_etf", "us10y"]})
ASSET_SESSION["target_gdr"] = "london"
ASSET_SESSION.update({n: "cont" for n in ["usdkrw", "dxy", "vix", "wti"]})


def _flatten_yf_columns(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    frame.columns = [str(c).strip().lower().replace(" ", "_") for c in frame.columns]
    return frame


def drop_unclosed_last_bar(frame, session):
    """아직 마감되지 않은 당일 봉을 제거한다(학습/서빙 분포 불일치 방지)."""
    if frame.empty:
        return frame
    tz, hour, minute = SESSION_CLOSE[session]
    now_local = pd.Timestamp.now(tz=tz)
    last_date = frame.index.max().date()
    if last_date >= now_local.date() and (now_local.hour, now_local.minute) < (hour, minute):
        return frame.iloc[:-1].copy()
    return frame


def find_placeholder_bars(frame):
    """거래량 0에 시가=고가=저가=종가인 Yahoo 유령봉(실제로는 거래가 있었던 날)."""
    if frame.empty or "volume" not in frame:
        return pd.Series(False, index=frame.index)
    return (frame["volume"] == 0) & (frame["high"] == frame["low"]) & (frame["open"] == frame["close"])


def drop_placeholder_bars(frame, name):
    """삼성전자만 실제로 제거한다.
    유령봉은 target_return을 정확히 0으로 만들어 '보합' 라벨을 조작하므로 타깃 자산에서는
    반드시 빼야 한다. 반면 보조 자산에서 빼면 그 날짜가 NaN이 되어 dropna가 멀쩡한
    삼성 관측치까지 학습에서 날려버리므로(약 100행), 개수만 보고하고 그대로 둔다."""
    bad = find_placeholder_bars(frame)
    n = int(bad.sum())
    if n == 0:
        return frame, []
    dates = [d.date().isoformat() for d in frame.index[bad]]
    if name == "target":
        return frame[~bad].copy(), dates
    return frame, dates


def download_one(ticker, start=START_DATE, end=END_DATE, retries=3):
    last_error = None
    for attempt in range(retries):
        try:
            frame = yf.download(
                ticker, start=start, end=end, auto_adjust=False,
                progress=False, threads=False, timeout=30,
            )
            frame = _flatten_yf_columns(frame)
            if frame.empty:
                raise ValueError("empty response")
            frame.index = pd.to_datetime(frame.index)
            if frame.index.tz is not None:
                frame.index = frame.index.tz_localize(None)
            frame.index = frame.index.normalize()
            frame = frame[~frame.index.duplicated(keep="last")].sort_index()
            if "adj_close" not in frame.columns and "close" in frame.columns:
                frame["adj_close"] = frame["close"]
            keep = [c for c in ["open", "high", "low", "close", "adj_close", "volume"] if c in frame]
            frame = frame[keep].astype(float)
            # Yahoo가 조정계수를 float32로 반올림해 재다운로드마다 미세하게 값이 달라진다.
            # 유효숫자 8자리로 잘라 실행 간 재현성을 확보한다.
            for col in ["open", "high", "low", "close", "adj_close"]:
                if col in frame:
                    frame[col] = frame[col].round(6)
            return frame
        except Exception as exc:
            last_error = exc
            time.sleep(1.5 * (attempt + 1))
    print(f"⚠️ {ticker} 다운로드 실패: {last_error}")
    return pd.DataFrame()


def _cache_write(frame, cache, name):
    """parquet을 우선 쓰되, 엔진이 없으면 CSV로 떨어뜨린다."""
    try:
        frame.to_parquet(cache / f"{name}.parquet")
    except Exception:
        frame.to_csv(cache / f"{name}.csv")


def _cache_read(cache, name):
    parquet, csv = cache / f"{name}.parquet", cache / f"{name}.csv"
    if parquet.exists():
        try:
            return pd.read_parquet(parquet)
        except Exception:
            pass
    if csv.exists():
        return pd.read_csv(csv, index_col=0, parse_dates=True)
    return None


def load_raw(assets, cache_dir=DATA_CACHE_DIR, use_cache=USE_DATA_CACHE):
    """캐시가 있으면 재사용하고, 없으면 내려받아 저장한다(스냅샷 재현성).
    같은 날 저녁에 다시 받기만 해도 Yahoo 값이 미세하게 달라져 백테스트가 흔들리므로,
    한 번 받은 원본을 파일로 남겨 두는 것이 재현의 전제다."""
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    out, from_cache = {}, []
    for name, ticker in tqdm(assets.items(), desc="Loading"):
        if use_cache:
            cached = _cache_read(cache, name)
            if cached is not None:
                out[name] = cached
                from_cache.append(name)
                continue
        frame = download_one(ticker)
        frame = drop_unclosed_last_bar(frame, ASSET_SESSION.get(name, "cont"))
        frame, flagged = drop_placeholder_bars(frame, name)
        if flagged:
            action = "제거" if name == "target" else "발견(유지)"
            print(f"  {name}: 유령봉 {len(flagged)}개 {action} (예: {flagged[:3]})")
        if not frame.empty:
            _cache_write(frame, cache, name)
        out[name] = frame
    if from_cache:
        print(f"캐시에서 로드: {len(from_cache)}개 ({cache}). 새로 받으려면 USE_DATA_CACHE=False.")
    return out


raw = load_raw(ASSETS)

if raw["target"].empty:
    raise RuntimeError("삼성전자 데이터 다운로드에 실패했습니다. 잠시 후 셀을 다시 실행하세요.")

# ---- 자산별 신선도·공백 검증 -------------------------------------------------
# 하나의 보조 시계열에 구멍이 나면 dropna가 최근 구간을 통째로 날리고, 라이브 행에는
# 며칠~몇 주짜리 이동이 "1일 수익률"로 들어간다. 그래서 여기서 먼저 걸러낸다.
MAX_STALE_DAYS = 7            # 마지막 봉이 삼성 마지막 거래일보다 이만큼 오래되면 제외
MAX_MISSING_RECENT = 5        # 최근 250거래일 중 결측 허용 개수

sam_calendar = raw["target"].index
last_samsung_date = sam_calendar.max()
quality, dropped_assets = [], []
for name, frame in list(raw.items()):
    if frame.empty:
        dropped_assets.append((name, "다운로드 실패"))
        continue
    recent = sam_calendar[-250:]
    missing = len(recent.difference(frame.index)) if name in KOREAN_ASSETS else 0
    stale_days = (last_samsung_date - frame.index.max()).days
    reason = None
    if name != "target":
        if stale_days > MAX_STALE_DAYS:
            reason = f"마지막 봉이 {stale_days}일 전"
        elif missing > MAX_MISSING_RECENT:
            reason = f"최근 250거래일 중 {missing}일 결측"
    quality.append({
        "asset": name, "ticker": ASSETS[name], "rows": len(frame),
        "first": frame.index.min(), "last": frame.index.max(),
        "stale_days": stale_days, "missing_recent": missing,
        "status": "제외됨" if reason else "정상",
    })
    if reason:
        dropped_assets.append((name, reason))
        raw.pop(name)

for name, reason in dropped_assets:
    print(f"⚠️ {name} 제외: {reason}")

DATA_SNAPSHOT_HASH = snapshot_hash(raw)
print("데이터 스냅샷 해시:", DATA_SNAPSHOT_HASH)

pd.DataFrame(quality)
# 새 모델 학습이 실패해도, 이미 저장한 예측의 실제값은 먼저 갱신한다.
forecast_log_path = STORAGE_ROOT / "forecast_log.csv"
if forecast_log_path.exists():
    previous_log = pd.read_csv(forecast_log_path)
    updated_log = evaluate_forecasts(previous_log, raw["target"])
    atomic_csv(updated_log, forecast_log_path)
    daily = daily_comparison(updated_log)
    atomic_csv(daily, STORAGE_ROOT / "daily_forecast_comparison.csv")
    atomic_csv(summarize_daily(daily), STORAGE_ROOT / "forecast_accuracy_summary.csv")
    print("기존 예측 실제값 갱신:", updated_log["status"].value_counts().to_dict())

## 3. 특징 생성과 시각 정렬

행의 날짜가 예측 대상 거래일입니다. 삼성전자와 한국시장 신호에는 `shift(1)`을 적용해 전 거래일까지만 사용합니다. 미국시장 날짜 `d`의 종가는 한국시간으로 다음 날 새벽에 확정되므로 가용 날짜를 `d+1일`로 옮긴 뒤 가장 최근 자료를 결합합니다. 주말은 이전 값을 이어받되 미래 값으로 뒤를 채우지 않습니다.


### 월별 경기·반도체 수출 지표
Colab 보안 비밀에 `KOSIS_API_KEY`를 등록하고 노트북 접근을 허용하세요. 키는 파일에 저장하지 않습니다. 또는 로컬 저장 폴더의 `macro_inputs/`에 공식 CSV 두 개를 업로드합니다(README 참조).
선행지수 순환변동치와 반도체 수출액 수준·변화율을 추가합니다. 월+2 첫날부터 정렬하는 보수적 추정이며 실제 발표일을 복원한 데이터가 아닙니다. 과거 값이 수정될 수 있으므로 백테스트는 탐색 결과입니다. `No macro ensemble`과 앞으로 쌓이는 일별 실적으로 추가 효과를 비교하세요.


In [ ]:
macro_data, macro_info = {}, {"enabled": False, "reason": "USE_MACRO_FEATURES=False"}
if USE_MACRO_FEATURES:
    try:
        macro_data, macro_info = load_macro_data(
            STORAGE_ROOT, pd.Timestamp(START_DATE) - pd.DateOffset(years=2),
            END_DATE or pd.Timestamp.now(tz="Asia/Seoul").date(), use_cache=USE_DATA_CACHE)
        macro_info["enabled"] = True
        print("월별 지표 최신월:", macro_info["latest_month"])
        print("⚠️", MACRO_HISTORY_NOTE)
        print("자료별 시점 처리:", macro_info.get("availability_modes", {}))
        print("추가 지표:", macro_info.get("optional_status", {}))
    except Exception as exc:
        if MACRO_STRICT:
            raise
        macro_data = {}
        macro_info = {"enabled": False, "reason": f"{type(exc).__name__}: {exc}"}
        print("=" * 72)
        print("⚠️ 월별 지표(선행지수·반도체 수출)를 사용할 수 없어 시세 모델만으로 진행합니다.")
        print("   사유:", macro_info["reason"])
        print("   → 이 실행의 결과에는 월별 지표가 빠져 있습니다. config.json의")
        print("     macro_enabled=false 와 종합 보고서 상단 안내로도 확인할 수 있습니다.")
        print("   → 반드시 넣어야 한다면 KOSIS_API_KEY를 등록하거나 macro_inputs CSV를 두고")
        print("     다시 실행하세요. 없으면 즉시 중단하려면 MACRO_STRICT=True 로 두세요.")
        print("=" * 72)

# 이후 모든 셀은 '쓰겠다는 설정'이 아니라 '실제로 실려 있는지'를 기준으로 분기한다.
MACRO_ACTIVE = bool(macro_info.get("enabled"))
macro_history_mode = ("release_aware_mixed" if any(
    v == "supplied_release_history" for v in macro_info.get("availability_modes", {}).values())
    else "lagged_latest_vintage")


In [ ]:
def rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def rolling_zscore(series, window=20):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std.replace(0, np.nan)


def safe_pct_change(close, periods=1, max_gap_days=None):
    """시계열에 구멍이 있으면 다기간 수익률이 1일 수익률로 둔갑한다.
    직전 관측이 너무 오래되었으면 NaN으로 만들어 결측으로 드러나게 한다."""
    if max_gap_days is None:
        max_gap_days = 4 * periods + 3
    ret = close.pct_change(periods)
    elapsed = close.index.to_series().diff(periods).dt.days
    return ret.where(elapsed <= max_gap_days)


def _as_ns(index):
    """merge_asof는 양쪽 키의 datetime 단위가 같아야 한다(pandas 3.0에서 us/ns가 섞인다)."""
    return pd.DatetimeIndex(pd.to_datetime(index)).as_unit("ns")


def merge_latest_available(base_index, series, availability_days=1):
    values = series.dropna().copy()
    values.index = (_as_ns(values.index).tz_localize(None).normalize()
                    + pd.Timedelta(days=availability_days))
    values = values.groupby(level=0).last().sort_index()
    left = pd.DataFrame({"date": _as_ns(base_index)}).sort_values("date")
    right = values.rename("value").reset_index()
    right.columns = ["available_date", "value"]
    merged = pd.merge_asof(
        left, right.sort_values("available_date"),
        left_on="date", right_on="available_date",
        direction="backward", tolerance=pd.Timedelta(days=7),
    )
    return pd.Series(merged["value"].to_numpy(), index=left["date"], name=series.name)


def next_krx_session(after):
    """다음 KRX 거래일. exchange_calendars가 있으면 공휴일까지 반영한다."""
    try:
        import exchange_calendars as xcals
        cal = xcals.get_calendar("XKRX")
        return pd.Timestamp(cal.next_session(pd.Timestamp(after))).normalize()
    except Exception:
        print("exchange_calendars 없음: 주말만 제외해 예측일을 계산합니다(공휴일 미반영).")
        return pd.bdate_range(pd.Timestamp(after) + pd.Timedelta(days=1), periods=1)[0]


def krx_sessions_ahead(start, n_sessions):
    """start(포함)부터 n_sessions번째 KRX 거래일."""
    try:
        import exchange_calendars as xcals
        cal = xcals.get_calendar("XKRX")
        return pd.Timestamp(cal.sessions_window(pd.Timestamp(start), n_sessions)[-1]).normalize()
    except Exception:
        return pd.bdate_range(pd.Timestamp(start), periods=n_sessions)[-1]


# sam / sam_* 는 "대상 종목"을 뜻한다(TARGET 설정으로 어떤 종목인지 정해진다).
sam = raw["target"].copy()
sam = sam.dropna(subset=["open", "high", "low", "close", "adj_close"])
last_samsung_date = sam.index.max()

if PREDICTION_DATE_OVERRIDE:
    prediction_date = pd.Timestamp(PREDICTION_DATE_OVERRIDE).normalize()
else:
    prediction_date = next_krx_session(last_samsung_date)

all_dates = sam.index.union(pd.DatetimeIndex([prediction_date])).sort_values()
feat = pd.DataFrame(index=all_dates)

sam_close = sam["adj_close"]          # 타깃과 수익률 계열은 배당조정 종가 기준
sam_raw_close = sam["close"]          # 갭/세션 분해와 표시 가격은 원본 종가 기준
sam_ret = sam_close.pct_change()
sam_gap = sam["open"] / sam_raw_close.shift(1) - 1
sam_range = (sam["high"] - sam["low"]) / sam_raw_close
sam_log_volume = np.log1p(sam["volume"].clip(lower=0))

# 예측일에는 전 거래일까지 확정된 값만 들어간다.
feat["sam_ret_1"] = sam_ret.reindex(all_dates).shift(1)
feat["sam_ret_2"] = sam_close.pct_change(2).reindex(all_dates).shift(1)
feat["sam_ret_5"] = sam_close.pct_change(5).reindex(all_dates).shift(1)
feat["sam_ret_20"] = sam_close.pct_change(20).reindex(all_dates).shift(1)
feat["sam_vol_5"] = sam_ret.rolling(5).std().reindex(all_dates).shift(1)
feat["sam_vol_20"] = sam_ret.rolling(20).std().reindex(all_dates).shift(1)
feat["sam_gap_1"] = sam_gap.reindex(all_dates).shift(1)
feat["sam_range_1"] = sam_range.reindex(all_dates).shift(1)
feat["sam_volume_z20"] = rolling_zscore(sam_log_volume, 20).reindex(all_dates).shift(1)
feat["sam_price_to_ma20"] = (sam_close / sam_close.rolling(20).mean() - 1).reindex(all_dates).shift(1)
feat["sam_price_to_ma60"] = (sam_close / sam_close.rolling(60).mean() - 1).reindex(all_dates).shift(1)
feat["sam_rsi14"] = (rsi(sam_close, 14) / 100).reindex(all_dates).shift(1)

# 한국시장: 같은 날짜의 종가는 오전 7시에 아직 없으므로 1거래일 지연.
# ⚠️ 자산 고유 인덱스가 아니라 "삼성 거래일 달력"에 먼저 맞춘 뒤 차분한다.
#    그렇지 않으면 시계열 구멍이 다기간 수익률을 1일 수익률로 둔갑시킨다.
for name in ["kospi", "peer"]:
    frame = raw.get(name, pd.DataFrame())
    if frame.empty or "adj_close" not in frame:
        continue
    close = frame["adj_close"].reindex(sam.index)
    feat[f"{name}_ret_1"] = safe_pct_change(close, 1).reindex(all_dates).shift(1)
    feat[f"{name}_ret_5"] = safe_pct_change(close, 5).reindex(all_dates).shift(1)
    feat[f"{name}_vol_20"] = safe_pct_change(close, 1).rolling(20).std().reindex(all_dates).shift(1)

# 해외시장: 미국 세션 d의 정보는 한국 날짜 d+1 아침에 사용 가능하다고 보수적으로 정렬.
for name in GLOBAL_ASSETS:
    frame = raw.get(name, pd.DataFrame())
    if frame.empty or "adj_close" not in frame:
        continue
    close = frame["adj_close"]
    feat[f"{name}_ret_1"] = merge_latest_available(all_dates, safe_pct_change(close, 1), 1)
    feat[f"{name}_ret_5"] = merge_latest_available(all_dates, safe_pct_change(close, 5), 1)
    if name in ["vix", "us10y", "usdkrw", "dxy", "wti"]:
        feat[f"{name}_level_z60"] = merge_latest_available(all_dates, rolling_zscore(close, 60), 1)

# 런던 GDR 세션 d 수익률 - 삼성전자 세션 d 수익률: 한국 장 마감 이후의 가격 변화(야간 신호).
if "target_gdr_ret_1" in feat:
    feat["gdr_overnight_signal"] = feat["target_gdr_ret_1"] - feat["sam_ret_1"]

# 달력 변수: 예측일 자체의 속성이므로 shift가 필요 없다.
feat["cal_dow"] = feat.index.dayofweek.astype(float)
feat["cal_is_monday"] = (feat.index.dayofweek == 0).astype(float)
feat["cal_is_friday"] = (feat.index.dayofweek == 4).astype(float)
feat["cal_month_end"] = (feat.index.day >= 26).astype(float)

# 발표 빈도가 다른 자료를 합칠 때 미래값으로 backfill하지 않는다.
feat = feat.replace([np.inf, -np.inf], np.nan).ffill(limit=5)

# 월별 자료는 일별 ffill 뒤에 별도 결합한다. 오래된 지표를 무기한 메우지 않는다.
macro_feature_cols = []
if MACRO_ACTIVE:
    # 예측 시각 뒤에 발표된 자료는 당일 오전 예측에서 제외한다.
    macro_prediction_hour = 9 if TARGET_MODE == "open_to_close" else 7
    monthly_daily = pd.DataFrame(index=all_dates)
    for series, monthly_frame in macro_data.items():
        series_features = macro_features({series: monthly_frame}, all_dates,
                                         prediction_hour=macro_prediction_hour)
        if series in OPTIONAL_MACRO_SERIES:
            coverage_ok = (series_features.loc[sam.index].notna().mean() >= .80).all()
            live_ok = series_features.loc[prediction_date].notna().all()
            if not coverage_ok or not live_ok:
                reason = "insufficient_history" if not coverage_ok else "stale_or_missing"
                macro_info.setdefault("optional_status", {})[series] = "excluded:" + reason
                print(f"⚠️ 추가 지표 {series} 제외: {reason}")
                continue
        monthly_daily = monthly_daily.join(series_features)
    macro_info["active_feature_cols"] = monthly_daily.columns.tolist()
    macro_feature_cols = monthly_daily.columns.tolist()
    feat = feat.join(monthly_daily)
    if feat.loc[prediction_date, macro_feature_cols].isna().any():
        stale = "최신 월별 지표가 누락되거나 오래되었습니다. KOSIS/CSV를 갱신하세요."
        if MACRO_STRICT:
            raise ValueError(stale)
        print("⚠️", stale, "→ 이번 실행에서는 월별 지표를 빼고 진행합니다.")
        feat = feat.drop(columns=macro_feature_cols)
        macro_feature_cols = []
        macro_data, macro_info = {}, {"enabled": False, "reason": stale}
        MACRO_ACTIVE = False

# 시가→종가 모드: 당일 시가는 09:00에 확정되므로 당일 갭을 특징으로 쓸 수 있다.
if TARGET_MODE == "open_to_close":
    feat["sam_gap_0"] = sam_gap.reindex(all_dates)

if TARGET_MODE == "open_to_close":
    target_return = (sam_raw_close / sam["open"] - 1).reindex(all_dates)
elif TARGET_MODE == "close_to_close":
    target_return = sam_close.pct_change().reindex(all_dates)
else:
    raise ValueError(f"Unknown TARGET_MODE: {TARGET_MODE}")

# 보합 밴드. vol_scaled는 전일까지의 변동성만 사용하므로 예측 시점에 알 수 있는 값이다.
if BAND_MODE == "vol_scaled":
    band_source = target_return if TARGET_MODE == "open_to_close" else sam_ret
    band = (VOL_BAND_MULT * band_source.rolling(20).std()).reindex(all_dates).shift(1)
elif BAND_MODE == "fixed":
    band = pd.Series(NEUTRAL_BAND, index=all_dates, dtype=float)
else:
    raise ValueError(f"Unknown BAND_MODE: {BAND_MODE}")
feat["band"] = band


def label_from_return(ret, band_value):
    """수익률과 밴드로부터 3-class 라벨. 사후 채점에서도 같은 함수를 쓴다."""
    ret = np.asarray(ret, dtype=float)
    band_value = np.asarray(band_value, dtype=float)
    out = np.select([ret < -band_value, ret > band_value], [0, 2], default=1).astype(float)
    out[np.isnan(ret) | np.isnan(band_value)] = np.nan
    return out


feat["target_return"] = target_return
feat["target"] = label_from_return(target_return.to_numpy(), band.to_numpy())

# 갭 / 세션 분해 — 평가에서만 쓰며 특징으로는 절대 넣지 않는다.
# ⚠️ 원본 close 기준이어야 (1+gap)(1+session)이 원본 종가→종가 수익률과 일치한다.
decomp = pd.DataFrame({
    "gap": (sam["open"] / sam_raw_close.shift(1) - 1),
    "session": (sam_raw_close / sam["open"] - 1),
})

# 결측이 지나치게 많은 특징은 제거한다.
NON_FEATURE_COLS = ["target", "target_return", "band"]
candidate_features = [c for c in feat.columns if c not in NON_FEATURE_COLS]
coverage = feat.loc[sam.index, candidate_features].notna().mean()
feature_cols = coverage[coverage >= 0.80].index.tolist()
removed_features = coverage[coverage < 0.80].index.tolist()
if set(macro_feature_cols) - set(feature_cols):
    raise ValueError("월별 지표의 과거 자료가 부족합니다. START_DATE보다 2년 이전부터 두 지표를 조회하세요.")

model_df = feat.loc[sam.index, feature_cols + NON_FEATURE_COLS].dropna().copy()
model_df["target"] = model_df["target"].astype(int)

# ---- 잘려나간 최근 구간이 있는지 반드시 확인한다 ----------------------------
trailing = feat.loc[sam.index].loc[model_df.index.max():].iloc[1:]
if len(trailing):
    blocking = (
        feat.loc[trailing.index, feature_cols].isna().sum()
        .sort_values(ascending=False).head(5)
    )
    print(f"⚠️ dropna로 최근 {len(trailing)}행이 제외되었습니다. 원인 컬럼:")
    print(blocking[blocking > 0].to_string())
staleness_days = (last_samsung_date - model_df.index.max()).days
assert staleness_days <= 10, (
    f"학습 데이터가 {staleness_days}일 뒤처져 있습니다. 위 원인 컬럼을 제거하거나 보정하세요."
)

live_row = feat.loc[[prediction_date], feature_cols].copy()
live_band = float(feat.loc[prediction_date, "band"]) if pd.notna(feat.loc[prediction_date, "band"]) else float(model_df["band"].iloc[-1])
if TARGET_MODE == "open_to_close":
    if LIVE_OPEN_PRICE is None:
        raise ValueError(
            "TARGET_MODE='open_to_close'에서는 예측일 09:00 시가(LIVE_OPEN_PRICE)가 필요합니다. "
            "갭을 0으로 가정하면 백테스트가 검증한 것과 다른 예측이 됩니다."
        )
    live_row.loc[prediction_date, "sam_gap_0"] = float(LIVE_OPEN_PRICE) / float(sam_raw_close.iloc[-1]) - 1

# 라이브 행의 결측을 과거값으로 메울 때는 무엇을 언제 값으로 메웠는지 반드시 남긴다.
imputed_live_features = {}
for col in feature_cols:
    if pd.isna(live_row.iloc[0][col]):
        source_date = model_df[col].last_valid_index()
        live_row.loc[prediction_date, col] = model_df[col].iloc[-1]
        imputed_live_features[col] = str(source_date.date()) if source_date is not None else None
if imputed_live_features:
    print("⚠️ 라이브 행에서 과거값으로 대체된 특징:", imputed_live_features)

# 라이브 특징이 학습 분포를 크게 벗어나면 데이터 문제일 가능성이 높다.
lo = model_df[feature_cols].quantile(0.001)
hi = model_df[feature_cols].quantile(0.999)
outliers = [c for c in feature_cols if not (lo[c] <= live_row.iloc[0][c] <= hi[c])]
if outliers:
    print("⚠️ 라이브 특징이 학습 분포의 0.1~99.9% 범위를 벗어남:", outliers)

print("Model rows:", len(model_df))
print("Features:", len(feature_cols))
print("Removed low-coverage features:", removed_features)
print("Backtest range:", model_df.index.min().date(), "~", model_df.index.max().date())
print("Last Samsung bar:", last_samsung_date.date(), f"(학습 데이터 지연 {staleness_days}일)")
print("Candidate prediction date:", prediction_date.date())
print("Target mode:", TARGET_MODE, "| Band mode:", BAND_MODE, "| Live band: ±{:.2%}".format(live_band))

assert prediction_date > last_samsung_date
assert model_df.index.is_monotonic_increasing
assert model_df[feature_cols].notna().all().all()

In [ ]:
class_counts = model_df["target"].value_counts().sort_index()
class_table = pd.DataFrame({
    "class": [LABEL_NAMES[i] for i in class_counts.index],
    "count": class_counts.values,
    "ratio": class_counts.values / class_counts.sum(),
}, index=class_counts.index)
display(class_table.style.format({"ratio": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_table.set_index("class")["count"].plot.bar(ax=axes[0], color=["#d95f5f", "#999999", "#4c78a8"])
axes[0].set_title("Target class distribution")
axes[0].set_xlabel("")
model_df["target_return"].clip(-0.08, 0.08).hist(bins=60, ax=axes[1], color="#4c78a8")
median_band = float(model_df["band"].median())
axes[1].axvline(-median_band, color="black", ls="--", label=f"median band ±{median_band:.2%}")
axes[1].axvline(median_band, color="black", ls="--")
axes[1].legend()
axes[1].set_title(f"Samsung {TARGET_MODE} return")
plt.tight_layout()
plt.show()


## 4. 워크포워드 검증과 공통 평가함수

각 폴드는 과거 5년을 학습하고 다음 6개월을 예측합니다. `QUICK_MODE=True`에서는 가장 최근 3개 폴드만 실행합니다. 모든 모델의 확률 열 순서는 `[하락, 보합, 상승]`으로 고정합니다.


In [ ]:
X = model_df[feature_cols].to_numpy(dtype=np.float32)
y = model_df["target"].to_numpy(dtype=np.int64)
dates = model_df.index
GAP = decomp["gap"].reindex(dates)
SESSION = decomp["session"].reindex(dates)


def make_walk_forward_folds(dates_index):
    folds = []
    start = pd.Timestamp(FIRST_TEST_DATE)
    max_date = dates_index.max()
    fold_no = 0
    while start <= max_date:
        end = start + pd.DateOffset(months=TEST_MONTHS)
        train_start = start - pd.DateOffset(years=ROLLING_TRAIN_YEARS)
        train_idx = np.where((dates_index >= train_start) & (dates_index < start))[0]
        test_idx = np.where((dates_index >= start) & (dates_index < end))[0]
        if len(train_idx) >= 500 and len(test_idx) >= 20:
            folds.append({
                "fold": fold_no, "train_idx": train_idx, "test_idx": test_idx,
                "train_start": dates_index[train_idx[0]], "train_end": dates_index[train_idx[-1]],
                "test_start": dates_index[test_idx[0]], "test_end": dates_index[test_idx[-1]],
            })
            fold_no += 1
        start = end
    if MAX_FOLDS is not None:
        folds = folds[-MAX_FOLDS:]
    return folds


folds = make_walk_forward_folds(dates)
if not folds:
    raise RuntimeError("워크포워드 폴드가 없습니다. START_DATE 또는 FIRST_TEST_DATE를 확인하세요.")

display(pd.DataFrame([{k: v for k, v in f.items() if not k.endswith("idx")} for f in folds]))


def multiclass_brier(y_true, probs):
    onehot = np.eye(3)[np.asarray(y_true, dtype=int)]
    return np.mean(np.sum((np.asarray(probs) - onehot) ** 2, axis=1))


def prediction_frame(model_name, row_dates, y_true, probs, fold_id=None, y_pred=None, extra=None):
    probs = np.asarray(probs, dtype=float)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    probs = probs / probs.sum(axis=1, keepdims=True)
    out = pd.DataFrame({
        "date": pd.to_datetime(row_dates),
        "y_true": np.asarray(y_true, dtype=int),
        "y_pred": probs.argmax(axis=1) if y_pred is None else np.asarray(y_pred, dtype=int),
        "p_down": probs[:, 0], "p_flat": probs[:, 1], "p_up": probs[:, 2],
        "model": model_name, "fold": fold_id,
    })
    if extra:
        for key, value in extra.items():
            out[key] = value
    return out


# ---- 월 블록 부트스트랩 ------------------------------------------------------
# 일별 예측은 서로 독립이 아니므로, 달력 월을 블록으로 복원추출해 신뢰구간을 만든다.
def month_blocks(date_index):
    key = pd.PeriodIndex(pd.DatetimeIndex(date_index), freq="M")
    return [np.where(key == m)[0] for m in key.unique()]


def block_bootstrap_ci(date_index, stat_fn, b=BOOTSTRAP_B, seed=SEED, alpha=0.05):
    """stat_fn(row_positions) -> 스칼라. (하한, 상한) 반환."""
    blocks = month_blocks(date_index)
    if not blocks:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(b):
        pick = rng.integers(0, len(blocks), len(blocks))
        idx = np.concatenate([blocks[i] for i in pick])
        try:
            draws.append(stat_fn(idx))
        except Exception:
            continue
    if not draws:
        return (np.nan, np.nan)
    return tuple(np.percentile(draws, [100 * alpha / 2, 100 * (1 - alpha / 2)]))


def metric_row(frame):
    true = frame["y_true"].to_numpy(dtype=int)
    pred = frame["y_pred"].to_numpy(dtype=int)
    probs = frame[PROB_COLS].to_numpy(dtype=float)
    return {
        "n": len(frame),
        "accuracy": accuracy_score(true, pred),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
        "macro_f1": f1_score(true, pred, average="macro", zero_division=0),
        "log_loss": log_loss(true, probs, labels=[0, 1, 2]),
        "brier": multiclass_brier(true, probs),
    }


def _auc_vs_sign(score, leg):
    """방향 점수가 특정 구간(갭 또는 세션)의 부호를 얼마나 맞히는가."""
    ok = leg.notna().to_numpy() & (leg.to_numpy() != 0)
    if ok.sum() < 30:
        return np.nan
    return roc_auc_score((leg.to_numpy()[ok] > 0).astype(int), np.asarray(score)[ok])


def decomposition_row(frame, cost_bp=COST_BP):
    """모델이 실제로 무엇을 예측하는지: 갭 vs 세션, 그리고 구간별 기대손익(bp)."""
    s = frame.set_index("date").sort_index()
    score = (s["p_up"] - s["p_down"]).to_numpy()
    gap = GAP.reindex(s.index)
    ses = SESSION.reindex(s.index)
    direction = np.sign(score)
    gap_bp = float(np.nanmean(direction * gap.to_numpy())) * 1e4
    ses_bp = float(np.nanmean(direction * ses.to_numpy())) * 1e4
    lo, hi = block_bootstrap_ci(
        s.index,
        lambda idx: float(np.nanmean(direction[idx] * ses.to_numpy()[idx])) * 1e4,
    )
    return {
        "auc_gap": _auc_vs_sign(score, gap),
        "auc_session": _auc_vs_sign(score, ses),
        "gap_bp": gap_bp,
        "session_bp": ses_bp,
        "session_bp_lo": lo,
        "session_bp_hi": hi,
        "session_bp_net": ses_bp - cost_bp,
    }


def summarize_predictions(predictions, with_ci=True, with_decomposition=False):
    rows = []
    for name, group in predictions.groupby("model"):
        row = {"model": name, **metric_row(group)}
        if with_ci:
            g = group.sort_values("date")
            true = g["y_true"].to_numpy(dtype=int)
            pred = g["y_pred"].to_numpy(dtype=int)
            probs = g[PROB_COLS].to_numpy(dtype=float)
            row["bal_acc_lo"], row["bal_acc_hi"] = block_bootstrap_ci(
                g["date"], lambda i: balanced_accuracy_score(true[i], pred[i]))
            row["log_loss_lo"], row["log_loss_hi"] = block_bootstrap_ci(
                g["date"], lambda i: log_loss(true[i], probs[i], labels=[0, 1, 2]))
        if with_decomposition:
            row.update(decomposition_row(group))
        rows.append(row)
    out = pd.DataFrame(rows).set_index("model")
    order = [m for m in MODEL_ORDER if m in out.index] + [m for m in out.index if m not in MODEL_ORDER]
    return out.loc[order]


def paired_delta_ci(predictions, model_a, model_b, metric="balanced_accuracy"):
    """같은 날짜에서 두 모델의 지표 차이와 95% 신뢰구간(월 블록 부트스트랩)."""
    a = predictions[predictions["model"] == model_a].set_index("date").sort_index()
    b = predictions[predictions["model"] == model_b].set_index("date").sort_index()
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    if len(common) < 60:
        return {"delta": np.nan, "lo": np.nan, "hi": np.nan, "n": len(common)}

    def stat(idx):
        fa, fb = a.iloc[idx], b.iloc[idx]
        if metric == "accuracy":
            return float(np.mean(fa["y_true"].to_numpy() == fa["y_pred"].to_numpy())
                         - np.mean(fb["y_true"].to_numpy() == fb["y_pred"].to_numpy()))
        if metric == "balanced_accuracy":
            return (balanced_accuracy_score(fa["y_true"], fa["y_pred"])
                    - balanced_accuracy_score(fb["y_true"], fb["y_pred"]))
        return (log_loss(fa["y_true"], fa[PROB_COLS].to_numpy(), labels=[0, 1, 2])
                - log_loss(fb["y_true"], fb[PROB_COLS].to_numpy(), labels=[0, 1, 2]))

    delta = stat(np.arange(len(common)))
    lo, hi = block_bootstrap_ci(common, stat)
    return {"delta": delta, "lo": lo, "hi": hi, "n": len(common)}


def folds_worse_than_prior(predictions, model_name):
    """폴드별 log loss가 '항상 보합'(클래스 사전확률)보다 나쁜 횟수."""
    g = predictions[predictions["model"] == model_name]
    base = predictions[predictions["model"] == "Always flat"].set_index(["fold", "date"])
    worse = 0
    for fold_id, grp in g.groupby("fold"):
        idx = pd.MultiIndex.from_arrays([grp["fold"], grp["date"]])
        try:
            ref = base.loc[idx]
        except KeyError:
            continue
        ll_model = log_loss(grp["y_true"], grp[PROB_COLS].to_numpy(), labels=[0, 1, 2])
        ll_base = log_loss(ref["y_true"], ref[PROB_COLS].to_numpy(), labels=[0, 1, 2])
        worse += int(ll_model >= ll_base)
    return worse

## 5. 기준선 · 로지스틱 회귀 · LightGBM

복잡한 모델은 반드시 단순 기준선보다 미래 구간에서 좋아야 합니다. 클래스 불균형을 고려해 로지스틱과 LightGBM 모두 균형 가중치를 사용합니다.


In [ ]:
# 셀을 다시 실행해도 예측 프레임이 중복 적재되지 않도록, 단계별 결과를 딕셔너리에
# 보관하고 매번 새로 조립한다.
PREDICTION_PARTS = {}


def assemble_predictions():
    parts = [p for p in PREDICTION_PARTS.values() if p is not None and len(p)]
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    assert not out.duplicated(["date", "model"]).any(), "예측 프레임에 중복 행이 있습니다."
    return out


def make_logistic():
    # C=0.25는 2026년 고변동 국면에서 과신이 심했다(마지막 폴드 log loss 1.25 > 사전확률 1.10).
    # C를 낮춰도 balanced accuracy는 그대로이고 확률 품질만 좋아진다.
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=0.01, max_iter=3000,
                                     class_weight="balanced", random_state=SEED)),
    ])


def make_lgbm(n_estimators=60, num_leaves=7):
    # 300트리 × 15리프는 그리드에서 가장 나쁜 조합이었고 클래스 사전확률보다도 log loss가
    # 높았다(1.099 vs 1.097). 60×7이 검증에서 최적이었다.
    # subsample은 subsample_freq를 켜지 않으면 아무 효과가 없으므로 아예 제거했다.
    return LGBMClassifier(
        objective="multiclass", num_class=3,
        n_estimators=n_estimators, learning_rate=0.03, num_leaves=num_leaves,
        min_child_samples=50, colsample_bytree=0.85,
        reg_alpha=0.5, reg_lambda=2.0, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )


tabular_predictions = []
lgb_importances = []
model_selection_rows = []
market_feature_idx = [i for i, c in enumerate(feature_cols) if not c.startswith("macro_")]
market_X = X[:, market_feature_idx]

for fold in tqdm(folds, desc="Tabular walk-forward"):
    tr, te = fold["train_idx"], fold["test_idx"]

    # 기준선: 학습구간의 클래스 사전확률을 그대로 확률로 쓰고, 하드 예측만 보합으로 고정한다.
    # (확률을 왜곡해 argmax를 맞추면 기준선의 log loss가 부풀려져 잣대가 어긋난다.)
    prior = np.bincount(y[tr], minlength=3).astype(float) + 1
    prior = prior / prior.sum()
    tabular_predictions.append(prediction_frame(
        "Always flat", dates[te], y[te], np.tile(prior, (len(te), 1)), fold["fold"],
        y_pred=np.ones(len(te), dtype=int),
    ))

    # 기존 고정설정을 동일한 외부 폴드에서 비교해 개선 여부를 확인한다.
    old_log = make_logistic().fit(market_X[tr], y[tr])
    old_lgb = make_lgbm().fit(market_X[tr], y[tr])
    old_mean = (aligned_probabilities(old_log, market_X[te]) + aligned_probabilities(old_lgb, market_X[te])) / 2
    tabular_predictions.append(prediction_frame("Previous ensemble", dates[te], y[te], old_mean, fold["fold"]))
    if MACRO_ACTIVE:
        no_macro_probs = []
        for family in ["Logistic", "LightGBM"]:
            fitted_market = fit_direction_model(market_X, y, tr, family, seed=SEED)
            no_macro_probs.append(predict_direction_model(fitted_market, market_X[te]))
            model_selection_rows.append({"fold": fold["fold"], "features": "market_only", **fitted_market["selection"]})
        tabular_predictions.append(prediction_frame("No macro ensemble", dates[te], y[te],
                                                    np.mean(no_macro_probs, axis=0), fold["fold"]))
    for family in ["Logistic", "LightGBM"]:
        fitted = fit_direction_model(X, y, tr, family, seed=SEED)
        probs = predict_direction_model(fitted, X[te])
        tabular_predictions.append(prediction_frame(family, dates[te], y[te], probs, fold["fold"]))
        model_selection_rows.append({"fold": fold["fold"], **fitted["selection"]})
        if family == "LightGBM":
            lgb_importances.append(fitted["estimator"].feature_importances_)

PREDICTION_PARTS["tabular"] = pd.concat(tabular_predictions, ignore_index=True)
predictions = assemble_predictions()
display(summarize_predictions(predictions, with_ci=False).style.format("{:.4f}", na_rep="—"))

## 6. 직접 학습하는 Transformer Encoder

30거래일의 특징 시퀀스를 입력받아 마지막 토큰에서 세 방향의 확률을 출력합니다. 각 폴드마다 scaler와 모델을 과거 데이터로만 다시 학습합니다. 작은 데이터에서 대형 Transformer는 과적합하기 쉬우므로 의도적으로 작은 구조를 사용합니다.


In [ ]:
if RUN_TRANSFORMER and TORCH_AVAILABLE:

    class DirectionTransformer(nn.Module):
        def __init__(self, n_features, seq_len, d_model=64, nhead=4, num_layers=2, dropout=0.20):
            super().__init__()
            self.seq_len = seq_len
            self.input_dropout = nn.Dropout(0.30)   # 입력 특징 단위 dropout
            self.input_proj = nn.Linear(n_features, d_model)
            # 고정 사인 위치 인코딩(학습 파라미터보다 소표본에서 안정적)
            position = torch.arange(seq_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe = torch.zeros(1, seq_len, d_model)
            pe[0, :, 0::2] = torch.sin(position * div)
            pe[0, :, 1::2] = torch.cos(position * div)
            self.register_buffer("position", pe)
            layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
                dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
            self.norm = nn.LayerNorm(d_model)
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model // 2), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(d_model // 2, 3),
            )

        def forward(self, x):
            z = self.input_proj(self.input_dropout(x)) + self.position[:, :x.size(1)]
            z = self.encoder(z)
            return self.head(self.norm(z[:, -1]))


    def sequences_for_indices(X_scaled, y_array, target_indices, seq_len):
        seq_x, seq_y, kept = [], [], []
        for idx in target_indices:
            if idx < seq_len - 1:
                continue
            seq_x.append(X_scaled[idx - seq_len + 1: idx + 1])
            seq_y.append(y_array[idx])
            kept.append(idx)
        if not seq_x:
            return np.empty((0, seq_len, X_scaled.shape[1])), np.array([]), np.array([])
        return np.asarray(seq_x, dtype=np.float32), np.asarray(seq_y, dtype=np.int64), np.asarray(kept)


    def _fit_epochs(X_seq, y_seq, n_features, epochs, seed, val=None):
        """지정한 epoch 수만큼 학습한다. val이 주어지면 조기종료 이력도 함께 반환."""
        torch.manual_seed(seed)          # 전역 RNG에 의존하지 않도록 매번 명시적으로 시드
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        generator = torch.Generator().manual_seed(seed)
        loader = DataLoader(
            TensorDataset(torch.from_numpy(X_seq), torch.from_numpy(y_seq)),
            batch_size=BATCH_SIZE, shuffle=True, generator=generator, drop_last=False,
        )
        model = DirectionTransformer(n_features, SEQ_LEN).to(DEVICE)
        counts = np.bincount(y_seq, minlength=3).astype(float)
        weights = len(y_seq) / (3 * np.maximum(counts, 1))
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
        optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-3)

        val_loader = None
        if val is not None:
            val_loader = DataLoader(
                TensorDataset(torch.from_numpy(val[0]), torch.from_numpy(val[1])),
                batch_size=BATCH_SIZE * 2, shuffle=False,
            )
        best_state, best_val, best_epoch = None, np.inf, epochs
        patience_left = TRANSFORMER_PATIENCE
        history = []
        for epoch in range(epochs):
            model.train()
            train_losses = []
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_losses.append(loss.item())
            train_loss = float(np.mean(train_losses))
            if val_loader is None:
                history.append((epoch + 1, train_loss, np.nan))
                continue
            model.eval()
            val_losses = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    val_losses.append(criterion(model(xb), yb).item())
            val_loss = float(np.mean(val_losses)) if val_losses else train_loss
            history.append((epoch + 1, train_loss, val_loss))
            if val_loss < best_val - 1e-4:
                best_val, best_epoch = val_loss, epoch + 1
                best_state = deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
                patience_left = TRANSFORMER_PATIENCE
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        return model, best_epoch, history


    def train_transformer(X_array, y_array, train_indices, seed=SEED, epochs=TRANSFORMER_EPOCHS):
        """조기종료로 최적 epoch를 정한 뒤, 학습 구간 전체로 다시 적합한다.
        (기존 코드는 마지막 15%를 검증에만 쓰고 학습에 넣지 않아, 실제로는 테스트
        구간보다 9개월 이른 데이터까지만 학습된 모델을 사용했다.)"""
        scaler = StandardScaler().fit(X_array[train_indices])
        X_scaled = scaler.transform(X_array).astype(np.float32)

        valid_targets = train_indices[train_indices >= SEQ_LEN - 1]
        cut = max(1, int(len(valid_targets) * 0.85))
        fit_targets, val_targets = valid_targets[:cut], valid_targets[cut:]
        if len(val_targets) < 10:
            val_targets = valid_targets[-max(10, len(valid_targets) // 10):]
            fit_targets = valid_targets[:len(valid_targets) - len(val_targets)]

        X_fit, y_fit, _ = sequences_for_indices(X_scaled, y_array, fit_targets, SEQ_LEN)
        X_val, y_val, _ = sequences_for_indices(X_scaled, y_array, val_targets, SEQ_LEN)
        _, best_epoch, history = _fit_epochs(
            X_fit, y_fit, X_array.shape[1], epochs, seed, val=(X_val, y_val))

        # 최적 epoch 수를 알았으니 검증 구간까지 포함해 다시 학습한다.
        X_all, y_all, _ = sequences_for_indices(X_scaled, y_array, valid_targets, SEQ_LEN)
        model, _, _ = _fit_epochs(X_all, y_all, X_array.shape[1], best_epoch, seed + 1)
        return model, scaler, X_scaled, history, best_epoch


    def transformer_probs(model, X_scaled, target_indices):
        dummy_y = np.zeros(len(X_scaled), dtype=np.int64)
        X_seq, _, kept = sequences_for_indices(X_scaled, dummy_y, target_indices, SEQ_LEN)
        loader = DataLoader(torch.from_numpy(X_seq), batch_size=BATCH_SIZE * 2, shuffle=False)
        probs = []
        model.eval()
        with torch.no_grad():
            for xb in loader:
                probs.append(torch.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy())
        return np.vstack(probs), kept

else:
    print("RUN_TRANSFORMER=False 또는 torch 없음: Transformer 섹션을 건너뜁니다.")

In [ ]:
if RUN_TRANSFORMER and TORCH_AVAILABLE:
    transformer_predictions, transformer_histories = [], []
    for fold in tqdm(folds, desc="Transformer walk-forward"):
        tr, te = fold["train_idx"], fold["test_idx"]
        model_t, scaler_t, X_scaled_t, hist_t, best_ep = train_transformer(
            X, y, tr, seed=SEED + fold["fold"])
        probs_t, kept_t = transformer_probs(model_t, X_scaled_t, te)
        transformer_predictions.append(prediction_frame(
            "Transformer", dates[kept_t], y[kept_t], probs_t, fold["fold"]))
        transformer_histories.append(
            pd.DataFrame(hist_t, columns=["epoch", "train_loss", "val_loss"]).assign(fold=fold["fold"]))
        del model_t
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    PREDICTION_PARTS["transformer"] = pd.concat(transformer_predictions, ignore_index=True)
    predictions = assemble_predictions()
    display(summarize_predictions(predictions, with_ci=False).style.format("{:.4f}", na_rep="—"))

    history_df = pd.concat(transformer_histories, ignore_index=True)
    sns.lineplot(data=history_df, x="epoch", y="val_loss", hue="fold", marker="o")
    plt.title("Transformer validation loss by fold")
    plt.show()
else:
    PREDICTION_PARTS["transformer"] = None
    predictions = assemble_predictions()

## 6.5 앙상블

`ENSEMBLE_MODELS`에 지정한 모델들의 확률을 **단순 평균**합니다.

이전 버전에는 OOF 스태킹 메타모델과 `exp(-2 × Δlog loss)` 성과가중 앙상블이 있었지만, 검증 결과 둘 다 제거했습니다.

- **스태킹**: 메타모델이 폴드에 따라 123행으로 학습되어 단순 평균보다 나빴습니다(log loss 1.0507 vs 1.0379).
- **성과가중**: 0.03 nats 차이 위에서 계산된 가중치라 사실상 균등 평균(0.32~0.34)이었고, 판별력을 잃고 사전확률에 가까워진 모델에 가장 큰 가중을 주는 부작용이 있었습니다.

정규화를 바로잡은 Logistic + LightGBM의 단순 평균이, 이전의 3모델 평균·스태킹보다 좋거나 같으면서 훨씬 단순합니다.

In [ ]:
# 검증 결과 스태킹 메타모델(폴드에 따라 123행으로 학습)은 단순 평균보다 나빴고
# (log loss 1.0507 vs 1.0379), exp(-2·Δlog loss) 성과가중은 0.03 nats 차이 위에서
# 계산되어 사실상 균등 평균이었다. 그래서 둘 다 제거하고 단순 평균만 남긴다.
available = [m for m in ENSEMBLE_MODELS if m in set(predictions["model"])]
if len(available) < 2:
    print("⚠️ 앙상블을 만들 기본 모델이 부족합니다:", available)
    PREDICTION_PARTS["ensemble"] = None
else:
    wide = (
        predictions[predictions["model"].isin(available)]
        .pivot_table(index="date", columns="model", values=PROB_COLS)
        .dropna()
    )
    stacked = np.stack([wide[[(p, m) for p in PROB_COLS]].to_numpy() for m in available], axis=1)
    p_mean = stacked.mean(axis=1)
    ens_dates = pd.DatetimeIndex(wide.index)
    fold_by_date = (
        predictions[predictions["model"] == available[0]]
        .set_index("date")["fold"].reindex(ens_dates)
    )
    PREDICTION_PARTS["ensemble"] = prediction_frame(
        "Mean ensemble", ens_dates,
        model_df.loc[ens_dates, "target"].to_numpy(dtype=int),
        p_mean, fold_by_date.to_numpy(),
    )
    print(f"Mean ensemble = {' + '.join(available)} 확률의 단순 평균 ({len(ens_dates)}일)")

predictions = assemble_predictions()

native_metrics = summarize_predictions(predictions, with_ci=False)
display(native_metrics[["n", "accuracy", "balanced_accuracy", "macro_f1", "log_loss", "brier"]]
        .style.format("{:.4f}", na_rep="—"))

# 폴드별 성능: 우위가 여러 구간에서 반복되는지 확인
fold_table = (
    predictions.groupby(["fold", "model"], group_keys=False)[["y_true", "y_pred", *PROB_COLS]]
    .apply(lambda g: pd.Series(metric_row(g)))
    .reset_index()
    .pivot(index="fold", columns="model", values="balanced_accuracy")
)
cols = [m for m in MODEL_ORDER if m in fold_table.columns]
display(fold_table[cols].style.format("{:.3f}", na_rep="—")
        .set_caption("Balanced accuracy by walk-forward fold"))

## 7. 금융 시계열 파운데이션 모델 Kronos-small (기본 꺼짐)

Kronos는 OHLCV를 금융시장 전용 토큰으로 변환한 뒤 다음 K-line을 생성합니다. 삼성전자에 별도 학습하지 않은 **zero-shot** 성능을 확인합니다.

**기본값이 `RUN_KRONOS = False`인 이유**: 60일 평가에서 log loss 1.87 / Brier 1.07로 "항상 보합" 기준선(1.10 / 0.67)보다 나빴고, 이미 앙상블에서도 제외되어 있었습니다. 반면 이 섹션 하나 때문에 git clone + HuggingFace 가중치라는 무거운 의존성이 붙습니다.

켜서 실행하려면 `RUN_KRONOS = True`로 두되, 다음을 유의하세요.

- 8샘플 Monte Carlo 빈도를 확률로 쓰므로 log loss는 표본 잡음에 지배됩니다.
- 60일 평가로는 예측력의 유무를 판정할 수 없습니다(무작위 예측의 60일 balanced accuracy 표준편차가 약 0.065). 최소 250일 이상을 권장합니다.

In [ ]:
kronos_predictor = None
kronos_predictions = pd.DataFrame()

if RUN_KRONOS:
    if not TORCH_AVAILABLE:
        print("torch가 없어 Kronos를 건너뜁니다.")
        RUN_KRONOS = False
    else:
        try:
            import subprocess
            subprocess.run(
                "pip -q install einops==0.8.1 huggingface_hub==0.33.1 tqdm==4.67.1 safetensors==0.6.2",
                shell=True, check=False,
            )
            if not Path("/content/Kronos").exists():
                subprocess.run(
                    "git clone --depth 1 https://github.com/shiyu-coder/Kronos.git /content/Kronos",
                    shell=True, check=False,
                )
            if "/content/Kronos" not in sys.path:
                sys.path.append("/content/Kronos")
            from model import Kronos, KronosTokenizer, KronosPredictor

            kronos_device = "cuda:0" if torch.cuda.is_available() else "cpu"
            tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
            kronos_model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
            kronos_predictor = KronosPredictor(kronos_model, tokenizer, device=kronos_device, max_context=512)
            print("Kronos loaded on", kronos_device)
        except Exception as exc:  # ImportError만으로는 부족(HF 레이트리밋/레포 변경 등)
            print("⚠️ Kronos 로드 실패, 건너뜁니다:", exc)
            RUN_KRONOS = False
else:
    print("RUN_KRONOS=False: Kronos 섹션을 건너뜁니다. "
          "(zero-shot 성능이 '항상 보합' 기준선보다 나빴고, 무거운 의존성의 유일한 원인입니다.)")

In [ ]:
def kronos_probability_for_date(target_date, mc_samples=KRONOS_MC_SAMPLES):
    target_date = pd.Timestamp(target_date).normalize()
    hist = sam.loc[sam.index < target_date].tail(KRONOS_LOOKBACK).copy()
    if len(hist) < 100:
        raise ValueError("Kronos context is too short")

    band_value = float(model_df.loc[target_date, "band"]) if target_date in model_df.index else live_band

    x_df = hist[["open", "high", "low", "close", "volume"]].copy()
    x_df["volume"] = x_df["volume"].fillna(0).clip(lower=0)
    x_timestamp = pd.Series(hist.index)
    y_timestamp = pd.Series([target_date])
    previous_close = float(hist["close"].iloc[-1])
    if TARGET_MODE == "open_to_close":
        if target_date in sam.index:
            reference_price = float(sam.loc[target_date, "open"])
        else:
            reference_price = float(LIVE_OPEN_PRICE)
    else:
        reference_price = previous_close

    # 전역 RNG를 오염시키지 않도록 상태를 저장했다가 복원한다.
    py_state, np_state = random.getstate(), np.random.get_state()
    torch_state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        sampled_returns = []
        for sample_no in range(mc_samples):
            sample_seed = SEED + sample_no
            random.seed(sample_seed)
            np.random.seed(sample_seed)
            torch.manual_seed(sample_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(sample_seed)
            pred_df = kronos_predictor.predict(
                df=x_df, x_timestamp=x_timestamp, y_timestamp=y_timestamp,
                pred_len=1, T=0.8, top_p=0.9, sample_count=1,
            )
            sampled_returns.append(float(pred_df["close"].iloc[0]) / reference_price - 1)
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.set_rng_state(torch_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)

    sampled_returns = np.asarray(sampled_returns)
    counts = np.array([
        np.sum(sampled_returns < -band_value),
        np.sum((sampled_returns >= -band_value) & (sampled_returns <= band_value)),
        np.sum(sampled_returns > band_value),
    ], dtype=float)
    probs = (counts + 0.5) / (counts.sum() + 1.5)   # Jeffreys smoothing
    return probs, sampled_returns


if RUN_KRONOS and kronos_predictor is not None:
    if KRONOS_EVAL_DAYS < 250:
        print(f"⚠️ KRONOS_EVAL_DAYS={KRONOS_EVAL_DAYS}일로는 예측력의 유무를 판정할 수 없습니다"
              f" (무작위 예측의 60일 balanced accuracy 표준편차가 약 0.065). 참고용으로만 보세요.")
    all_test_dates = pd.DatetimeIndex(sorted(predictions["date"].unique()))
    kronos_dates = all_test_dates.intersection(sam.index)[-KRONOS_EVAL_DAYS:]
    rows = []
    for target_date in tqdm(kronos_dates, desc="Kronos zero-shot backtest"):
        try:
            probs_k, sampled_ret = kronos_probability_for_date(target_date)
            rows.append(prediction_frame(
                "Kronos-small", [target_date], [int(model_df.loc[target_date, "target"])],
                probs_k.reshape(1, -1), fold_id="zero-shot",
                extra={"pred_return_mean": [float(sampled_ret.mean())],
                       "pred_return_std": [float(sampled_ret.std(ddof=0))]},
            ))
        except Exception as exc:
            print(f"⚠️ {target_date.date()} Kronos 실패: {exc}")
    PREDICTION_PARTS["kronos"] = pd.concat(rows, ignore_index=True) if rows else None
else:
    PREDICTION_PARTS["kronos"] = None

predictions = assemble_predictions()
if PREDICTION_PARTS.get("kronos") is not None:
    display(summarize_predictions(PREDICTION_PARTS["kronos"], with_ci=False).style.format("{:.4f}", na_rep="—"))

## 8. 성능 표와 갭/세션 분해

세 종류의 표를 봅니다.

1. **전체 평가 구간 성능** — 모든 지표에 월 블록 부트스트랩 95% 신뢰구간을 붙였습니다. 일별 예측은 서로 독립이 아니므로 달력 월을 블록으로 복원추출합니다.
2. **갭 / 세션 분해** — 이 노트북에서 가장 중요한 표입니다. 모델의 방향 점수가 *전일 종가→시가 갭*의 부호를 맞히는지, *시가→종가 세션*의 부호를 맞히는지 따로 측정하고, 각 구간에서 모델 방향대로 포지션을 잡았을 때의 일평균 손익(bp)을 왕복 비용 차감 전후로 보여줍니다. **07:00 예측으로 실제 취할 수 있는 것은 세션 구간뿐입니다.**
3. **'항상 보합' 대비 쌍체 차이** — 같은 날짜에서 계산한 차이와 신뢰구간입니다. CI가 0을 포함하면 "동률"로 읽어야 합니다.

Kronos를 켰을 때만 공통 날짜 참고표가 추가됩니다. 그 표본(기본 60일)에서는 무작위 예측도 약 49%의 확률로 1/3 기준선을 넘으므로 순위를 읽으면 안 됩니다.

In [ ]:
native_metrics = summarize_predictions(predictions, with_decomposition=True)

print("■ 전체 평가 구간 성능 (95% 신뢰구간은 달력 월 블록 부트스트랩)")
display(
    native_metrics[["n", "accuracy", "balanced_accuracy", "bal_acc_lo", "bal_acc_hi",
                    "macro_f1", "log_loss", "log_loss_lo", "log_loss_hi", "brier"]]
    .style.format("{:.4f}", na_rep="—")
)

# ---- 이 모델이 실제로 무엇을 예측하는가 --------------------------------------
print("\n■ 갭 / 세션 분해 — 07:00 예측으로 실제 취할 수 있는 것은 '세션'뿐이다")
print("   갭     = 전일 종가 → 당일 시가 (09:00에 이미 가격에 반영됨)")
print("   세션   = 당일 시가 → 당일 종가 (시가에 진입하면 취할 수 있는 유일한 구간)")
decomp_cols = ["auc_gap", "auc_session", "gap_bp", "session_bp", "session_bp_lo",
               "session_bp_hi", "session_bp_net"]
display(
    native_metrics[decomp_cols].style.format(
        {"auc_gap": "{:.3f}", "auc_session": "{:.3f}", "gap_bp": "{:+.1f}",
         "session_bp": "{:+.1f}", "session_bp_lo": "{:+.1f}", "session_bp_hi": "{:+.1f}",
         "session_bp_net": "{:+.1f}"}, na_rep="—")
    .set_caption(f"bp = 일평균 수익(베이시스포인트). net은 왕복 비용 {COST_BP:.0f}bp 차감 후.")
)

# ---- 차이가 실재하는가: 쌍체 비교 --------------------------------------------
reference = "Always flat"
compare = [m for m in MODEL_ORDER if m in set(predictions["model"]) and m != reference]
rows = []
for name in compare:
    for metric in ("balanced_accuracy", "log_loss"):
        d = paired_delta_ci(predictions, name, reference, metric=metric)
        rows.append({
            "model": name, "metric": metric, "n": d["n"],
            "delta_vs_flat": d["delta"], "lo": d["lo"], "hi": d["hi"],
            "판정": "유의" if (pd.notna(d["lo"]) and (d["lo"] > 0) == (d["hi"] > 0)) else "동률(CI가 0 포함)",
        })
    rows.append({"model": name, "metric": "folds_worse_than_prior",
                 "n": len(folds), "delta_vs_flat": folds_worse_than_prior(predictions, name),
                 "lo": np.nan, "hi": np.nan, "판정": ""})
print("\n■ '항상 보합' 대비 쌍체 차이 (CI가 0을 포함하면 동률로 읽어야 한다)")
display(pd.DataFrame(rows).set_index(["model", "metric"])
        .style.format({"delta_vs_flat": "{:+.4f}", "lo": "{:+.4f}", "hi": "{:+.4f}"}, na_rep="—"))

# ---- Kronos가 있을 때만: 공통 날짜 참고표 -----------------------------------
common_metrics = None
if PREDICTION_PARTS.get("kronos") is not None:
    model_date_sets = {n: set(pd.to_datetime(g["date"])) for n, g in predictions.groupby("model")}
    common_dates = set.intersection(*model_date_sets.values())
    if len(common_dates) >= 10:
        common_metrics = summarize_predictions(
            predictions[predictions["date"].isin(common_dates)], with_ci=False)
        print(f"\n■ [참고] 모든 모델이 예측한 공통 날짜 {len(common_dates)}일")
        print("   ⚠️ 이 표본에서는 무작위 예측도 약 49%의 확률로 1/3 기준선을 넘습니다. 순위를 읽지 마세요.")
        display(common_metrics.style.format("{:.4f}", na_rep="—"))
# 외부 평가 구간의 기존 설정 대비 개선 폭. 내부 선택 점수와 구분한다.
improvement_rows = []
for metric in ["accuracy", "balanced_accuracy", "log_loss"]:
    delta = paired_delta_ci(predictions, "Mean ensemble", "Previous ensemble", metric)
    improvement_rows.append({"metric": metric, **delta})
improvement_table = pd.DataFrame(improvement_rows)
print("■ 개선 모델 - 기존 고정설정 (외부 평가 구간, 95% CI)")
display(improvement_table)
macro_comparison = pd.DataFrame()
if MACRO_ACTIVE:
    macro_comparison = pd.DataFrame([
        {"metric": metric, **paired_delta_ci(predictions, "Mean ensemble", "No macro ensemble", metric)}
        for metric in ["accuracy", "balanced_accuracy", "log_loss"]])
    print("■ 월별 지표 추가 효과: 같은 학습/평가 날짜, 같은 내부 선택 방법")
    print("⚠️ 최신 수정치로 계산한 탐색 결과입니다. 실제 누적 예측으로 확인하세요.")
    display(macro_comparison)

In [ ]:
# 그림은 항상 전체 평가 구간에서 그린다(60일 창은 검정력이 없다).
plot_metrics = native_metrics.reset_index()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

axes[0].barh(plot_metrics["model"], plot_metrics["balanced_accuracy"], color="#4c78a8")
axes[0].errorbar(
    plot_metrics["balanced_accuracy"], plot_metrics["model"],
    xerr=[plot_metrics["balanced_accuracy"] - plot_metrics["bal_acc_lo"],
          plot_metrics["bal_acc_hi"] - plot_metrics["balanced_accuracy"]],
    fmt="none", ecolor="black", capsize=3, lw=1,
)
axes[0].axvline(1 / 3, color="black", ls="--", lw=1, label="random 3-class")
axes[0].set_title("Balanced accuracy (95% CI) ↑")
axes[0].legend()

axes[1].barh(plot_metrics["model"], plot_metrics["log_loss"], color="#e15759")
axes[1].errorbar(
    plot_metrics["log_loss"], plot_metrics["model"],
    xerr=[plot_metrics["log_loss"] - plot_metrics["log_loss_lo"],
          plot_metrics["log_loss_hi"] - plot_metrics["log_loss"]],
    fmt="none", ecolor="black", capsize=3, lw=1,
)
axes[1].set_title("Log loss (95% CI) ↓")
axes[1].set_xlim(left=min(0.95, plot_metrics["log_loss_lo"].min() - 0.02))

width = 0.38
positions = np.arange(len(plot_metrics))
axes[2].barh(positions + width / 2, plot_metrics["auc_gap"], height=width,
             color="#4c78a8", label="갭 (거래 불가)")
axes[2].barh(positions - width / 2, plot_metrics["auc_session"], height=width,
             color="#f28e2b", label="세션 (거래 가능)")
axes[2].set_yticks(positions)
axes[2].set_yticklabels(plot_metrics["model"])
axes[2].axvline(0.5, color="black", ls="--", lw=1)
axes[2].set_xlim(0.35, 0.9)
axes[2].set_title("방향 판별 AUC: 갭 vs 세션")
axes[2].legend()

for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
models_to_plot = [m for m in MODEL_ORDER if m in set(predictions["model"])]
ncols = 3
nrows = math.ceil(len(models_to_plot) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, name in zip(axes, models_to_plot):
    group = predictions[predictions["model"] == name]
    cm = confusion_matrix(group["y_true"], group["y_pred"], labels=[0, 1, 2], normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=["down", "flat", "up"], yticklabels=["down", "flat", "up"], ax=ax)
    ax.set_title(f"{name} (n={len(group)})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
for ax in axes[len(models_to_plot):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def reliability_by_class(frame, class_index, bins=10):
    """클래스별 예측확률을 분위로 나눠 실제 발생률과 비교한다.
    (최대확률 기준 구간화는 3-class에서 (0,1/3) 구간이 구조적으로 비어 버린다.)"""
    work = frame.copy()
    p = work[PROB_COLS[class_index]].to_numpy()
    hit = (work["y_true"].to_numpy() == class_index).astype(int)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return pd.DataFrame()
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, len(edges) - 2)
    out = pd.DataFrame({"bin": idx, "p": p, "hit": hit}).groupby("bin").agg(
        n=("hit", "size"), mean_p=("p", "mean"), observed=("hit", "mean"))
    return out[out["n"] >= 20]


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for class_index, ax in enumerate(axes):
    for name in models_to_plot:
        group = predictions[predictions["model"] == name]
        if len(group) < 200:
            continue
        rel = reliability_by_class(group, class_index)
        if len(rel):
            ax.plot(rel["mean_p"], rel["observed"], marker="o", ms=4, label=name)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim(0, 0.8)
    ax.set_ylim(0, 0.8)
    ax.set_title(f"P({LABEL_NAMES[class_index]}) 신뢰도")
    ax.set_xlabel("예측 확률")
    ax.set_ylabel("실제 발생률")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 9. LightGBM 변수 중요도

중요도는 인과관계가 아니라 분할에 자주 사용된 정도입니다. 여러 폴드의 중요도를 평균합니다.


In [ ]:
if lgb_importances:
    importance = pd.Series(np.mean(lgb_importances, axis=0), index=feature_cols).sort_values(ascending=False)
    display(importance.head(20).to_frame("mean_split_importance"))
    importance.head(20).sort_values().plot.barh(figsize=(8, 7), color="#4c78a8")
    plt.title("LightGBM mean feature importance")
    plt.show()


## 10. 최근 5년으로 재학습하고 다음 거래일 방향·중기 가격 예측

아래 결과는 노트북 실행 시점의 최신 공개 데이터로 계산됩니다. 기존 다음 거래일 방향과 함께 5거래일(1주일), 20거래일(1개월) 뒤 종가를 직접 회귀 방식으로 예측합니다. 한국 휴일이 끼어 예측 날짜가 틀리면 설정 셀의 `PREDICTION_DATE_OVERRIDE`를 수정하고 특징 생성 이후 셀을 다시 실행하세요.


In [ ]:
RUN_ID = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:12]
OUTPUT_DIR = STORAGE_ROOT / "runs" / RUN_ID
live_X = live_row[feature_cols].to_numpy(dtype=np.float32)
live_probs = {}

live_train_idx = rolling_train_indices(dates, prediction_date, ROLLING_TRAIN_YEARS)
logistic_full = fit_direction_model(X, y, live_train_idx, "Logistic", seed=SEED)
lgbm_full = fit_direction_model(X, y, live_train_idx, "LightGBM", seed=SEED)
live_probs["Logistic"] = predict_direction_model(logistic_full, live_X)[0]
live_probs["LightGBM"] = predict_direction_model(lgbm_full, live_X)[0]
for fitted in [logistic_full, lgbm_full]:
    model_selection_rows.append({"fold": "live", **fitted["selection"]})

transformer_full = transformer_scaler = None
if RUN_TRANSFORMER and TORCH_AVAILABLE:
    all_train_idx = live_train_idx
    transformer_full, transformer_scaler, _, transformer_full_history, _ = train_transformer(
        X, y, all_train_idx, seed=SEED + 1000)
    X_plus_live_scaled = transformer_scaler.transform(np.vstack([X, live_X])).astype(np.float32)
    live_seq = torch.from_numpy(X_plus_live_scaled[-SEQ_LEN:][None, ...]).to(DEVICE)
    transformer_full.eval()
    with torch.no_grad():
        live_probs["Transformer"] = torch.softmax(transformer_full(live_seq), dim=1).cpu().numpy()[0]

if RUN_KRONOS and kronos_predictor is not None:
    try:
        p_kronos_live, kronos_live_returns = kronos_probability_for_date(prediction_date)
        live_probs["Kronos-small"] = p_kronos_live
    except Exception as exc:
        print("⚠️ Latest Kronos forecast failed:", exc)

# 앙상블은 ENSEMBLE_MODELS의 단순 평균 하나만 쓴다.
# (성과가중 exp(-2·Δlog loss)는 0.03 nats 차이 위에서 계산되어 사실상 균등 평균이었고,
#  판별력을 잃고 사전확률에 가까워진 모델에 가장 큰 가중을 주는 부작용이 있었다.)
ensemble_members = [m for m in ENSEMBLE_MODELS if m in live_probs]
if len(ensemble_members) >= 2:
    live_probs["Mean ensemble"] = np.mean([live_probs[m] for m in ensemble_members], axis=0)
ensemble_weights = {m: 1.0 / len(ensemble_members) for m in ensemble_members}

market_live_models = {}
if MACRO_ACTIVE:
    for family in ["Logistic", "LightGBM"]:
        market_live_models[family] = fit_direction_model(market_X, y, live_train_idx, family, seed=SEED)
        model_selection_rows.append({"fold": "live", "features": "market_only", **market_live_models[family]["selection"]})
    live_probs["No macro ensemble"] = np.mean([
        predict_direction_model(fitted, live_X[:, market_feature_idx])[0]
        for fitted in market_live_models.values()], axis=0)

live_table = []
for name, probs in live_probs.items():
    probs = np.asarray(probs, dtype=float)
    probs = probs / probs.sum()
    live_table.append({
        "model": name,
        "prediction_date": prediction_date.date().isoformat(),
        "prediction": LABEL_NAMES[int(np.argmax(probs))],
        "p_down": probs[0], "p_flat": probs[1], "p_up": probs[2],
    })
live_table = pd.DataFrame(live_table).set_index("model")

print("앙상블 구성:", ensemble_members, "(단순 평균)")

# 이전 버전과 모델 수가 다른 이유를 매번 명시한다(빠진 것이 아니라 뺀 것이다).
disabled = []
if not RUN_TRANSFORMER or not TORCH_AVAILABLE:
    disabled.append("Transformer — RUN_TRANSFORMER=True 로 켜세요"
                    + ("" if TORCH_AVAILABLE else " (torch 필요)"))
if not RUN_KRONOS:
    disabled.append("Kronos-small — RUN_KRONOS=True 로 켜세요. "
                    "zero-shot 성능이 '항상 보합'보다 나빠(log loss 1.87 vs 1.10) 기본 꺼짐")
disabled.append("Stacked / Performance-weighted ensemble — 제거됨. "
                "검증에서 단순 평균보다 나빴습니다(1.0507 vs 1.0379)")
print("\n[모델 구성 안내] 표에 없는 모델:")
for item in disabled:
    print("  ·", item)

if imputed_live_features:
    print("⚠️ 과거값으로 대체된 라이브 특징:", imputed_live_features)
print("⚠️ 이 예측의 타깃은", TARGET_MODE, "입니다.",
      "close_to_close라면 예측력의 대부분은 09:00 시가에 이미 반영되는 '갭'이며,"
      " 시가 진입 후 종가 청산으로 얻을 수 있는 몫은 8절의 session_bp 열을 보세요."
      if TARGET_MODE == "close_to_close" else "")
display(live_table.style.format({"p_down": "{:.1%}", "p_flat": "{:.1%}", "p_up": "{:.1%}"}))

### 10.1 1거래일·1주일·1개월 뒤 예상 종가

각 시점의 미래 수익률을 **변동성 스케일 타깃 + 강한 축소(Ridge)** 로 추정합니다. 이전 버전에서 바뀐 점은 다음과 같습니다.

- 검증에서 두 회귀 모델(Ridge, LightGBM) 모두 **"수익률 0% 예측" 기준선보다 나빴는데도** 점 예측이 그대로 보고되었습니다. 이제는 손실 차이의 신뢰구간이 0을 포함하면 `signal`을 "없음"으로 두고 `predicted_return`·`predicted_close`를 **결측(`—`)으로 비웁니다.** 0.00%로 찍으면 "0%라고 예측했다"로 읽히지만, 실제로는 **예측을 하지 않은 것**이기 때문입니다. 이때 쓸 수 있는 값은 `center_close`(현재가)와 구간뿐입니다.
- 예측값은 OOF에서 추정한 **축소계수**(실제 수익률을 예측값에 회귀시킨 기울기, 0~1)를 곱한 값입니다. 축소가 없으면 20일 지평에서 예측 분산이 근거보다 10배 이상 컸습니다.
- 단일 예상 종가 대신 **변동성 스케일 구간**(`low_close` ~ `high_close`)을 함께 제공하고, 별도 최종 평가 구간의 커버리지를 `band_coverage`에 적습니다.
- LightGBM 회귀는 OOF 우위가 없고 시드에 따라 라이브 값이 몇 %p씩 흔들려 제거했습니다.
- `target_date`는 KRX 거래일 달력 기준입니다(`exchange_calendars`가 없으면 주말만 제외하고 경고합니다).
- **예상 시초가**를 별도로 예측합니다. 1거래일 예상 종가가 검증을 통과하는 이유는 거의 전부 전일 종가→시가 갭이므로(8절), 갭 자체를 같은 방법으로 예측해 `kind="open"`으로 원장에 남기고 **시가로 채점**합니다. 이 값은 09:00 이전에만 의미가 있고, 시가에 매수해서 얻을 수 있는 수익이 아닙니다.


In [ ]:
FORECAST_HORIZONS = {"1거래일": 1, "1주일": 5, "1개월": 20}
BAND_COVERAGE = 0.80        # 예측 구간의 목표 커버리지
# 구간 폭은 q x sigma로 정해진다. sigma를 무엇으로 예측할지 고른다.
#   "simple": 최근 20일 표준편차 x sqrt(h)   ← 기본값
#   "har"   : 일/주/월 실현변동성 HAR 회귀
#
# HAR을 붙여 측정한 결과 채택하지 않았다. 적중률을 80%로 맞춰 놓고 폭을 비교하면
# 1거래일 -5.6%, 1주일 -10.0%, 1개월 +3.9%로 이득이 작거나 없고, 실제 운용 방식
# (조용한 구간에서 q를 정해 격변 구간에 적용)에서는 HAR이 평균회귀로 변동성을
# 과소예측해 적중률이 80% -> 58~67%로 무너졌다.
# simple은 실제/sigma 비율이 구간을 옮겨도 0.78~0.91로 안정적이라 적중률이 유지된다.
# 두 방식을 매 실행 같은 행에서 함께 평가해 아래 비교표로 남긴다.
VOL_MODEL = "simple"


def make_price_model():
    # 강한 축소(Ridge alpha 큼) + 변동성 스케일 타깃. LightGBM 회귀는 OOF에서 우위가 없고
    # 시드에 따라 라이브 값이 몇 %p씩 움직여 제거했다.
    return Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1e4))])


def price_forecast_variants(horizon, live_features):
    """변동성 모형별로 점 예측과 구간을 만든다. 두 방식을 같은 행에서 평가한다.

    행 d의 특징은 d-1 종가까지만 포함한다. 목표값은 d-1 종가 대비
    horizon번째 거래일(d+horizon-1)의 원본 종가 수익률이다.
    """
    future_return = sam_raw_close.shift(-(horizon - 1)) / sam_raw_close.shift(1) - 1
    har_series, har_live = har_sigma_forecast(sam_raw_close.pct_change(), horizon)
    sigma_live = {
        "simple": float(live_row["sam_vol_20"].iloc[0]) * np.sqrt(horizon),
        "har": har_live,
    }

    reg = feat.loc[sam.index, feature_cols].copy()
    reg["future_return"] = future_return.reindex(reg.index)
    reg["sigma_simple"] = feat.loc[sam.index, "sam_vol_20"] * np.sqrt(horizon)
    reg["sigma_har"] = har_series.reindex(reg.index)
    # 두 방식을 같은 행에서 비교해야 공정하다(HAR은 앞부분 학습 구간이 비어 있다).
    reg = reg.replace([np.inf, -np.inf], np.nan).dropna()

    X_reg = reg[feature_cols].to_numpy(dtype=np.float32)
    y_reg = reg["future_return"].to_numpy(dtype=np.float64)
    splitter = TimeSeriesSplit(n_splits=3 if QUICK_MODE else 5, gap=horizon - 1)
    template = make_price_model()

    # 동일한 날짜·변동성·분할로 20거래일 거시지표의 추가 효과를 측정한다.
    # 최종 평가 결과로 발행 모델을 자동 변경하지 않는다.
    if horizon == 20 and any(c.startswith("macro_") for c in feature_cols):
        comparison, comparison_oof = price_macro_ablation(
            X_reg, y_reg, reg[f"sigma_{VOL_MODEL}"].to_numpy(dtype=float), reg.index,
            feature_cols, horizon, template, block_bootstrap_ci,
            n_splits=3 if QUICK_MODE else 5, coverage=BAND_COVERAGE)
        comparison["vol_model"] = VOL_MODEL
        price_macro_comparison_rows.append(comparison)
        price_macro_oof[horizon] = comparison_oof

    results = {}
    for name in ("simple", "har"):
        sigma_h = reg[f"sigma_{name}"].to_numpy(dtype=np.float64)
        z = y_reg / np.maximum(sigma_h, 1e-6)
        oof = np.full(len(z), np.nan)
        for train_idx, valid_idx in splitter.split(X_reg):
            fold_model = clone(template).fit(X_reg[train_idx], z[train_idx])
            oof[valid_idx] = fold_model.predict(X_reg[valid_idx]) * sigma_h[valid_idx]
        mask = ~np.isnan(oof)
        stats = calibrate_price_forecast(
            y_reg[mask], oof[mask], sigma_h[mask], reg.index[mask], horizon,
            block_bootstrap_ci, coverage=BAND_COVERAGE)
        fitted = clone(template).fit(X_reg, z)
        live_sigma = sigma_live[name]
        stats["live_sigma_h"] = live_sigma
        stats["vol_model"] = name
        raw_point = float(fitted.predict(live_features)[0] * live_sigma)   # 축소 전 원시 예측
        stats["raw_point"] = raw_point
        point = float(stats["oof_slope"] * raw_point)
        results[name] = (point, live_sigma * stats["band_q"], stats, fitted)
    return results


current_close = float(sam_raw_close.iloc[-1])
price_forecast_rows = []
price_forecast_stats = {}
price_forecast_models = {}
vol_comparison_rows = []
price_macro_comparison_rows = []
price_macro_oof = {}

for horizon_label, horizon in FORECAST_HORIZONS.items():
    variants = price_forecast_variants(horizon, live_X)
    for name, (_, _, s, _) in variants.items():
        vol_comparison_rows.append({
            "horizon": horizon_label, "vol_model": name,
            "구간폭_평균": s["band_halfwidth_mean"],
            "적중률": s["band_coverage_realized"],
            "라이브_sigma": s["live_sigma_h"],
        })
    point, half_width, stats, fitted = variants[VOL_MODEL]
    price_forecast_stats[horizon_label] = stats
    price_forecast_models[horizon_label] = fitted
    target_date = krx_sessions_ahead(prediction_date, horizon)
    if not stats["beats_baseline"]:
        # 판정은 '신호 선택' 구간에서 내린다. 최종 평가 구간의 MAE를 함께 적으면
        # 두 구간의 숫자가 한 문장에 섞여 "모델이 더 좋은데 왜 안 쓰나"로 읽힌다.
        # (mae_diff_vs_zero는 기울기를 0으로 만든 뒤 값이라 정의상 항상 0이다.)
        print(f"ℹ️ horizon={horizon}: 신호 선택 구간에서 '가격 유지' 대비 MAE 차이 "
              f"[{stats['selection_mae_diff_lo']:+.4f}, "
              f"{stats['selection_mae_diff_hi']:+.4f}] — 상한이 0 아래로 내려가지 "
              f"않아 점 예측을 내지 않습니다(기울기를 0으로 두어 현재가를 씁니다).")
    center = current_close * (1 + point)
    price_forecast_rows.append({
        "horizon": horizon_label,
        "trading_days": horizon,
        "as_of_date": last_samsung_date.date().isoformat(),
        "target_date": target_date.date().isoformat(),
        "current_close": current_close,
        "signal": "있음" if stats["beats_baseline"] else "없음",
        "predicted_return": point if stats["beats_baseline"] else np.nan,
        # 축소 전 값. 발행 예측은 아니고, 원장에서 실현 기울기(실제/원시)를 추적해 축소계수가
        # 맞는지 확인하는 데 쓴다(review_ledger).
        "raw_predicted_return": stats["raw_point"],
        "predicted_close": center if stats["beats_baseline"] else np.nan,
        "center_close": center,
        "low_close": current_close * (1 + point - half_width),
        "high_close": current_close * (1 + point + half_width),
        "band_coverage": stats["band_coverage_realized"],
        "vol_model": VOL_MODEL,
        "model_mae": stats["raw_model_mae"],
        "zero_baseline_mae": stats["zero_baseline_mae"],
        "mae_diff_lo": stats["mae_diff_lo"],
        "mae_diff_hi": stats["mae_diff_hi"],
        "oof_slope": stats["oof_slope"],
    })

price_forecast_table = pd.DataFrame(price_forecast_rows).set_index("horizon")
vol_comparison = pd.DataFrame(vol_comparison_rows)

print("\n■ 변동성 모형 비교 — 같은 행·같은 평가 구간에서, 적중률은 비슷하고 구간은 좁을수록 좋다")
_pivot = vol_comparison.pivot(index="horizon", columns="vol_model",
                              values=["구간폭_평균", "적중률"])
_pivot[("구간폭_평균", "개선")] = (
    1 - _pivot[("구간폭_평균", "har")] / _pivot[("구간폭_평균", "simple")])
display(_pivot.style.format({
    ("구간폭_평균", "simple"): "{:.2%}", ("구간폭_평균", "har"): "{:.2%}",
    ("구간폭_평균", "개선"): "{:+.1%}",
    ("적중률", "simple"): "{:.1%}", ("적중률", "har"): "{:.1%}"}))
print(f"※ 발행에 사용한 변동성 모형: VOL_MODEL = \"{VOL_MODEL}\"")
print(f"※ 목표 적중률 {BAND_COVERAGE:.0%}. 적중률이 크게 낮아지면서 구간만 좁아진 것은 개선이 아닙니다.")

print("\n■ 중기 가격 예측 요약")
for row in price_forecast_rows:
    if row["signal"] == "있음":
        print(f"  · {row['horizon']} 뒤({row['target_date']}): "
              f"{row['predicted_close']:,.0f}원 예상 ({row['predicted_return']:+.2%}), "
              f"명목 {BAND_COVERAGE:.0%} 구간 {row['low_close']:,.0f}~{row['high_close']:,.0f}원")
    else:
        print(f"  · {row['horizon']} 뒤({row['target_date']}): "
              f"{row['center_close']:,.0f}원 (변화 없음) — 검증 미통과로 현재가를 그대로 "
              f"씁니다, 명목 {BAND_COVERAGE:.0%} 구간 "
              f"{row['low_close']:,.0f}~{row['high_close']:,.0f}원")
print("※ 표의 predicted_return/predicted_close가 '—'인 것은 출력 오류가 아니라,")
print("   신호 선택 구간에서 우위를 확인하지 못해 '예측하지 않음'으로 처리했다는 뜻입니다.")
print("   그 경우 center_close(= 현재가)를 '변화 없음'으로 보고서에 표시하며,")
print("   원장에는 예측을 내지 않았다는 사실이 남도록 계속 빈 값으로 기록합니다.")
print("※ 구간은 변동성 밴드이며 band_coverage는 독립 평가 구간의 적중률입니다. "
      "target_date는 KRX 거래일 기준입니다.")
print("※ 1거래일 뒤 예상 종가가 게이트를 통과하는 것은 거의 전부 '전일 종가→시가 갭' 덕분입니다"
      "(8절 갭 AUC ≈ 0.8, 세션 AUC ≈ 0.5). 갭은 아래 '예상 시초가'가 따로 예측합니다.")

display_cols = ["trading_days", "as_of_date", "target_date", "signal", "vol_model",
                "current_close", "predicted_return", "predicted_close",
                "center_close", "low_close", "high_close", "band_coverage",
                "model_mae", "zero_baseline_mae", "mae_diff_lo", "mae_diff_hi", "oof_slope"]
display(price_forecast_table[display_cols].style.format({
    "current_close": "{:,.0f}원", "predicted_return": "{:+.2%}",
    "predicted_close": "{:,.0f}원", "center_close": "{:,.0f}원",
    "low_close": "{:,.0f}원", "high_close": "{:,.0f}원",
    "band_coverage": "{:.1%}", "model_mae": "{:.2%}", "zero_baseline_mae": "{:.2%}",
    "mae_diff_lo": "{:+.4f}", "mae_diff_hi": "{:+.4f}", "oof_slope": "{:.3f}",
}, na_rep="—"))

price_macro_comparison = pd.DataFrame(price_macro_comparison_rows)
if len(price_macro_comparison):
    print("\n■ 20거래일 거시지표 추가 효과 — 같은 날짜의 수익률 MAE 비교")
    print("차이 = 거시지표 포함 − 시세만 사용. 음수가 유리하며 신뢰구간이 0을 포함하면 개선 미확인.")
    print("raw는 축소 전, center는 독립 보정·신호 선택 후 실제 발행 중심값의 오차입니다.")
    display(price_macro_comparison)
else:
    print("ℹ️ 거시지표를 사용하지 않아 20거래일 추가 효과 비교를 생략했습니다.")


# ---- 예상 시초가(갭) 예측 ----------------------------------------------------
# 1거래일 종가 예측의 신호는 거의 전부 갭이다. 그래서 갭 자체를 별도 타깃으로 두고
# "예상 시초가"로 보고한다. 종가 예측과 같은 방법(변동성 스케일 타깃 + 강한 축소 Ridge,
# OOF 보정/신호 선택/최종 평가 분리)을 쓰고, 원장에는 kind="open"으로 남겨 시가로 채점한다.
# 이 값은 09:00 이전에만 의미가 있다. 시가에 매수해서 얻을 수 있는 수익이 아니다.
def open_gap_forecast(live_features):
    gap_return = sam_gap.reindex(sam.index)                             # open_d / close_{d-1} - 1
    gap_sigma = sam_gap.rolling(20).std().shift(1).reindex(sam.index)    # d-1까지의 갭 변동성
    reg = feat.loc[sam.index, feature_cols].copy()
    reg["gap"], reg["sigma"] = gap_return, gap_sigma
    reg = reg.replace([np.inf, -np.inf], np.nan).dropna()

    X_reg = reg[feature_cols].to_numpy(dtype=np.float32)
    y_reg = reg["gap"].to_numpy(dtype=np.float64)
    sigma_h = reg["sigma"].to_numpy(dtype=np.float64)
    z = y_reg / np.maximum(sigma_h, 1e-6)
    template = make_price_model()
    oof = np.full(len(z), np.nan)
    for train_idx, valid_idx in TimeSeriesSplit(n_splits=3 if QUICK_MODE else 5).split(X_reg):
        fold_model = clone(template).fit(X_reg[train_idx], z[train_idx])
        oof[valid_idx] = fold_model.predict(X_reg[valid_idx]) * sigma_h[valid_idx]
    mask = ~np.isnan(oof)
    stats = calibrate_price_forecast(y_reg[mask], oof[mask], sigma_h[mask], reg.index[mask], 1,
                                     block_bootstrap_ci, coverage=BAND_COVERAGE)
    # 갭 방향 판별력(OOF 예측 부호 vs 실제 갭 부호). 종가 방향 모델의 갭 AUC와 같은 잣대.
    ok = mask & (y_reg != 0)
    stats["gap_sign_auc"] = float(roc_auc_score((y_reg[ok] > 0).astype(int), oof[ok])) if ok.sum() >= 30 else np.nan
    stats["gap_corr"] = float(np.corrcoef(y_reg[mask], oof[mask])[0, 1])
    fitted = clone(template).fit(X_reg, z)
    live_sigma = float(sam_gap.dropna().iloc[-20:].std())
    stats["live_sigma_h"] = live_sigma
    stats["raw_point"] = float(fitted.predict(live_features)[0] * live_sigma)
    point = float(stats["oof_slope"] * stats["raw_point"])
    return point, live_sigma * stats["band_q"], stats, fitted


open_point, open_half_width, open_forecast_stats, open_forecast_model = open_gap_forecast(live_X)
open_center = current_close * (1 + open_point)
open_forecast_row = {
    "horizon": "시초가", "trading_days": 1,
    "as_of_date": last_samsung_date.date().isoformat(),
    "target_date": prediction_date.date().isoformat(),
    "current_close": current_close,
    "signal": "있음" if open_forecast_stats["beats_baseline"] else "없음",
    "predicted_return": open_point if open_forecast_stats["beats_baseline"] else np.nan,
    "raw_predicted_return": open_forecast_stats["raw_point"],
    "predicted_open": open_center if open_forecast_stats["beats_baseline"] else np.nan,
    "center_open": open_center,
    "low_open": current_close * (1 + open_point - open_half_width),
    "high_open": current_close * (1 + open_point + open_half_width),
    "band_coverage": open_forecast_stats["band_coverage_realized"],
    "model_mae": open_forecast_stats["raw_model_mae"],
    "zero_baseline_mae": open_forecast_stats["zero_baseline_mae"],
    "mae_diff_lo": open_forecast_stats["mae_diff_lo"],
    "mae_diff_hi": open_forecast_stats["mae_diff_hi"],
    "oof_slope": open_forecast_stats["oof_slope"],
    "gap_sign_auc": open_forecast_stats["gap_sign_auc"],
    "gap_corr": open_forecast_stats["gap_corr"],
}
open_forecast_table = pd.DataFrame([open_forecast_row]).set_index("horizon")

print("\n■ 예상 시초가 — 전일 종가 대비 당일 시가(갭). 09:00 이전에만 의미가 있고, 시가에 사서 얻는 수익이 아닙니다.")
if open_forecast_row["signal"] == "있음":
    print(f"  · {prediction_date.date()} 시가: {open_center:,.0f}원 예상 ({open_point:+.2%}), "
          f"명목 {BAND_COVERAGE:.0%} 구간 {open_forecast_row['low_open']:,.0f}~{open_forecast_row['high_open']:,.0f}원")
else:
    print(f"  · {prediction_date.date()} 시가: 점 예측 없음(신호 선택 구간에서 '갭 0%' 기준선 대비 우위 미확인), "
          f"명목 {BAND_COVERAGE:.0%} 구간 {open_forecast_row['low_open']:,.0f}~{open_forecast_row['high_open']:,.0f}원")
print(f"  · 갭 방향 AUC {open_forecast_stats['gap_sign_auc']:.3f} · 실제 갭과 상관 {open_forecast_stats['gap_corr']:.3f} · "
      f"모델 MAE {open_forecast_stats['raw_model_mae']:.2%} vs 갭 0% {open_forecast_stats['zero_baseline_mae']:.2%}")
display(open_forecast_table[["trading_days", "target_date", "signal", "current_close", "predicted_return",
                             "predicted_open", "center_open", "low_open", "high_open", "band_coverage",
                             "model_mae", "zero_baseline_mae", "mae_diff_lo", "mae_diff_hi", "oof_slope",
                             "gap_sign_auc", "gap_corr"]].style.format({
    "current_close": "{:,.0f}원", "predicted_return": "{:+.2%}", "predicted_open": "{:,.0f}원",
    "center_open": "{:,.0f}원", "low_open": "{:,.0f}원", "high_open": "{:,.0f}원",
    "band_coverage": "{:.1%}", "model_mae": "{:.2%}", "zero_baseline_mae": "{:.2%}",
    "mae_diff_lo": "{:+.4f}", "mae_diff_hi": "{:+.4f}", "oof_slope": "{:.3f}",
    "gap_sign_auc": "{:.3f}", "gap_corr": "{:.3f}",
}, na_rep="—"))


## 11. 결과 저장

백테스트 예측, 공통기간 성능, 다음 거래일 방향, 1주일·1개월 가격 예측, 학습 모델과 설정을 `/content/samsung_direction_outputs.zip`으로 묶습니다. 마지막 줄의 다운로드 코드는 주석을 해제해 사용하세요.


In [ ]:
import io
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



predictions.sort_values(["date", "model"]).to_csv(OUTPUT_DIR / "backtest_predictions.csv", index=False)
native_metrics.to_csv(OUTPUT_DIR / "metrics_native_window.csv")
if common_metrics is not None:
    common_metrics.to_csv(OUTPUT_DIR / "metrics_common_dates.csv")
live_table.to_csv(OUTPUT_DIR / "latest_forecast.csv")
price_forecast_table.to_csv(OUTPUT_DIR / "multi_horizon_price_forecast.csv")
open_forecast_table.to_csv(OUTPUT_DIR / "open_price_forecast.csv")
pd.Series(feature_cols, name="feature").to_csv(OUTPUT_DIR / "feature_list.csv", index=False)

# ---- 매일 예측 원장: 예측 불변, 실제값/오차만 후속 실행에서 갱신 ----------------
created_at = pd.Timestamp.now(tz="UTC").isoformat()
experiment_config = {
    "schema_version": 3, "model_version": "temporal-selection-macro-v3",
    "macro_enabled": MACRO_ACTIVE, "macro_history_mode": macro_history_mode,
    # 어떤 모델을 실제로 돌렸는지도 설정의 일부다. 이것이 빠져 있으면 구성이 다른
    # 실행이 같은 config_hash로 기록되어, 나중에 "무엇을 돌렸는지" 복원할 수 없다.
    # (의도한 스위치가 아니라 실제 실행 여부를 남긴다: GPU 없음/Kronos 로드 실패 등)
    "run_transformer": bool(RUN_TRANSFORMER and TORCH_AVAILABLE),
    "run_kronos": bool(RUN_KRONOS and kronos_predictor is not None),
    "target_mode": TARGET_MODE, "band_mode": BAND_MODE,
    "vol_band_mult": VOL_BAND_MULT, "neutral_band": NEUTRAL_BAND,
    "training_years": ROLLING_TRAIN_YEARS, "ensemble_models": ENSEMBLE_MODELS,
    "seed": SEED, "feature_cols": feature_cols, "quick_mode": QUICK_MODE,
}
config_hash = hashlib.sha256(json.dumps(experiment_config, sort_keys=True).encode()).hexdigest()[:20]
common_record = {
    # 3: runtime/versions_hash 추가. config_hash(무엇을 선택했는가)와 분리해서
    #    둔다 — 환경이 바뀐 것을 "설정이 바뀌었다"고 기록하면 의미가 흐려진다.
    "schema_version": 3, "run_id": RUN_ID, "created_at_utc": created_at,
    "runtime": RUNTIME, "versions_hash": VERSIONS_HASH,
    "prediction_date": prediction_date.date().isoformat(),
    "as_of_date": last_samsung_date.date().isoformat(),
    "data_snapshot_hash": DATA_SNAPSHOT_HASH, "config_hash": config_hash,
    "target_mode": TARGET_MODE, "band": live_band,
    "imputed_features": json.dumps(imputed_live_features, ensure_ascii=False),
    "macro_snapshot_hash": macro_info.get("snapshot_hash", ""),
    "macro_history_mode": macro_history_mode if MACRO_ACTIVE else "disabled",
}
log_records = []
for model_name, row in live_table.iterrows():
    log_records.append({**common_record, **row.to_dict(), "model": model_name,
                        "record_id": f"{RUN_ID}:direction:{model_name}",
                        "kind": "direction", "horizon_days": 1,
                        "target_date": prediction_date.date().isoformat()})
for row in price_forecast_rows:
    log_records.append({**common_record, **row,
                        "model": "Ridge", "kind": "price", "horizon_days": row["trading_days"],
                        "record_id": f"{RUN_ID}:price:{row['trading_days']}"})
log_records.append({**common_record, **open_forecast_row,
                    "model": "Ridge", "kind": "open", "horizon_days": 1,
                    "record_id": f"{RUN_ID}:open:1"})
forecast_log_path = STORAGE_ROOT / "forecast_log.csv"
all_log = append_forecasts(forecast_log_path, pd.DataFrame(log_records))
evaluated_log = evaluate_forecasts(all_log, sam)
atomic_csv(evaluated_log, forecast_log_path)
daily = daily_comparison(evaluated_log)
atomic_csv(daily, STORAGE_ROOT / "daily_forecast_comparison.csv")
scored = summarize_daily(daily)
atomic_csv(scored, STORAGE_ROOT / "forecast_accuracy_summary.csv")

# ---- 실제 사전 예측만으로 계산한 최근 성능·경고 ---------------------------------
# 백테스트와 별개다. 어제 예측이 오늘 실제와 얼마나 달랐는지, 최근 20·60일 창에서
# 방향 적중률·구간 적중률·수익률 MAE가 기준선을 이기는지, 축소계수가 맞는지를 본다.
# 여기서 나온 경고는 판단 근거이지 자동 조정 트리거가 아니다(하루 결과로 설정을 바꾸면
# 백테스트가 증명한 규율이 무너진다).
_ens_name = "Mean ensemble" if "Mean ensemble" in live_probs else ENSEMBLE_MODELS[0]
ledger_review = review_ledger(daily, sam, ensemble_model=_ens_name, windows=(20, 60))
if ledger_review["n_scored_days"]:
    print(f"■ 채점된 사전 예측: {ledger_review['n_scored_days']}거래일 "
          f"(마지막 {ledger_review['latest_date'].date()})")
    display(ledger_review["rolling"].style.format({
        "hit_rate": "{:.1%}", "flat_share": "{:.1%}", "mean_log_loss": "{:.3f}", "prior_log_loss": "{:.3f}",
        "interval_coverage": "{:.0%}", "mae_return": "{:.2%}", "zero_mae_return": "{:.2%}",
        "realized_slope": "{:.2f}", "oof_slope_mean": "{:.2f}"}, na_rep="—"))
    for _a in ledger_review["alerts"]:
        print("⚠️", _a)
    ledger_review["rolling"].to_csv(OUTPUT_DIR / "ledger_rolling_metrics.csv", index=False)
    ledger_review["latest"].to_csv(OUTPUT_DIR / "ledger_latest_scored.csv", index=False)
else:
    print("■ 아직 채점된 사전 예측이 없습니다. 첫 예측일 다음 실행부터 실제값과 대조됩니다.")
print("■ 일별 예측/실제값 비교 (미래 날짜는 pending, 오차 = 예측 - 실제)")
display(daily.tail(20))
if len(scored):
    print("■ 실제 누적 예측 실적 (날짜별 최초 사전 예측만 집계)")
    display(scored)
print("전체 실행 기록:", forecast_log_path)
print("일별 비교:", STORAGE_ROOT / "daily_forecast_comparison.csv")

config = {
    "run_id": RUN_ID,
    "created_at_utc": created_at,
    "config_hash": config_hash,
    "experiment_config": experiment_config,
    "model_selection": model_selection_rows,
    "price_macro_comparison": price_macro_comparison.to_dict("records"),
    "macro_info": macro_info,
    "data_snapshot_hash": DATA_SNAPSHOT_HASH,
    "versions": VERSIONS,
    "assets": ASSETS,
    "start_date": START_DATE,
    "neutral_band": NEUTRAL_BAND,
    "target_mode": TARGET_MODE,
    "band_mode": BAND_MODE,
    "vol_band_mult": VOL_BAND_MULT,
    "live_band": live_band,
    "ensemble_models": ENSEMBLE_MODELS,
    "ensemble_weights": {k: float(v) for k, v in ensemble_weights.items()},
    "first_test_date": FIRST_TEST_DATE,
    "test_months": TEST_MONTHS,
    "rolling_train_years": ROLLING_TRAIN_YEARS,
    "quick_mode": QUICK_MODE,
    "seq_len": SEQ_LEN,
    "run_transformer": RUN_TRANSFORMER,
    "transformer_epochs": TRANSFORMER_EPOCHS,
    "run_kronos": RUN_KRONOS,
    "kronos_eval_days": KRONOS_EVAL_DAYS,
    "kronos_mc_samples": KRONOS_MC_SAMPLES,
    "cost_bp": COST_BP,
    "bootstrap_b": BOOTSTRAP_B,
    "last_samsung_date": last_samsung_date.date().isoformat(),
    "prediction_date": prediction_date.date().isoformat(),
    "model_rows": int(len(model_df)),
    "backtest_range": [model_df.index.min().date().isoformat(), model_df.index.max().date().isoformat()],
    "n_folds": len(folds),
    "imputed_live_features": imputed_live_features,
    "forecast_horizons": FORECAST_HORIZONS,
    "price_forecast_stats": price_forecast_stats,
    "open_forecast_stats": open_forecast_stats,
    "asset_last_bar": {n: f.index.max().date().isoformat() for n, f in raw.items()},
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

joblib.dump(logistic_full, OUTPUT_DIR / "logistic_model.joblib")
joblib.dump(lgbm_full, OUTPUT_DIR / "lightgbm_model.joblib")
joblib.dump(open_forecast_model, OUTPUT_DIR / "open_gap_ridge.joblib")
for horizon_label, fitted_model in price_forecast_models.items():
    joblib.dump(fitted_model, OUTPUT_DIR / f"price_{FORECAST_HORIZONS[horizon_label]}d_ridge.joblib")
if RUN_TRANSFORMER and TORCH_AVAILABLE and transformer_full is not None:
    joblib.dump(transformer_scaler, OUTPUT_DIR / "transformer_scaler.joblib")
    torch.save({
        "state_dict": {k: v.detach().cpu() for k, v in transformer_full.state_dict().items()},
        "n_features": len(feature_cols), "seq_len": SEQ_LEN,
        "feature_cols": feature_cols, "version": 2,
    }, OUTPUT_DIR / "transformer_model.pt")

import shutil
for name in ["forecast_log.csv", "daily_forecast_comparison.csv", "forecast_accuracy_summary.csv"]:
    shutil.copy2(STORAGE_ROOT / name, OUTPUT_DIR / name)
pd.DataFrame(model_selection_rows).to_csv(OUTPUT_DIR / "model_selection.csv", index=False)
improvement_table.to_csv(OUTPUT_DIR / "improvement_vs_previous.csv", index=False)
if MACRO_ACTIVE:
    macro_comparison.to_csv(OUTPUT_DIR / "macro_comparison.csv", index=False)
    price_macro_comparison.to_csv(OUTPUT_DIR / "price_macro_comparison.csv", index=False)
    for horizon, oof_rows in price_macro_oof.items():
        oof_rows.to_csv(OUTPUT_DIR / f"price_macro_oof_{horizon}d.csv")
    pd.concat([f.assign(series=s) for s, f in macro_data.items()]).to_csv(OUTPUT_DIR / "macro_monthly.csv", index=False)
    feat[macro_feature_cols].to_csv(OUTPUT_DIR / "macro_daily_features.csv")
    joblib.dump({"models": market_live_models, "feature_cols": [feature_cols[i] for i in market_feature_idx]},
                OUTPUT_DIR / "market_only_models.joblib")
zip_path = shutil.make_archive(str(STORAGE_ROOT / "latest_outputs"), "zip", OUTPUT_DIR)
print("Saved:", zip_path, "| run_id:", RUN_ID)

# 필요할 때 아래 두 줄의 주석을 해제하세요.
# from google.colab import files
# files.download(zip_path)


# ---- 예측 원장을 GitHub에 올리기 ----------------------------------------------
# 다른 기기/다른 실행이 먼저 올렸을 수 있으므로, 충돌하면 원격을 다시 읽어
# record_id 기준으로 합친 뒤 재시도한다(예측은 불변, 실제값만 갱신).
def github_put(path, text, token, message, attempt=0):
    import base64
    import urllib.request
    _, sha = github_get(path, token)
    body = {"message": message, "branch": GITHUB_BRANCH,
            "content": base64.b64encode(text.encode("utf-8")).decode()}
    if sha:
        body["sha"] = sha
    url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{path}"
    request = urllib.request.Request(url, method="PUT", data=json.dumps(body).encode(),
                                     headers={"Authorization": f"Bearer {token}",
                                              "Accept": "application/vnd.github+json",
                                              "X-GitHub-Api-Version": "2022-11-28"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode())["content"]["sha"][:7]
    except Exception as exc:
        # 종목별 Actions 잡이 같은 브랜치에 동시에 커밋하면 드물게 409/422가 온다.
        # 파일은 서로 다르므로 잠깐 기다렸다가 sha를 다시 읽어 재시도하면 된다.
        if getattr(exc, "code", None) in (409, 422) and attempt < 3:
            time.sleep(5 * (attempt + 1))
            return github_put(path, text, token, message, attempt=attempt + 1)
        raise RuntimeError(f"GitHub 업로드 실패({type(exc).__name__} "
                           f"{getattr(exc, 'code', '')})") from None


if SYNC_LEDGER_TO_GITHUB:
    try:
        token = github_token()
        # 1) 원격 원장과 이번 실행 결과를 record_id로 합친다(예측은 먼저 것을 유지).
        remote_text, _ = github_get(f"{GITHUB_LEDGER_DIR}/forecast_log.csv", token)
        merged = evaluated_log
        if remote_text:
            remote_log = pd.read_csv(io.StringIO(remote_text))
            merged = pd.concat([remote_log, evaluated_log], ignore_index=True)
            merged = merged.drop_duplicates("record_id", keep="last")
        atomic_csv(merged, forecast_log_path)
        merged_daily = daily_comparison(merged)
        atomic_csv(merged_daily, STORAGE_ROOT / "daily_forecast_comparison.csv")
        atomic_csv(summarize_daily(merged_daily), STORAGE_ROOT / "forecast_accuracy_summary.csv")
        daily = merged_daily
        ledger_review = review_ledger(daily, sam, ensemble_model=_ens_name, windows=(20, 60))

        # 2) 세 파일을 저장소에 올린다.
        for name in LEDGER_FILES:
            commit = github_put(f"{GITHUB_LEDGER_DIR}/{name}",
                                (STORAGE_ROOT / name).read_text(encoding="utf-8"),
                                token, f"data: {name} ({RUN_ID})")
            print(f"GitHub 저장: {GITHUB_LEDGER_DIR}/{name} @ {commit}")
        print(f"누적 예측 {len(merged):,}건 · https://github.com/{GITHUB_REPO}/tree/"
              f"{GITHUB_BRANCH}/{GITHUB_LEDGER_DIR}")
    except Exception as exc:
        print("⚠️ GitHub 저장 실패:", exc)
        print("   이번 실행 기록은 런타임 로컬에만 남았습니다. latest_outputs.zip을 내려받아 두세요.")

## 12. 종합 보고서

지금까지의 결과를 한 화면으로 정리합니다. 위 셀들을 모두 실행한 뒤 이 셀을 실행하세요.

보고서는 여섯 부분으로 구성됩니다.

0. **어제 예측 vs 실제** — 실제로 미리 낸 예측만 채점한 결과와 최근 20·60일 누적 성능, 경고. 백테스트가 아니라 원장 기준입니다.
1. **실행 정보** — 언제·어떤 데이터로 돌렸는지, 재현에 필요한 식별자
2. **다음 거래일 방향** — 앙상블 예측과 클래스별 확률
3. **중기 가격 예측** — 점 예측(있을 때)과 구간
4. **이 예측을 어떻게 읽어야 하는가** — 갭/세션 분해와 거래 가능한 기대손익
5. **자동 판정** — 해석 체크리스트를 코드가 직접 평가한 결과

In [ ]:
def _fmt(x, kind="num"):
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "—"
    if kind == "won":
        return f"{x:,.0f}원"
    if kind == "pct":
        return f"{x:+.2%}"
    if kind == "bp":
        return f"{x:+.1f}bp"
    return f"{x:.4f}"


def build_summary():
    """보고서에 들어갈 값을 한 번에 계산한다(계산과 표시를 분리)."""
    ens = "Mean ensemble" if "Mean ensemble" in native_metrics.index else ENSEMBLE_MODELS[0]
    row = native_metrics.loc[ens]

    bal = paired_delta_ci(predictions, ens, "Always flat", metric="balanced_accuracy")
    ll = paired_delta_ci(predictions, ens, "Always flat", metric="log_loss")

    def significant(d, better):
        if not np.isfinite(d["lo"]) or not np.isfinite(d["hi"]):
            return False
        if d["lo"] > 0 and d["hi"] > 0:
            return better == "higher"
        if d["lo"] < 0 and d["hi"] < 0:
            return better == "lower"
        return False

    return {
        "ensemble": ens,
        "metrics": row,
        "bal_delta": bal,
        "ll_delta": ll,
        "bal_significant": significant(bal, "higher"),
        "ll_significant": significant(ll, "lower"),
        # 비용을 넘는 기대수익의 하한이 0보다 커야 "거래 가능"이라고 부를 수 있다.
        "session_tradeable": bool(np.isfinite(row["session_bp_lo"]) and row["session_bp_lo"] > COST_BP),
        "live": live_table.loc[ens] if ens in live_table.index else live_table.iloc[-1],
        "worse_folds": folds_worse_than_prior(predictions, ens),
    }


S = build_summary()

checks = [
    ("거래 가능한 구간(시가→종가)에 예측력이 있는가",
     bool(np.isfinite(S["metrics"]["auc_session"]) and S["metrics"]["auc_session"] > 0.53),
     f"세션 AUC {_fmt(S['metrics']['auc_session'])} · 0.5 = 정보 없음"),
    (f"왕복 비용 {COST_BP:.0f}bp 차감 후에도 기대수익이 양수인가",
     S["session_tradeable"],
     f"세션 일평균 {_fmt(S['metrics']['session_bp'], 'bp')} "
     f"[{_fmt(S['metrics']['session_bp_lo'], 'bp')}, {_fmt(S['metrics']['session_bp_hi'], 'bp')}] "
     f"→ 비용 차감 {_fmt(S['metrics']['session_bp_net'], 'bp')}"),
    ("balanced accuracy가 '항상 보합'보다 유의하게 높은가",
     S["bal_significant"],
     f"차이 {S['bal_delta']['delta']:+.4f} [{S['bal_delta']['lo']:+.4f}, {S['bal_delta']['hi']:+.4f}]"),
    ("확률 품질(log loss)이 클래스 사전확률보다 유의하게 좋은가",
     S["ll_significant"],
     f"차이 {S['ll_delta']['delta']:+.4f} [{S['ll_delta']['lo']:+.4f}, {S['ll_delta']['hi']:+.4f}]"),
    ("사전확률보다 나쁜 폴드가 절반 미만인가",
     S["worse_folds"] * 2 < len(folds),
     f"{S['worse_folds']} / {len(folds)} 폴드"),
    ("예상 시초가(갭)가 '갭 0%' 기준선을 이기는가",
     open_forecast_row["signal"] == "있음",
     f"갭 방향 AUC {_fmt(open_forecast_stats['gap_sign_auc'])} · "
     f"MAE {open_forecast_stats['raw_model_mae']:.2%} vs 0% {open_forecast_stats['zero_baseline_mae']:.2%} — "
     "09:00 이전에만 의미 있음"),
    ("중기(5·20거래일) 가격 예측에 신호가 있는가",
     any(r["signal"] == "있음" for r in price_forecast_rows if r["trading_days"] >= 5),
     " · ".join(f"{r['horizon']} {r['signal']}" for r in price_forecast_rows if r["trading_days"] >= 5)
     + " (1거래일은 갭이라 예상 시초가로 따로 봄)"),
    ("라이브 특징이 과거값으로 대체되지 않았는가",
     len(imputed_live_features) == 0,
     "대체 없음" if not imputed_live_features
     else f"{len(imputed_live_features)}개 대체: {list(imputed_live_features)}"),
    ("학습 데이터가 최신인가",
     (last_samsung_date - model_df.index.max()).days <= 5,
     f"마지막 학습일 {model_df.index.max().date()} / 최신 봉 {last_samsung_date.date()}"),
]
passed = sum(1 for _, ok, _ in checks if ok)


def _prob_bar(p_down, p_flat, p_up):
    segs = [("하락", p_down, "#b5453c"), ("보합", p_flat, "#7c848c"), ("상승", p_up, "#2b6ca3")]
    cells = "".join(
        f'<td style="width:{v*100:.1f}%;background:{c};color:#fff;text-align:center;'
        f'padding:8px 2px;font-size:12px;white-space:nowrap">'
        f'{n if v > 0.14 else ""} {format(v, ".0%") if v > 0.07 else ""}</td>'
        for n, v, c in segs)
    return ('<table style="width:100%;border-collapse:collapse;border-radius:4px;'
            f'overflow:hidden"><tr>{cells}</tr></table>')


def _range_bar(low, center, high, current):
    span = max(high - low, 1e-9)
    pos_center = (center - low) / span * 100
    pos_now = (current - low) / span * 100
    return (
        '<div style="position:relative;height:32px;margin:8px 0 4px">'
        '<div style="position:absolute;top:13px;left:0;right:0;height:8px;border-radius:4px;'
        'background:linear-gradient(90deg,#eef2f7,#c3d4e6,#eef2f7)"></div>'
        f'<div style="position:absolute;top:8px;left:{pos_now:.1f}%;width:2px;height:18px;'
        'background:#8a9199"></div>'
        f'<div style="position:absolute;top:5px;left:{pos_center:.1f}%;width:3px;height:24px;'
        'background:#1a5490"></div>'
        f'<div style="position:absolute;top:0;left:0;font-size:10px;color:#8a9199">{low:,.0f}</div>'
        f'<div style="position:absolute;top:0;right:0;font-size:10px;color:#8a9199">{high:,.0f}</div>'
        '</div>')


rows_price = ""
# 예상 시초가를 맨 위에 둔다. 1거래일 예상 종가의 '신호'는 사실상 이 갭이다.
_o = open_forecast_row
_o_signal = _o["signal"] == "있음"
_o_badge_bg, _o_badge_fg = ("#e6f2ea", "#1e6b34") if _o_signal else ("#f1f2f4", "#6b7178")
_o_point = (f'<b style="font-size:18px">{_fmt(_o["predicted_open"], "won")}</b>'
            f'<span style="color:#6b7178"> ({_fmt(_o["predicted_return"], "pct")})</span>'
            if _o_signal else
            f'<b style="font-size:18px">{_fmt(_o["center_open"], "won")}</b>'
            '<span style="color:#6b7178"> (변화 없음)</span><br>'
            '<span style="font-size:11px;color:#8a9199">검증 미통과 — 갭 0% 기준선보다 낫다는 근거를 찾지 못했습니다</span>')
rows_price += (
    '<tr style="background:#fbfcfe"><td style="padding:13px 12px;border-top:1px solid #e8e8e8;vertical-align:top;width:24%">'
    f'<b>예상 시초가</b><br>'
    f'<span style="font-size:11px;color:#8a9199">{_o["target_date"]} 09:00 · 전일 종가 대비 갭</span><br>'
    f'<span style="display:inline-block;margin-top:6px;background:{_o_badge_bg};color:{_o_badge_fg};'
    f'font-size:11px;padding:2px 9px;border-radius:10px">신호 {_o["signal"]}</span></td>'
    '<td style="padding:13px 12px;border-top:1px solid #e8e8e8">'
    f'{_o_point}'
    f'{_range_bar(_o["low_open"], _o["center_open"], _o["high_open"], _o["current_close"])}'
    f'<span style="font-size:11px;color:#8a9199">명목 {BAND_COVERAGE:.0%} 구간 · 독립 평가 적중률 {_o["band_coverage"]:.0%} · '
    f'갭 방향 AUC {_fmt(_o["gap_sign_auc"])} · 모델 MAE {_o["model_mae"]:.2%} vs '
    f'0% 기준선 {_o["zero_baseline_mae"]:.2%}<br>'
    '이 값은 09:00 이전에만 의미가 있습니다. 시가에 매수해서 얻을 수 있는 수익이 아닙니다.</span></td></tr>')
for r in price_forecast_rows:
    has_signal = r["signal"] == "있음"
    badge_bg, badge_fg = ("#e6f2ea", "#1e6b34") if has_signal else ("#f1f2f4", "#6b7178")
    # 신호가 없으면 기울기를 0으로 두므로 center_close는 정의상 현재가와 같다.
    # 빈칸으로 두면 "계산이 안 됐나" 싶으므로 숫자를 그대로 보여주되, 그 숫자가
    # 예측이 아니라 '변화 없음'이라는 것을 함께 적는다.
    point = (f'<b style="font-size:18px">{_fmt(r["predicted_close"], "won")}</b>'
             f'<span style="color:#6b7178"> ({_fmt(r["predicted_return"], "pct")})</span>'
             if has_signal else
             f'<b style="font-size:18px">{_fmt(r["center_close"], "won")}</b>'
             '<span style="color:#6b7178"> (변화 없음)</span><br>'
             '<span style="font-size:11px;color:#8a9199">검증 미통과 — 모델이 '
             '&#39;가격 유지&#39;보다 낫다는 근거를 찾지 못해 현재가를 그대로 씁니다</span>')
    rows_price += (
        '<tr><td style="padding:13px 12px;border-top:1px solid #e8e8e8;vertical-align:top;width:24%">'
        f'<b>{r["horizon"]} 뒤</b><br>'
        f'<span style="font-size:11px;color:#8a9199">{r["target_date"]} · {r["trading_days"]}거래일</span><br>'
        f'<span style="display:inline-block;margin-top:6px;background:{badge_bg};color:{badge_fg};'
        f'font-size:11px;padding:2px 9px;border-radius:10px">신호 {r["signal"]}</span></td>'
        '<td style="padding:13px 12px;border-top:1px solid #e8e8e8">'
        f'{point}'
        f'{_range_bar(r["low_close"], r["center_close"], r["high_close"], r["current_close"])}'
        f'<span style="font-size:11px;color:#8a9199">명목 {BAND_COVERAGE:.0%} 구간 · 독립 평가 적중률 {r["band_coverage"]:.0%} · '
        f'중심 {_fmt(r["center_close"], "won")} · 모델 MAE {r["model_mae"]:.2%} vs '
        f'0% 기준선 {r["zero_baseline_mae"]:.2%}</span></td></tr>')

rows_metric = ""
for name in native_metrics.index:
    m = native_metrics.loc[name]
    style = ' style="background:#f4f8fc;font-weight:600"' if name == S["ensemble"] else ""
    cell = 'style="padding:7px 11px;border-top:1px solid #eee;text-align:right"'
    rows_metric += (
        f'<tr{style}>'
        f'<td style="padding:7px 11px;border-top:1px solid #eee">{name}</td>'
        f'<td {cell}>{m["balanced_accuracy"]:.4f}<span style="color:#a5abb2;font-size:11px"> '
        f'[{m["bal_acc_lo"]:.3f}, {m["bal_acc_hi"]:.3f}]</span></td>'
        f'<td {cell}>{m["log_loss"]:.4f}<span style="color:#a5abb2;font-size:11px"> '
        f'[{m["log_loss_lo"]:.3f}, {m["log_loss_hi"]:.3f}]</span></td>'
        f'<td {cell}>{_fmt(m["auc_gap"])}</td>'
        f'<td {cell}>{_fmt(m["auc_session"])}</td>'
        f'<td {cell}>{_fmt(m["session_bp_net"], "bp")}</td></tr>')

rows_check = ""
for label, ok, detail in checks:
    mark, color = ("통과", "#1e6b34") if ok else ("미달", "#a8322a")
    rows_check += (
        f'<tr><td style="padding:8px 11px;border-top:1px solid #eee;white-space:nowrap;'
        f'color:{color};font-weight:600">{mark}</td>'
        f'<td style="padding:8px 11px;border-top:1px solid #eee">{label}</td>'
        f'<td style="padding:8px 11px;border-top:1px solid #eee;color:#8a9199;font-size:12px">'
        f'{detail}</td></tr>')

# ---- 데이터 출처 표 ---------------------------------------------------------
_ASSET_LABEL = {
    "target": f"{TARGET_NAME} (예측 대상)", "peer": "동종업체", "kospi": "KOSPI",
    "target_gdr": "런던 GDR (야간 대리변수)", "sox": "필라델피아 반도체 지수",
    "nasdaq": "Nasdaq", "sp500": "S&P 500", "micron": "Micron", "nvidia": "Nvidia",
    "tsmc_adr": "TSMC ADR", "korea_etf": "EWY 한국 ETF", "usdkrw": "원/달러 환율",
    "dxy": "달러지수", "vix": "VIX", "us10y": "미국 10년물 금리", "wti": "WTI",
}
_GROUP = {"target": "한국", "peer": "한국", "kospi": "한국", "target_gdr": "해외"}

rows_data = ""
for _q in sorted(quality, key=lambda q: (q["asset"] != "target", q["asset"])):
    _stale = _q["stale_days"]
    _warn = "" if _q["status"] == "정상" and _stale <= 7 else "color:#a8322a;font-weight:600"
    rows_data += (
        f'<tr><td style="padding:6px 10px;border-top:1px solid #eee">'
        f'{_ASSET_LABEL.get(_q["asset"], _q["asset"])}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid #eee;font-family:ui-monospace,monospace;'
        f'font-size:12px;color:#6b7178">{_q["ticker"]}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid #eee;text-align:right">{_q["rows"]:,}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid #eee;text-align:right;{_warn}">'
        f'{pd.Timestamp(_q["last"]).date()}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid #eee;text-align:right;{_warn}">'
        f'{_q["status"]}</td></tr>')

if MACRO_ACTIVE:
    _macro_html = "<ul style='margin:6px 0 0;padding-left:18px'>" + "".join(
        f"<li>{_spec['name']} — KOSIS <code>{_spec['tblId']}</code>, "
        f"최신월 {str(macro_info.get('latest_month', {}).get(_key, '?'))[:7]}</li>"
        for _key, _spec in MACRO_SERIES.items()) + "</ul>"
else:
    _macro_html = (f"<div style='color:#a8322a'>미포함 — "
                   f"{macro_info.get('reason', '')}</div>")

_versions = " · ".join(f"{_k} {_v}" for _k, _v in VERSIONS.items() if _v)

verdict = ("이 설정에서는 <b>거래 가능한 예측력이 확인되지 않습니다.</b> "
           "방향 지표가 좋아 보이는 것은 대부분 09:00 시가에 이미 반영된 갭 때문입니다."
           if not S["session_tradeable"] else
           "거래 가능한 구간에서도 비용 차감 후 기대수익의 하한이 0보다 큽니다. "
           "표본 외 기간으로 재검증한 뒤에 판단하세요.")

# 상단 안내: 무엇이 빠진 채로 계산된 결과인지 먼저 알린다.
notices = []
if not MACRO_ACTIVE:
    notices.append("월별 지표(선행지수·반도체 수출) <b>미포함</b> — " + str(macro_info.get("reason", "")))
if not (RUN_TRANSFORMER and TORCH_AVAILABLE):
    notices.append("Transformer 미실행 (RUN_TRANSFORMER=True 로 켤 수 있습니다)")
if not RUN_KRONOS:
    notices.append("Kronos 미실행 (기본 꺼짐 — 기준선보다 성능이 나빴습니다)")
notice_html = ""
if notices:
    notice_html = (
        '<div style="background:#fdf3f2;border-left:4px solid #b5453c;padding:12px 16px;'
        'border-radius:0 5px 5px 0;margin-bottom:18px;font-size:13px">'
        '<b>이 실행에서 빠진 것</b><ul style="margin:6px 0 0;padding-left:18px">'
        + "".join(f"<li>{n}</li>" for n in notices) + "</ul></div>")

# ---- 어제 예측 vs 실제 (실제 사전 예측만, 백테스트 아님) --------------------
def _ledger_section_html(review, ensemble_name):
    cell = 'style="padding:7px 11px;border-top:1px solid #eee;text-align:right"'
    head = ('<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
            '어제 예측 vs 실제 <span style="font-weight:400;color:#8a9199;font-size:12px">'
            '&nbsp;실제로 미리 낸 예측만 채점 · 백테스트 숫자가 아님</span></h3>')
    if not review["n_scored_days"]:
        return head + ('<div style="border:1px solid #e5e5e5;border-radius:6px;padding:14px;font-size:13px;color:#6b7178">'
                       '아직 채점된 사전 예측이 없습니다. 오늘 예측은 다음 거래일 실행에서 실제 시가·종가와 대조됩니다.</div>')
    latest = review["latest"]
    d = latest[(latest["kind"] == "direction") & (latest["model"] == ensemble_name)]
    o = latest[latest["kind"] == "open"]
    p1 = latest[(latest["kind"] == "price") & (latest["horizon_days"] == 1)]
    rows = ""
    def _row(label, predicted, actual, verdict, ok):
        color = "#1e6b34" if ok else "#a8322a"
        return (f'<tr><td style="padding:8px 11px;border-top:1px solid #eee">{label}</td>'
                f'<td {cell}>{predicted}</td><td {cell}>{actual}</td>'
                f'<td {cell};color:{color};font-weight:600">{verdict}</td></tr>')
    if len(d):
        r = d.iloc[0]
        actual_label = LABEL_NAMES.get(int(r["actual_class"]), "?") if pd.notna(r["actual_class"]) else "—"
        rows += _row("종가 방향", f'{r["prediction"]} (상승 {r["p_up"]:.0%}·보합 {r["p_flat"]:.0%}·하락 {r["p_down"]:.0%})',
                     f'{actual_label} ({_fmt(r["actual_return"], "pct")}, 밴드 ±{r["band"]:.2%})',
                     "적중" if r["direction_correct"] == 1 else "미적중", r["direction_correct"] == 1)
    if len(o):
        r = o.iloc[0]
        pred = (f'{_fmt(r["predicted_open"], "won")} ({_fmt(r["predicted_return"], "pct")})'
                if pd.notna(r["predicted_open"]) else f'{_fmt(r["center_open"], "won")} (신호 없음)')
        rows += _row("시초가(갭)", pred, f'{_fmt(r["actual_open"], "won")} ({_fmt(r["actual_gap"], "pct")})',
                     f'구간 {"적중" if r["interval_hit"] == 1 else "이탈"} · 오차 {_fmt(r["return_error"], "pct") if pd.notna(r["return_error"]) else "—"}',
                     r["interval_hit"] == 1)
    if len(p1):
        r = p1.iloc[0]
        pred = (f'{_fmt(r["predicted_close"], "won")} ({_fmt(r["predicted_return"], "pct")})'
                if pd.notna(r["predicted_close"]) else f'{_fmt(r["center_close"], "won")} (신호 없음)')
        rows += _row("1거래일 종가", pred,
                     f'{_fmt(r["actual_close"], "won")} ({_fmt(r["actual_return"], "pct")} = 갭 {_fmt(r["actual_gap"], "pct")} + 세션 {_fmt(r["actual_session"], "pct")})',
                     f'구간 {"적중" if r["interval_hit"] == 1 else "이탈"}', r["interval_hit"] == 1)
    table = ('<div style="overflow-x:auto"><table style="width:100%;min-width:520px;border-collapse:collapse;'
             'font-size:13px;border:1px solid #e5e5e5"><tr style="background:#fafafa;font-size:11px;color:#6b7178">'
             '<th style="padding:8px 11px;text-align:left">항목</th><th style="padding:8px 11px;text-align:right">예측</th>'
             '<th style="padding:8px 11px;text-align:right">실제</th><th style="padding:8px 11px;text-align:right">판정</th></tr>'
             f'{rows}</table></div>')
    # 누적 창
    roll = review["rolling"]
    rrows = ""
    for _, r in roll.iterrows():
        if r["kind"] == "direction":
            label, detail = "종가 방향", (f'적중률 {r["hit_rate"]:.0%} (보합 비중 {r["flat_share"]:.0%}) · '
                                        f'log loss {r["mean_log_loss"]:.3f} vs 빈도기준 {r["prior_log_loss"]:.3f}')
        else:
            label = "시초가(갭)" if r["kind"] == "open" else f'{int(r["horizon_days"])}거래일 종가'
            detail = (f'구간 적중 {_fmt(r["interval_coverage"], "num") if pd.isna(r["interval_coverage"]) else format(r["interval_coverage"], ".0%")} · '
                      f'MAE {r["mae_return"]:.2%} vs 변화없음 {r["zero_mae_return"]:.2%} · 신호 {int(r["signal_days"])}/{int(r["n"])}일')
            if pd.notna(r["realized_slope"]):
                detail += f' · 실현 기울기 {r["realized_slope"]:.2f} / OOF {r["oof_slope_mean"]:.2f}'
        rrows += (f'<tr><td style="padding:7px 11px;border-top:1px solid #eee">최근 {int(r["window"])}일 · {label}</td>'
                  f'<td style="padding:7px 11px;border-top:1px solid #eee;text-align:right">{int(r["n"])}</td>'
                  f'<td style="padding:7px 11px;border-top:1px solid #eee">{detail}</td></tr>')
    rtable = ('<div style="overflow-x:auto;margin-top:10px"><table style="width:100%;min-width:520px;border-collapse:collapse;'
              'font-size:12px;border:1px solid #e5e5e5"><tr style="background:#fafafa;font-size:11px;color:#6b7178">'
              '<th style="padding:8px 11px;text-align:left">창</th><th style="padding:8px 11px;text-align:right">n</th>'
              '<th style="padding:8px 11px;text-align:left">누적 성능 (사전 예측만)</th></tr>'
              f'{rrows}</table></div>')
    alerts = ""
    if review["alerts"]:
        alerts = ('<div style="background:#fdf3f2;border-left:4px solid #b5453c;padding:10px 14px;margin-top:10px;font-size:13px">'
                  '<b>경고</b><ul style="margin:6px 0 0;padding-left:18px">'
                  + "".join(f"<li>{a}</li>" for a in review["alerts"]) + "</ul>"
                  '<div style="font-size:11px;color:#8a9199;margin-top:4px">경고는 판단 근거이며 자동으로 설정을 바꾸지 않습니다. '
                  '같은 경고가 두 달 이상 이어질 때 사람이 조정합니다.</div></div>')
    note = (f'<div style="font-size:11px;color:#8a9199;margin-top:6px">채점된 예측일 {review["n_scored_days"]}일 · '
            f'마지막 채점 {review["latest_date"].date()} · 하루 결과는 잡음입니다(1거래일 MAE ≈ 2~3%). '
            '판단은 60일 창으로 하세요.</div>')
    return head + table + rtable + alerts + note


ledger_html = _ledger_section_html(ledger_review, S["ensemble"])

tiles = [
    ("학습 데이터", f"{len(model_df):,}행", f"{model_df.index.min().date()} ~ {model_df.index.max().date()}"),
    ("워크포워드 폴드", f"{len(folds)}", f"검증일 {int(S['metrics']['n']):,}일"),
    ("특징 수", f"{len(feature_cols)}", f"타깃 {TARGET_MODE}"),
    ("자동 판정", f"{passed} / {len(checks)}", "체크리스트 통과"),
]
tiles_html = "".join(
    '<div style="flex:1;min-width:150px;border:1px solid #e5e5e5;border-radius:6px;padding:13px 15px">'
    f'<div style="font-size:11px;color:#8a9199">{label}</div>'
    f'<div style="font-size:20px;font-weight:600;margin:2px 0">{value}</div>'
    f'<div style="font-size:11px;color:#8a9199">{note}</div></div>'
    for label, value, note in tiles)

html = (
    '<div style="font-family:-apple-system,\'Malgun Gothic\',sans-serif;max-width:980px;'
    'line-height:1.65;color:#1a1a1a">'

    '<div style="border-bottom:3px solid #1a1a1a;padding-bottom:11px;margin-bottom:18px">'
    '<div style="font-size:11px;letter-spacing:2px;color:#8a9199">'
    f'{TARGET_NAME} {TARGET_SPEC["ticker"]} · DIRECTION &amp; PRICE FORECAST</div>'
    '<h2 style="margin:6px 0 5px;font-size:27px">종합 보고서</h2>'
    f'<div style="font-size:12px;color:#8a9199">예측일 <b>{prediction_date.date()}</b> · '
    f'데이터 기준 <b>{last_samsung_date.date()}</b> · run_id <code>{RUN_ID}</code></div></div>'

    f'{notice_html}'
    f'<div style="display:flex;gap:14px;flex-wrap:wrap;margin-bottom:22px">{tiles_html}</div>'

    f'{ledger_html}'

    '<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
    f'1. 다음 거래일 방향 <span style="font-weight:400;color:#8a9199;font-size:12px">'
    f'&nbsp;{S["ensemble"]}</span></h3>'
    '<div style="border:1px solid #e5e5e5;border-radius:6px;padding:15px">'
    f'<div style="font-size:24px;font-weight:700;margin-bottom:9px">{S["live"]["prediction"]}</div>'
    f'{_prob_bar(S["live"]["p_down"], S["live"]["p_flat"], S["live"]["p_up"])}'
    f'<div style="font-size:11px;color:#8a9199;margin-top:8px">보합 밴드 ±{live_band:.2%} · '
    '세 확률 중 최댓값이 예측 클래스입니다.</div></div>'

    '<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
    '2. 예상 시초가와 중기 예상 종가</h3>'
    '<div style="overflow-x:auto;-webkit-overflow-scrolling:touch">'
    '<table style="width:100%;min-width:320px;border-collapse:collapse;border:1px solid #e5e5e5">'
    f'{rows_price}</table></div>'
    '<div style="font-size:11px;color:#8a9199;margin-top:6px">1거래일 뒤 예상 종가가 검증을 통과하는 것은 '
    '거의 전부 전일 종가→시가 갭 때문입니다(아래 3절). 그 갭은 위 "예상 시초가"가 따로 예측합니다.</div>'

    '<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
    '3. 이 예측을 어떻게 읽어야 하는가</h3>'
    '<div style="background:#fdf8ec;border-left:4px solid #c8952a;padding:14px 17px;'
    'border-radius:0 5px 5px 0">'
    f'<b>{S["ensemble"]}</b>의 방향 점수는 <b>전일 종가→시가 갭</b>의 부호를 AUC '
    f'<b>{_fmt(S["metrics"]["auc_gap"])}</b>로 맞히지만, <b>시가→종가 세션</b>의 부호는 AUC '
    f'<b>{_fmt(S["metrics"]["auc_session"])}</b>입니다(0.5 = 무작위). '
    '갭은 09:00 시가에 이미 가격에 반영되므로 07:00 예측으로는 취할 수 없습니다.<br>'
    f'모델 방향대로 시가에 진입해 종가에 청산하면 일평균 '
    f'<b>{_fmt(S["metrics"]["session_bp"], "bp")}</b> '
    f'[{_fmt(S["metrics"]["session_bp_lo"], "bp")}, {_fmt(S["metrics"]["session_bp_hi"], "bp")}], '
    f'왕복 비용 {COST_BP:.0f}bp 차감 후 '
    f'<b>{_fmt(S["metrics"]["session_bp_net"], "bp")}</b>입니다.</div>'

    '<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
    '4. 모델 성능 <span style="font-weight:400;color:#8a9199;font-size:12px">'
    '&nbsp;대괄호는 월 블록 부트스트랩 95% 신뢰구간</span></h3>'
    '<div style="overflow-x:auto;-webkit-overflow-scrolling:touch">'
    '<table style="width:100%;min-width:520px;border-collapse:collapse;font-size:13px;border:1px solid #e5e5e5">'
    '<tr style="background:#fafafa;font-size:11px;color:#6b7178;letter-spacing:.5px">'
    '<th style="padding:8px 11px;text-align:left">모델</th>'
    '<th style="padding:8px 11px;text-align:right">BAL. ACC</th>'
    '<th style="padding:8px 11px;text-align:right">LOG LOSS</th>'
    '<th style="padding:8px 11px;text-align:right">갭 AUC</th>'
    '<th style="padding:8px 11px;text-align:right">세션 AUC</th>'
    '<th style="padding:8px 11px;text-align:right">순 BP/일</th></tr>'
    f'{rows_metric}</table></div>'

    '<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
    '5. 자동 판정</h3>'
    '<div style="overflow-x:auto;-webkit-overflow-scrolling:touch">'
    '<table style="width:100%;min-width:520px;border-collapse:collapse;font-size:13px;border:1px solid #e5e5e5">'
    f'{rows_check}</table></div>'

    '<h3 style="font-size:15px;margin:24px 0 9px;padding-bottom:6px;border-bottom:1px solid #ddd">'
    '6. 이 보고서의 데이터</h3>'
    '<div style="font-size:13px;color:#6b7178;margin-bottom:10px">'
    f'예측 대상 <b style="color:#1a1a1a">{TARGET_NAME} {TARGET_SPEC["ticker"]}</b> · '
    f'수집 기간 {START_DATE} ~ {last_samsung_date.date()} · '
    f'특징 {len(feature_cols)}개(시세 {len(feature_cols) - len(macro_feature_cols)} + '
    f'월별 지표 {len(macro_feature_cols)})</div>'
    '<div style="overflow-x:auto;-webkit-overflow-scrolling:touch">'
    '<table style="width:100%;min-width:520px;border-collapse:collapse;font-size:13px;'
    'border:1px solid #e5e5e5">'
    '<tr style="background:#fafafa;font-size:11px;color:#6b7178;letter-spacing:.5px">'
    '<th style="padding:8px 10px;text-align:left">자산</th>'
    '<th style="padding:8px 10px;text-align:left">티커</th>'
    '<th style="padding:8px 10px;text-align:right">행 수</th>'
    '<th style="padding:8px 10px;text-align:right">마지막 봉</th>'
    '<th style="padding:8px 10px;text-align:right">상태</th></tr>'
    f'{rows_data}</table></div>'
    '<div style="margin-top:12px;font-size:13px"><b>월별 지표</b>'
    f'{_macro_html}</div>'
    '<div style="margin-top:12px;font-size:12px;color:#8a9199">'
    f'데이터 스냅샷 <code>{DATA_SNAPSHOT_HASH}</code> · 변동성 모형 <code>{VOL_MODEL}</code> · '
    f'{_versions}<br>'
    'Yahoo Finance는 연구·교육용 편의 데이터입니다. 시세는 자산군별 마감 시각 기준으로 '
    '미완성 봉을 제거하고, 한국 지수는 대상 종목 거래일 달력에 맞춘 뒤 차분합니다.</div>'

    '<div style="margin-top:20px;padding:14px 17px;background:#f5f6f8;border-radius:6px;'
    f'font-size:13px"><b>결론.</b> {verdict} '
    '이 노트북은 연구·교육용이며 투자 자문이 아닙니다. 결과는 '
    f'<code>{OUTPUT_DIR}</code> 와 <code>forecast_log.csv</code> 에 기록되었습니다.</div>'
    '</div>')

try:
    from IPython.display import HTML, display as _display_html
    _display_html(HTML(html))
except Exception:
    print("=" * 74)
    print(f"종합 보고서 | 예측일 {prediction_date.date()} | 데이터 기준 {last_samsung_date.date()}")
    for _n in notices:
        print("  ! 빠진 것:", _n.replace("<b>", "").replace("</b>", ""))
    print("=" * 74)
    if ledger_review["n_scored_days"]:
        print(f"채점된 사전 예측 {ledger_review['n_scored_days']}일 · 마지막 {ledger_review['latest_date'].date()}")
        for _a in ledger_review["alerts"]:
            print("  ! 경고:", _a)
    print(f"방향({S['ensemble']}): {S['live']['prediction']}  "
          f"하락 {S['live']['p_down']:.1%} / 보합 {S['live']['p_flat']:.1%} / 상승 {S['live']['p_up']:.1%}")
    print(f"예상 시초가: {_fmt(open_forecast_row['predicted_open'], 'won')} ({_fmt(open_forecast_row['predicted_return'], 'pct')}) · "
          f"신호 {open_forecast_row['signal']} · 구간 {open_forecast_row['low_open']:,.0f}~{open_forecast_row['high_open']:,.0f}원")
    for r in price_forecast_rows:
        if r["signal"] == "있음":
            print(f"{r['horizon']} 뒤: {r['predicted_close']:,.0f}원 ({r['predicted_return']:+.2%}) · "
                  f"명목 {BAND_COVERAGE:.0%} 구간 {r['low_close']:,.0f}~{r['high_close']:,.0f}원")
        else:
            print(f"{r['horizon']} 뒤: 점 예측 없음 · 명목 {BAND_COVERAGE:.0%} 구간 "
                  f"{r['low_close']:,.0f}~{r['high_close']:,.0f}원 (중심 {r['center_close']:,.0f}원)")
    print(f"갭 AUC {S['metrics']['auc_gap']:.3f} / 세션 AUC {S['metrics']['auc_session']:.3f} / "
          f"세션 순 {S['metrics']['session_bp_net']:+.1f}bp")
    print("-" * 74)
    for label, ok, detail in checks:
        print(f"[{'통과' if ok else '미달'}] {label}\n         {detail}")
    print("-" * 74)
    print("결론:", verdict.replace("<b>", "").replace("</b>", ""))
    print(f"데이터: {TARGET_NAME} {TARGET_SPEC['ticker']} · "
          f"{START_DATE}~{last_samsung_date.date()} · 자산 {len(quality)}개 · "
          f"특징 {len(feature_cols)}개 · 스냅샷 {DATA_SNAPSHOT_HASH}")


# ---- 보고서를 독립 HTML로 저장하고 GitHub Pages에 발행 --------------------------
# 노트북을 열지 않고도 브라우저에서 최신 보고서만 볼 수 있게 한다.
# docs/index.html      항상 최신 보고서
# docs/reports/<날짜>.html  예측일별 보관본(같은 날 다시 돌리면 덮어씀)
PUBLISH_REPORT_TO_PAGES = True

# ---- 조회수 -----------------------------------------------------------------
# 종목별로 따로 센다. 날짜별 보관본도 같은 키를 쓰므로, 한 종목의 조회수는 최신
# 보고서와 보관본을 합친 값이다. 카운터가 죽어도 보고서는 그대로 보여야 하므로
# 호출 실패는 조용히 무시한다.
_counter_html = ""
if COUNTER_ENDPOINT:
    _counter_html = (
        '<div style="margin-top:10px;font-variant-numeric:tabular-nums">'
        '조회 <span id="view-count">—</span></div>'
        '<script>(function(){'
        f'var E="{COUNTER_ENDPOINT}",P="{TARGET}";'
        'var el=document.getElementById("view-count");if(!el)return;'
        'fetch(E+"/hit?page="+encodeURIComponent(P))'
        '.then(function(r){return r.ok?r.json():null;})'
        '.then(function(d){if(d&&typeof d.total==="number")'
        'el.textContent=d.total.toLocaleString("ko-KR");})'
        '.catch(function(){});})();</script>')

_page = (
    '<!doctype html>\n<html lang="ko"><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width, initial-scale=1">'
    f'<title>{TARGET_NAME} 예측 보고서 {prediction_date.date()}</title>'
    # 넓은 표는 자기 상자 안에서만 가로로 스크롤되고, 페이지 자체는 절대
    # 옆으로 밀리지 않게 한다(휴대폰에서 보고서를 읽을 때 중요).
    '<style>html,body{overflow-x:hidden}'
    'body{margin:0;padding:24px 20px 48px;background:#fff;max-width:100%;'
    '-webkit-font-smoothing:antialiased}'
    # 긴 경로·run_id가 줄바꿈 없이 페이지 전체를 가로로 밀어내는 것을 막는다.
    'code{word-break:break-all;overflow-wrap:anywhere}'
    'img{max-width:100%}'
    '@media(max-width:640px){body{padding:16px 12px 32px}}</style></head><body>'
    # 자동 발행이 실패하면 Pages에는 옛 보고서가 그대로 남는다. 보는 쪽 브라우저에서
    # 예측일과 오늘(KST)을 비교해, 평일인데 예측일이 지났으면 상단에 알린다.
    # 휴장일에는 오탐이 나므로 문구에 그 가능성을 함께 적는다.
    '<div id="stale-note" hidden style="max-width:980px;margin:0 auto 18px;padding:12px 16px;'
    'background:#fff4e5;border:1px solid #f0c58a;border-radius:6px;'
    'font-family:-apple-system,\'Malgun Gothic\',sans-serif;font-size:13px;color:#7a4b00"></div>'
    '<script>(function(){var d=new Date(Date.now()+9*3600e3);'
    f'var p="{prediction_date.date().isoformat()}";'
    'var t=d.toISOString().slice(0,10),w=d.getUTCDay();'
    'if(t>p&&w>=1&&w<=5){var n=Math.round((Date.parse(t)-Date.parse(p))/864e5);'
    'var el=document.getElementById("stale-note");if(!el)return;el.hidden=false;'
    'el.textContent="이 보고서는 "+p+" 예측분입니다("+n+"일 지남). 오늘 자 보고서가 아직 '
    '발행되지 않았습니다. 휴장일이면 정상이고, 아니면 자동 실행(Actions)이 실패했을 수 있습니다.";}'
    '})();</script>'
    f'{html}'
    '<div style="max-width:980px;margin:28px auto 0;padding-top:14px;'
    'border-top:1px solid #e5e5e5;font-family:-apple-system,\'Malgun Gothic\',sans-serif;'
    'font-size:12px;color:#8a9199">'
    f'생성 {pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y-%m-%d %H:%M")} KST · '
    f'run_id <code>{RUN_ID}</code> · '
    f'<a href="https://github.com/{GITHUB_REPO}" style="color:#1a5490">저장소</a> · '
    f'<a href="https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/{GITHUB_LEDGER_DIR}" '
    'style="color:#1a5490">예측 원장</a>'
    f'{_counter_html}'
    '</div></body></html>')

_report_path = STORAGE_ROOT / "report.html"
_report_path.write_text(_page, encoding="utf-8")
print("보고서 HTML 저장:", _report_path)

PUBLISH_STATUS = globals().get("PUBLISH_STATUS", {})

if PUBLISH_REPORT_TO_PAGES and SYNC_LEDGER_TO_GITHUB:
    try:
        _token = github_token()
        _dated = f"{GITHUB_PAGES_DIR}/reports/{prediction_date.date().isoformat()}.html"
        for _path in [f"{GITHUB_PAGES_DIR}/index.html", _dated]:
            _sha = github_put(_path, _page, _token, f"report: {prediction_date.date()} ({RUN_ID})")
            print(f"GitHub Pages 발행: {_path} @ {_sha}")
        _owner, _name = GITHUB_REPO.split("/")
        PUBLISH_STATUS[TARGET] = ("발행됨", f"https://{_owner}.github.io/{_name}/{TARGET}/")
        print(f"→ https://{_owner}.github.io/{_name}/{TARGET}/")
        print("  (첫 발행이라면 저장소 Settings → Pages에서 Source를 "
              "'Deploy from a branch → main → /docs'로 한 번 설정해야 합니다.)")
    except Exception as exc:
        PUBLISH_STATUS[TARGET] = ("실패", f"{type(exc).__name__}: {str(exc)[:70]}")
        print("⚠️ GitHub Pages 발행 실패:", exc)
        print("   보고서는", _report_path, "에 저장되어 있습니다.")
elif PUBLISH_REPORT_TO_PAGES:
    # 이 분기는 SYNC_LEDGER_TO_GITHUB가 꺼졌을 때만 온다. 왜 꺼졌는지는 첫 셀이
    # PUBLISH_SKIP_REASON에 남겨 둔다(발행 스위치 / 토큰 없음 / 원장 조회 실패).
    _reason = globals().get("PUBLISH_SKIP_REASON") or "발행이 꺼져 있습니다"
    PUBLISH_STATUS[TARGET] = ("건너뜀", _reason)
    print("ℹ️ Pages 발행을 건너뜁니다 —", _reason)
    print("   report.html을 내려받아 브라우저에서 열 수 있습니다.")

## 13. Transformer 실험 기록 (자동 저장)

Transformer가 현재 앙상블(Logistic + LightGBM)에 무엇을 보태는지 **매 실행마다 같은 방식으로 재고**, 결과를 저장소 `experiments/transformer/`에 남깁니다. 예측 원장·보고서와는 별개의 기록이라 Colab에서도 올립니다(발행 계보가 섞이지 않습니다).

- 판정 규칙은 8절과 같습니다: Transformer를 넣은 3모델 평균과 현재 2모델 평균의 log loss 쌍체 차이 95% 신뢰구간이 **0을 포함하면 동률**, 상한이 0 아래면 편입 권장.
- torch가 없는 환경(GitHub Actions)에서는 건너뜁니다. 그래서 `RUN_TRANSFORMER = True`여도 매일 발행은 지금처럼 2모델로 갑니다.
- 편입은 자동으로 하지 않습니다. 누적된 `experiments/transformer/summary.csv`를 보고 `ENSEMBLE_MODELS`를 바꾸는 것은 사람이 결정합니다.

In [ ]:
# == transformer-experiment ==
# Transformer가 현재 앙상블에 무엇을 보태는지 매 실행마다 재고 experiments/에 남긴다.
# 예측 원장·보고서와 별개의 기록이므로 발행 스위치와 무관하게 Colab에서도 올린다.
EXPERIMENT_DIR = "experiments/transformer"
_models_seen = set(predictions["model"])
if not (RUN_TRANSFORMER and TORCH_AVAILABLE and "Transformer" in _models_seen):
    print("Transformer가 이 실행에 없어 실험 기록을 남기지 않습니다"
          + ("" if TORCH_AVAILABLE else " (torch 없음 — GitHub Actions에서는 정상)") + ".")
else:
    # 3모델 단순 평균의 OOF. 6.5절의 2모델 평균과 같은 방식으로 만든다.
    _members = [m for m in ENSEMBLE_MODELS if m in _models_seen] + ["Transformer"]
    _wide = (predictions[predictions["model"].isin(_members)]
             .pivot_table(index="date", columns="model", values=PROB_COLS).dropna())
    _p3 = np.stack([_wide[[(p, m) for p in PROB_COLS]].to_numpy() for m in _members], axis=1).mean(axis=1)
    _dates3 = pd.DatetimeIndex(_wide.index)
    _fold3 = predictions[predictions["model"] == _members[0]].set_index("date")["fold"].reindex(_dates3)
    _plus = prediction_frame("Mean ensemble +Transformer", _dates3,
                             model_df.loc[_dates3, "target"].to_numpy(dtype=int), _p3, _fold3.to_numpy())
    _preds = pd.concat([predictions, _plus], ignore_index=True)

    _rows = []
    for _a, _b in [("Mean ensemble +Transformer", "Mean ensemble"),
                   ("Transformer", "Mean ensemble"), ("Transformer", "Always flat")]:
        for _metric in ("log_loss", "balanced_accuracy"):
            _d = paired_delta_ci(_preds, _a, _b, _metric)
            _sig = bool(pd.notna(_d["lo"]) and (_d["lo"] > 0) == (_d["hi"] > 0))
            _better = (_d["delta"] < 0) if _metric == "log_loss" else (_d["delta"] > 0)
            _rows.append({"비교": f"{_a} − {_b}", "지표": _metric, "n": int(_d["n"]),
                          "차이": float(_d["delta"]), "lo": float(_d["lo"]), "hi": float(_d["hi"]),
                          "판정": ("유의하게 " + ("우위" if _better else "열위")) if _sig else "동률(CI가 0 포함)"})
    experiment_table = pd.DataFrame(_rows)
    _cols = ["n", "balanced_accuracy", "log_loss", "brier"]
    _metrics = summarize_predictions(_preds, with_ci=False)
    _show = [m for m in ["Always flat", "Logistic", "LightGBM", "Transformer",
                         "Mean ensemble", "Mean ensemble +Transformer"] if m in _metrics.index]
    print("■ Transformer 실험 — 모델별 성능 (외부 평가 구간)")
    display(_metrics.loc[_show, _cols].style.format("{:.4f}", na_rep="—"))
    print("\n■ 쌍체 차이 (95% CI, 달력 월 블록 부트스트랩) — log_loss는 음수가 좋고, balanced_accuracy는 양수가 좋다")
    display(experiment_table.set_index(["비교", "지표"])
            .style.format({"차이": "{:+.4f}", "lo": "{:+.4f}", "hi": "{:+.4f}"}, na_rep="—"))

    _key = experiment_table[(experiment_table["비교"] == "Mean ensemble +Transformer − Mean ensemble")
                            & (experiment_table["지표"] == "log_loss")].iloc[0]
    if pd.notna(_key["hi"]) and _key["hi"] < 0:
        experiment_verdict = "편입 권장 — Transformer를 넣은 3모델 평균이 현재 2모델 평균보다 log loss가 유의하게 낮다"
    elif pd.notna(_key["lo"]) and _key["lo"] > 0:
        experiment_verdict = "편입 반대 — 넣으면 유의하게 나빠진다"
    else:
        experiment_verdict = "동률 — 현재 구성 유지 (CI가 0을 포함)"
    print(f"\n■ 판정: {experiment_verdict}")
    print("   (편입은 자동으로 하지 않는다. summary.csv가 쌓이면 ENSEMBLE_MODELS를 바꿀지 사람이 정한다.)")

    experiment_record = {
        "run_id": RUN_ID, "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "target": TARGET, "runtime": RUNTIME, "versions": VERSIONS, "versions_hash": VERSIONS_HASH,
        "data_snapshot_hash": DATA_SNAPSHOT_HASH, "config_hash": config_hash,
        "quick_mode": bool(QUICK_MODE), "seq_len": SEQ_LEN, "transformer_epochs": TRANSFORMER_EPOCHS,
        "transformer_patience": TRANSFORMER_PATIENCE, "device": str(DEVICE),
        "n_folds": len(folds), "evaluation_days": int(_key["n"]),
        "metrics": _metrics.loc[_show, _cols].astype(float).reset_index().to_dict("records"),
        "paired": experiment_table.to_dict("records"),
        "verdict": experiment_verdict,
    }
    _summary_row = {
        "run_id": RUN_ID, "created_at_utc": experiment_record["created_at_utc"], "target": TARGET,
        "runtime": RUNTIME, "quick_mode": bool(QUICK_MODE), "device": str(DEVICE), "n": int(_key["n"]),
        "logloss_transformer": float(_metrics.loc["Transformer", "log_loss"]),
        "logloss_ensemble2": float(_metrics.loc["Mean ensemble", "log_loss"]),
        "logloss_ensemble3": float(_metrics.loc["Mean ensemble +Transformer", "log_loss"]),
        "delta_3_vs_2": float(_key["차이"]), "lo": float(_key["lo"]), "hi": float(_key["hi"]),
        "verdict": experiment_verdict.split(" — ")[0],
    }
    _local = STORAGE_ROOT / "experiments"
    _local.mkdir(parents=True, exist_ok=True)
    _payload = json.dumps(experiment_record, ensure_ascii=False, indent=2, default=str)
    (_local / f"transformer_{RUN_ID}.json").write_text(_payload, encoding="utf-8")
    try:
        _token = github_token()
        if not _token:
            raise RuntimeError("GITHUB_TOKEN 없음")
        github_put(f"{EXPERIMENT_DIR}/{TARGET}/{RUN_ID}.json", _payload, _token,
                   f"experiment: transformer {TARGET} ({RUN_ID})")
        _path = f"{EXPERIMENT_DIR}/summary.csv"
        _old, _ = github_get(_path, _token)
        _prev = pd.read_csv(io.StringIO(_old)) if _old else pd.DataFrame()
        _all = (pd.concat([_prev, pd.DataFrame([_summary_row])], ignore_index=True)
                .drop_duplicates(["run_id", "target"], keep="last"))
        _sha = github_put(_path, _all.to_csv(index=False), _token, f"experiment: summary ({RUN_ID})")
        print(f"실험 기록 저장: {EXPERIMENT_DIR}/{TARGET}/{RUN_ID}.json · summary.csv @ {_sha} (누적 {len(_all)}건)")
        print(f"→ https://github.com/{GITHUB_REPO}/blob/{GITHUB_BRANCH}/{_path}")
    except Exception as exc:
        print(f"⚠️ 실험 기록을 GitHub에 올리지 못했습니다: {exc}")
        print(f"   로컬 사본: {_local}")

## 14. 결과 해석 체크리스트

1. **갭/세션 분해 표**의 `auc_session`이 0.5보다 의미 있게 큰가? `session_bp_net`(비용 차감 후)이 0보다 큰가? 그렇지 않다면 이 모델로 거래할 수 있는 예측력은 없다.
2. `balanced_accuracy`의 신뢰구간이 "항상 보합"의 값을 배제하는가? 점추정치만 보고 판단하지 말 것.
3. 쌍체 비교표에서 **CI가 0을 포함하는 차이는 전부 "동률"** 이다. 시드를 바꾸면 순위가 뒤집힌다.
4. `folds_worse_than_prior`가 몇 폴드인가? 절반 이상이면 그 모델은 확률 모델로서 클래스 빈도보다 못하다.
5. 하락·보합·상승 중 한 클래스만 잘 맞히는 것은 아닌가(혼동행렬 확인)?
6. **1거래일 예상 종가의 신호는 갭이다.** 종가가 아니라 '예상 시초가'로 읽어야 하며, 그 값은 09:00 이전에만 의미가 있다. 원장에서는 `kind="open"` 행이 시가로 채점된다.
7. 중기 가격 예측의 `signal` 열이 "없음"이면 예상 종가는 현재가와 같다는 뜻이다. 구간(`low_close`~`high_close`) 폭을 반드시 함께 보라.
8. `balanced_accuracy`는 `VOL_BAND_MULT`에 따라 단조 증가한다. **이 지표로 밴드를 튜닝하지 말 것.**
9. 라이브 예측에 "과거값으로 대체된 특징" 경고가 떴는가? 떴다면 그 예측은 신뢰하지 말고 데이터부터 고쳐라.

### 다음 개선 순서

- **09:00 제품으로 전환**: `TARGET_MODE="open_to_close"` + 09:00 시가(`LIVE_OPEN_PRICE`) 필수 입력. 일간 종가 데이터로는 세션 구간에 정보가 거의 없음이 확인되었으므로, 08:50 예상체결가·호가 잔량 같은 장직전 정보가 유일한 돌파구다.
- KRX 투자자별 수급(외국인·기관 순매수, `pykrx`로 d+1 시점 사용), VKOSPI, 실적·배당 캘린더 추가.
- 폴드 0–5에서 모든 재량 선택(밴드 배수, `C`, `num_leaves`, 피처 목록)을 동결하고 6–10을 사전등록 홀드아웃으로 보고.
- `src/predict_stock/` 패키지 + CLI(`backtest / predict / score`)로 분리해 매일 예측·채점을 자동화.
- 최소 월 1회 재학습, `forecast_log.csv`로 매일 예측과 실제 결과를 대조.

### 하지 말 것 (측정 결과 효과가 없거나 잡음 이내)

- 시드 배깅(LightGBM 5시드 평균: bal. acc +0.004 [−0.010, +0.016])
- 일간 야후 종가에서 파생되는 값싼 피처 추가(14종 합계 +0.011, 표준오차 0.014)
- `VOL_BAND_MULT`를 balanced accuracy로 튜닝
- Transformer 확대나 Kronos 파인튜닝(모든 피처의 lag-1·lag-2를 더하면 선형 모델도 나빠진다 = 시퀀스 신호가 없다)

### 참고 자료

- [Kronos 논문](https://arxiv.org/abs/2508.02739) · [공식 구현](https://github.com/shiyu-coder/Kronos)
- [KRX Data Marketplace](https://data.krx.co.kr/)

## 15. 나머지 종목 이어서 실행

`RUN_TARGETS`에 두 종목 이상을 적었으면, 이 셀이 **설정 셀부터 보고서 셀까지를 종목만 바꿔 한 번 더 실행**합니다. 위에서 이미 끝난 종목은 저장·발행까지 완료된 상태이므로 다시 돌리지 않습니다.

출력은 종목별로 이어서 나오고, 원장과 보고서도 종목별 경로에 따로 쌓입니다.

- 종목 하나만 돌리려면 첫 셀에서 `RUN_TARGETS = ["samsung"]` 처럼 하나만 남기세요.
- 실행이 끝나면 노트북 변수(`live_table`, `predictions` 등)에는 **마지막 종목**의 결과가 남습니다. 앞 종목 결과는 저장된 파일과 보고서에서 확인하세요.

In [ ]:
# 첫 종목은 위에서 이미 끝났다. 나머지를 같은 코드로 이어서 실행한다.
_remaining = list(RUN_TARGETS[1:])
if not _remaining:
    print("RUN_TARGETS에 종목이 하나뿐입니다. 추가 실행 없음.")
else:
    try:
        _history = list(In)          # IPython이 보관한 실행 셀 원문
    except NameError:
        _history = []
    _start = next((i for i, s in enumerate(_history) if "RUN_TARGETS = [" in s and "TARGET_SPEC" in s), None)
    _end = next((i for i, s in enumerate(_history) if "== transformer-experiment ==" in s), None)

    if _start is None or _end is None or _end <= _start:
        print("⚠️ 실행 히스토리에서 파이프라인 구간을 찾지 못해 추가 종목을 건너뜁니다.")
        print("   위 셀들을 순서대로 실행한 뒤 이 셀을 실행하세요"
              " (런타임 → 모두 실행이면 자동으로 충족됩니다).")
    else:
        _block = _history[_start:_end + 1]
        print(f"이어서 실행할 종목: {_remaining} (셀 {len(_block)}개 재실행)\n")
        for _t in _remaining:
            print("=" * 78)
            print(f"■ {_t} 실행 시작")
            print("=" * 78, flush=True)
            for _src in _block:
                # 매직(%%capture, !pip)은 재실행에서 제외한다.
                _code = "\n".join(l for l in _src.splitlines()
                                   if not l.lstrip().startswith(("%", "!")))
                _code = _code.replace("TARGET = RUN_TARGETS[0]", f'TARGET = "{_t}"')
                if not _code.strip():
                    continue
                exec(compile(_code, f"<{_t}>", "exec"), globals())
        print("\n" + "=" * 78)
        print(f"■ 재실행 완료: {_remaining}")
        print("   노트북 변수에는 마지막 종목의 결과가 남아 있습니다.")
        print("   각 종목 결과는 저장 폴더와 보고서에서 확인하세요.")
        print("=" * 78)


# ---- 종목별 발행 결과 -------------------------------------------------------
# 출력이 수백 줄이라 중간 경고는 묻힌다. 무엇이 실제로 발행됐는지 맨 마지막에
# 다시 모아 둔다. 여기만 보면 한 종목이 빠졌는지 바로 알 수 있다.
_done = globals().get("PUBLISH_STATUS", {})
print("\n" + "=" * 78)
print("■ 종목별 발행 결과")
print("=" * 78)
_marks = {"발행됨": "OK  ", "건너뜀": "SKIP", "실패": "FAIL", "실행 안 됨": "MISS"}
_bad = []
for _t in RUN_TARGETS:
    _state, _detail = _done.get(_t, ("실행 안 됨", "이 실행에서 처리되지 않았습니다"))
    print(f"  [{_marks.get(_state, '?   ')}] {_t:<10} {_state:<7} {_detail}")
    if _state != "발행됨":
        _bad.append(_t)
if _bad:
    print()
    print(f"  ⚠️ 발행되지 않은 종목: {_bad}")
    print("     그 종목의 보고서는 이전 실행 결과 그대로 남아 있습니다.")
    print("     위 출력에서 해당 종목 구간의 경고를 확인하세요.")
print("=" * 78)